# 🌊 DAQathon Session 1: QC with Random Forests, CNNs, and k-means

This notebook is written to meet you where you are. You might already be comfortable with machine learning, or you might be seeing terms like *Random Forest*, *k-means*, or *convolutional neural network* for the first time.

The goal is to make the workflow feel approachable while still using a real ONC quality-control problem:

- start from raw scalar ONC data and QC flags,
- build one clear supervised baseline,
- build one clear unsupervised example,
- and leave with a better sense of how to improve the pipeline later.


## 🧰 One-Time FIR Setup

Before you use these notebooks on FIR for the first time, open a FIR terminal and run:

```bash
jupyter kernelspec install --user /project/def-kmoran/shared/daqathon/kernels/daqathon-ml
```

This is a **one-time per-user step**. After that:

- refresh JupyterHub if it was already open,
- open the notebook from your cloned repo,
- select the shared `Daqathon ML` kernel.


## 🧭 How To Use This Notebook

This notebook is meant to feel like a first clean pass through the workflow:

- understand the dataset and QC flags,
- build one reasonable tabular baseline with a Random Forest,
- build one reasonable sequential baseline with a CNN,
- then ask what we might improve next.

A few suggestions while you work through the notebook:

- read the markdown cells first, then run the code cells underneath them,
- use `DATA_FRACTION` in the config cell to switch between a quick demo and a longer run,
- treat the default model settings as **baseline choices**, not as final answers.

Optional background reading:

- [Random Forests in scikit-learn](https://scikit-learn.org/stable/modules/ensemble.html#forest)
- [k-means clustering in scikit-learn](https://scikit-learn.org/stable/modules/clustering.html#k-means)
- [Neural networks in PyTorch](https://docs.pytorch.org/tutorials/beginner/blitz/neural_networks_tutorial)
- [ONC QAQC reference](https://wiki.oceannetworks.ca/spaces/DP/pages/42174414/Quality+Assurance+Quality+Control)


## ⚙️ Configuration

The next few cells separate the main configuration ideas:

- choose a dataset preset,
- optionally override a few important dataset-specific fields,
- then adjust broad run controls like how much data to use.

On FIR, if `$SLURM_TMPDIR` is available, this notebook first stages the shared raw CSV directory and the prepared parquet cache into node-local job storage. It writes all generated outputs into a runtime directory chosen in this order:

1. `$SLURM_TMPDIR`
2. `$SCRATCH`
3. repo-local `tmp/session1_outputs`


In [ ]:
from pathlib import Path
import json

def setup_jsonable(value):
    if isinstance(value, Path):
        return str(value)
    if isinstance(value, dict):
        return {str(key): setup_jsonable(item) for key, item in value.items()}
    if isinstance(value, (list, tuple, set)):
        return [setup_jsonable(item) for item in value]
    return value

def show_setup_json(value):
    print(json.dumps(setup_jsonable(value), indent=2, ensure_ascii=False))

# Small helpers used by the setup cells below. We define them once here so later
# configuration cells can stay focused on one job each.
def first_existing_path(candidates: list[Path]) -> Path | None:
    for candidate in candidates:
        if candidate.exists():
            return candidate
    return None

def first_existing_csv_dir(candidates: list[Path]) -> Path | None:
    for candidate in candidates:
        if candidate.exists() and candidate.is_dir() and any(candidate.glob("*.csv")):
            return candidate
    return None

# Find the repo root for the cloned notebook repo.
NOTEBOOK_ROOT = Path.cwd()
for candidate_root in [NOTEBOOK_ROOT, *NOTEBOOK_ROOT.parents]:
    if (candidate_root / "notebooks").exists() and (candidate_root / "scripts").exists():
        NOTEBOOK_ROOT = candidate_root
        break

# Detect the shared FIR project space that holds the long-lived data,
# shared environment, and kernel.
shared_root_candidates = [
    Path("/project/def-kmoran/shared/daqathon"),
    Path("/project/6062898/shared/daqathon"),
    Path.home() / "projects" / "def-kmoran" / "shared" / "daqathon",
]
SHARED_DAQATHON_ROOT = first_existing_path(shared_root_candidates)
LOCAL_CACHE_DIR = NOTEBOOK_ROOT / "data" / "cache" / "session1"
SHARED_CACHE_DIR = (
    first_existing_path([SHARED_DAQATHON_ROOT / "data" / "cache" / "session1"])
    if SHARED_DAQATHON_ROOT
    else None
)

show_setup_json(
    {
        "NOTEBOOK_ROOT": str(NOTEBOOK_ROOT),
        "SHARED_DAQATHON_ROOT": str(SHARED_DAQATHON_ROOT) if SHARED_DAQATHON_ROOT else None,
        "LOCAL_CACHE_DIR": str(LOCAL_CACHE_DIR),
        "SHARED_CACHE_DIR": str(SHARED_CACHE_DIR) if SHARED_CACHE_DIR else None,
    }
)


### Dataset Presets

The next cell defines the **supported scalar dataset presets** for this notebook.

Each preset bundles together the choices that usually travel as a group:

- where the raw files live,
- which QC flag is the supervised target,
- which measurement columns to use,
- which two columns should appear in plots,
- which device family is the primary stream,
- and whether k-means should work on window summaries or row-level features.

In most cases, you only need to change `DATASET_PROFILE_ID`.


In [ ]:
# Each profile captures the dataset-specific choices that would otherwise be
# repeated throughout the notebook. Once a profile is selected, later cells
# work with normalized variables such as TARGET_FLAG or MEASUREMENT_COLUMNS.
DATASET_PROFILES = {
    "ctd_conductivity": {
        # A classic CTD example: one instrument export with conductivity QC as the label.
        "label": "CTD conductivity QC",
        "description": "Strait of Georgia East CTD data with conductivity QC as the supervised target.",
        "raw_subpaths": ["SoGEast_CTD_202503_202603"],
        "cache_stem": "conductivity_scalar_session1",
        "target_flag": "Conductivity QC Flag",
        "task_mode": "multiclass",
        "good_labels": [1],
        "issue_labels": [3, 4, 9],
        "flag_example_classes": [1, 3, 4, 9],
        "measurement_columns": [
            "Conductivity (S/m)",
            "Density (kg/m3)",
            "Depth (m)",
            "Practical Salinity (psu)",
            "Pressure (decibar)",
            "Sigma-t (kg/m3)",
            "Sigma-theta (0 dbar) (kg/m3)",
            "Sound Speed (m/s)",
            "Temperature (C)",
        ],
        "optional_qc_columns": ["Temperature QC Flag"],
        "plot_measurement_column": "Conductivity (S/m)",
        "plot_secondary_column": "Temperature (C)",
        "primary_device": "ctd",
        "kmeans_feature_mode": "window_summary",
        "default_sequence_output_mode": "window",
        "default_sequence_target_strategy": "collapsed_1_34_9",
        "benchmark_sample_rows": None,
        "auto_build_missing_cache": True,
        "default_prep_sample_rows": None,
    },
    "fluorometer_turbidity": {
        # A merged scalar example: fluorometer is the primary device, but CTD and oxygen
        # provide useful context columns during the merge.
        "label": "Fluorometer turbidity QC",
        "description": "Merged scalar data around a fluorometer/turbidity target, including CTD and oxygen context columns.",
        "raw_subpaths": ["Fluorometer/SoGCentral", "Fluorometer/Folger", "Fluorometer/SoGCentral_test", "SoGCentral_test"],
        "cache_stem": "fluorometer_scalar_session1",
        "target_flag": "Turbidity QC Flag",
        "task_mode": "multiclass",
        "good_labels": [1],
        "issue_labels": [3, 4, 9],
        "flag_example_classes": [1, 3, 4, 9],
        "measurement_columns": [
            "Chlorophyll (ug/l)",
            "Turbidity (NTU)",
            "Conductivity (S/m)",
            "Density (kg/m3)",
            "Practical Salinity (psu)",
            "Pressure (decibar)",
            "Sigma-t (kg/m3)",
            "Sigma-theta (0 dbar) (kg/m3)",
            "Sound Speed (m/s)",
            "Temperature (C)",
            "Oxygen Concentration Corrected (ml/l)",
            "Oxygen Concentration Uncorrected (ml/l)",
        ],
        "optional_qc_columns": ["Chlorophyll QC Flag"],
        "plot_measurement_column": "Turbidity (NTU)",
        "plot_secondary_column": "Temperature (C)",
        "primary_device": "fluorometer",
        "kmeans_feature_mode": "row_level",
        "default_sequence_output_mode": "per_timestep",
        "default_sequence_target_strategy": "collapsed_1_34_9",
        "benchmark_sample_rows": None,
        "auto_build_missing_cache": True,
        "default_prep_sample_rows": None,
    },
    "oxygen": {
        # A smaller oxygen-focused profile for the same scalar workflow.
        "label": "Oxygen QC",
        "description": "Scalar oxygen data with oxygen concentration QC as the supervised target.",
        "raw_subpaths": ["Fluorometer/SoGCentral", "Fluorometer/Folger", "SoGEast_Oxygen_202503_202603", "Oxygen", "oxygen"],
        "cache_stem": "oxygen_scalar_session1",
        "target_flag": "Oxygen Concentration Corrected QC Flag",
        "task_mode": "multiclass",
        "good_labels": [1],
        "issue_labels": [3, 4, 9],
        "flag_example_classes": [1, 2, 3, 4, 9],
        "measurement_columns": [
            "Oxygen Concentration Corrected (ml/l)",
            "Oxygen Concentration Uncorrected (ml/l)",
            "Temperature (C)",
            "Pressure (decibar)",
        ],
        "optional_qc_columns": ["Temperature QC Flag"],
        "plot_measurement_column": "Oxygen Concentration Corrected (ml/l)",
        "plot_secondary_column": "Temperature (C)",
        "primary_device": "oxygen",
        "kmeans_feature_mode": "window_summary",
        "default_sequence_output_mode": "window",
        "default_sequence_target_strategy": "collapsed_12_34_9",
        "benchmark_sample_rows": None,
        "auto_build_missing_cache": True,
        "default_prep_sample_rows": None,
    },
    "conductivity_plugs": {
        # A labelled conductivity-plug example with a compact CSV schema.
        "label": "Conductivity plugs",
        "description": "Conductivity-plug data with a custom ml_label target and CTD/oxygen context columns.",
        "raw_subpaths": ["ConductivityPlugs"],
        "cache_stem": "conductivity_plugs_session1",
        "target_flag": "ml_label",
        "task_mode": "multiclass",
        # Treat the plug labels like ONC-style QC labels:
        # 1/2 are acceptable, 3/4 are problematic, and 0 is
        # "no QC / unlabeled" rather than an issue label.
        "good_labels": [1, 2],
        "issue_labels": [3, 4],
        "flag_example_classes": [1, 2, 3, 4],
        "measurement_columns": [
            "cond_value_ctd",
            "density_value_ctd",
            "Pressure_value_ctd",
            "salinity_value_ctd",
            "sigmaT_value_ctd",
            "SIGMA_THETA_value_ctd",
            "Sound_Speed_value_ctd",
            "Temperature_value_ctd",
            "oxygen_corrected_value_oxy",
            "oxygen_uncorrected_value_oxy",
            "temperature_value_oxy",
            "temperature_offset",
            "temperature_offset_anomaly",
            "temperature_offset_over_start_mean",
        ],
        "optional_qc_columns": ["cond_qaqc_ctd", "oxygen_corrected_qaqc_oxy", "temperature_qaqc_oxy"],
        "plot_measurement_column": "cond_value_ctd",
        "plot_secondary_column": "Temperature_value_ctd",
        "primary_device": "other",
        "kmeans_feature_mode": "window_summary",
        "default_sequence_output_mode": "per_timestep",
        "default_sequence_target_strategy": "raw_multiclass",
        "benchmark_sample_rows": 250_000,
        "auto_build_missing_cache": False,
        "default_prep_sample_rows": 500_000,
    },
}

# Change this to switch the whole notebook to a different supported preset.
DATASET_PROFILE_ID = "ctd_conductivity"
DATASET_PROFILE = DATASET_PROFILES[DATASET_PROFILE_ID]

show_setup_json(
    {
        "DATASET_PROFILE_ID": DATASET_PROFILE_ID,
        "DATASET_LABEL": DATASET_PROFILE["label"],
        "DESCRIPTION": DATASET_PROFILE["description"],
        "PRIMARY_DEVICE": DATASET_PROFILE["primary_device"],
        "GOOD_LABELS": DATASET_PROFILE.get("good_labels", [1]),
        "ISSUE_LABELS": DATASET_PROFILE.get("issue_labels", [3, 4, 9]),
    }
)


### Optional Dataset Overrides

Presets are the normal starting point, but these overrides let you change a few high-value pieces without editing the rest of the notebook.

Leave them as `None` to use the preset values. Typical reasons to override something here are:

- testing a different raw folder,
- changing the target flag,
- narrowing the measurement columns,
- or forcing a different k-means feature mode.


In [ ]:
# Leave these as None unless you intentionally want to override part of the preset.
# The simplest workflow is "pick a preset, then tweak only one or two fields."
MANUAL_RAW_DATA_DIR = None
MANUAL_TARGET_FLAG = None
MANUAL_MEASUREMENT_COLUMNS = None
MANUAL_OPTIONAL_QC_COLUMNS = None
MANUAL_PLOT_MEASUREMENT_COLUMN = None
MANUAL_PLOT_SECONDARY_COLUMN = None
MANUAL_CACHE_BUNDLE_NAME = None
MANUAL_KMEANS_FEATURE_MODE = None

# Build a short list of likely raw-data locations for this profile.
# On FIR we prefer the shared project directory; locally we fall back to repo data.
profile_raw_candidates = [Path(subpath) for subpath in DATASET_PROFILE["raw_subpaths"]]
local_raw_candidates = [NOTEBOOK_ROOT / "data" / "raw" / subpath for subpath in profile_raw_candidates]
shared_raw_candidates = (
    [SHARED_DAQATHON_ROOT / "data" / "raw" / subpath for subpath in profile_raw_candidates]
    if SHARED_DAQATHON_ROOT
    else []
)

# Resolve the best available raw-data and cache locations without forcing
# you to edit paths in multiple notebook sections.
detected_raw_data_dir = (
    first_existing_csv_dir([candidate for candidate in [*shared_raw_candidates, *local_raw_candidates] if candidate is not None])
    or first_existing_path([candidate for candidate in [*shared_raw_candidates, *local_raw_candidates, NOTEBOOK_ROOT / "data" / "raw"] if candidate is not None])
)
detected_cache_dir = first_existing_path([candidate for candidate in [LOCAL_CACHE_DIR, SHARED_CACHE_DIR] if candidate is not None])

# Normalize everything into the small set of variables used by the rest of the notebook.
# From this point onward, later cells should not need to care which preset was chosen.
READ_RAW_DATA_DIR = MANUAL_RAW_DATA_DIR or (
    str(detected_raw_data_dir)
    if detected_raw_data_dir is not None
    else str(local_raw_candidates[0] if local_raw_candidates else NOTEBOOK_ROOT / "data" / "raw")
)
READ_CACHE_DIR = str(detected_cache_dir) if detected_cache_dir is not None else str(LOCAL_CACHE_DIR)

TARGET_FLAG = MANUAL_TARGET_FLAG or DATASET_PROFILE["target_flag"]
TASK_MODE = DATASET_PROFILE["task_mode"]
GOOD_LABELS = [int(label) for label in DATASET_PROFILE.get("good_labels", [1])]
ISSUE_LABELS = [int(label) for label in DATASET_PROFILE.get("issue_labels", [3, 4, 9])]
FLAG_EXAMPLE_CLASSES = tuple(int(label) for label in DATASET_PROFILE.get("flag_example_classes", [1, 3, 4, 9]))
CACHE_BUNDLE_NAME = MANUAL_CACHE_BUNDLE_NAME or DATASET_PROFILE["cache_stem"]
PRIMARY_DEVICE = DATASET_PROFILE["primary_device"]
KMEANS_FEATURE_MODE = MANUAL_KMEANS_FEATURE_MODE or DATASET_PROFILE["kmeans_feature_mode"]
DEFAULT_SEQUENCE_OUTPUT_MODE = DATASET_PROFILE.get("default_sequence_output_mode", "window")
DEFAULT_SEQUENCE_TARGET_STRATEGY = DATASET_PROFILE.get("default_sequence_target_strategy", "collapsed_1_34_9")
RAW_BENCHMARK_SAMPLE_ROWS = DATASET_PROFILE.get("benchmark_sample_rows")
AUTO_BUILD_MISSING_CACHE = bool(DATASET_PROFILE.get("auto_build_missing_cache", True))
DEFAULT_PREP_SAMPLE_ROWS = DATASET_PROFILE.get("default_prep_sample_rows")

# Keep the feature list in one place so every later section reuses the same columns.
MEASUREMENT_COLUMNS = list(MANUAL_MEASUREMENT_COLUMNS or DATASET_PROFILE["measurement_columns"])
OPTIONAL_QC_COLUMNS = list(dict.fromkeys(MANUAL_OPTIONAL_QC_COLUMNS or DATASET_PROFILE["optional_qc_columns"]))
PLOT_MEASUREMENT_COLUMN = MANUAL_PLOT_MEASUREMENT_COLUMN or DATASET_PROFILE["plot_measurement_column"]
PLOT_SECONDARY_COLUMN = MANUAL_PLOT_SECONDARY_COLUMN or DATASET_PROFILE["plot_secondary_column"]

# These explicit column lists are one of the main reasons the notebook can stay fast:
# we only read the row-level or window-level fields we actually need downstream.
ROW_USE_COLUMNS = list(dict.fromkeys(["Time UTC", "source_file", TARGET_FLAG, *OPTIONAL_QC_COLUMNS, *MEASUREMENT_COLUMNS]))
WINDOW_FEATURE_COLUMNS = [
    f"{column}_{stat}"
    for column in MEASUREMENT_COLUMNS
    for stat in ("mean", "std")
]
WINDOW_USE_COLUMNS = [
    "window_start",
    "window_end",
    "source_file",
    "issue_rate",
    *WINDOW_FEATURE_COLUMNS,
]

show_setup_json(
    {
        "READ_RAW_DATA_DIR": READ_RAW_DATA_DIR,
        "READ_CACHE_DIR": READ_CACHE_DIR,
        "TARGET_FLAG": TARGET_FLAG,
        "CACHE_BUNDLE_NAME": CACHE_BUNDLE_NAME,
        "KMEANS_FEATURE_MODE": KMEANS_FEATURE_MODE,
        "DEFAULT_SEQUENCE_OUTPUT_MODE": DEFAULT_SEQUENCE_OUTPUT_MODE,
        "DEFAULT_SEQUENCE_TARGET_STRATEGY": DEFAULT_SEQUENCE_TARGET_STRATEGY,
        "RAW_BENCHMARK_SAMPLE_ROWS": RAW_BENCHMARK_SAMPLE_ROWS,
        "AUTO_BUILD_MISSING_CACHE": AUTO_BUILD_MISSING_CACHE,
        "DEFAULT_PREP_SAMPLE_ROWS": DEFAULT_PREP_SAMPLE_ROWS,
        "GOOD_LABELS": GOOD_LABELS,
        "ISSUE_LABELS": ISSUE_LABELS,
        "MEASUREMENT_COLUMNS": MEASUREMENT_COLUMNS,
        "PLOT_MEASUREMENT_COLUMN": PLOT_MEASUREMENT_COLUMN,
        "PLOT_SECONDARY_COLUMN": PLOT_SECONDARY_COLUMN,
    }
)


### Run Controls

These are the notebook-wide knobs you are most likely to change.

The most important is `DATA_FRACTION`:

- use a small value for quick live demos,
- increase it when you want a stronger run,
- keep it near `1.0` when you want to use the largest training budget.

`DATA_FRACTION` mostly affects the **train-only budget** and a few heavier visualizations. The split diagnostics in Part 3 use the full reviewed dataset from the selected parquet parts.


In [ ]:
# One easy top-level knob: start at 0.1 for a quick run, then raise it if you want a bigger experiment.
DATA_FRACTION = 0.1
if not 0 < DATA_FRACTION <= 1:
    raise ValueError("DATA_FRACTION must be in the interval (0, 1].")

# These base values are reused by smaller section-local config cells later on.
BASE_ISSUE_ROWS_PER_FILE = 12000
BASE_MODEL_ROW_LIMIT = 1_000_000
BASE_WINDOW_POINT_LIMIT = 1.0
BASE_FLAG_EXAMPLE_POINTS_PER_PANEL = 30000

# Global row/window selection policy.
PART_SELECTION_MODE = "spread"
ROW_FILE_LIMIT = None

# Split fractions are reused across every split strategy. The strategy itself is
# configured in the next cell.
TRAIN_FRACTION = 0.70
VALIDATION_FRACTION = 0.15

# Toggle the sequence-model sections and keep runs reproducible.
RUN_CNN_APPENDIX = True
SEED = 21

show_setup_json(
    {
        "DATA_FRACTION": DATA_FRACTION,
        "RUN_CNN_APPENDIX": RUN_CNN_APPENDIX,
        "SEED": SEED,
        "TRAIN_FRACTION": TRAIN_FRACTION,
        "VALIDATION_FRACTION": VALIDATION_FRACTION,
    }
)


In [ ]:
from __future__ import annotations

import builtins
import copy
import importlib
import inspect
import json
import os
import pickle
import subprocess
import sys
from collections.abc import Mapping
from pathlib import Path
from time import perf_counter

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
from IPython.display import display
from matplotlib.lines import Line2D
from matplotlib.ticker import PercentFormatter
from sklearn.cluster import KMeans
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import ConfusionMatrixDisplay, classification_report, f1_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.utils.class_weight import compute_class_weight

if str(NOTEBOOK_ROOT) not in sys.path:
    sys.path.insert(0, str(NOTEBOOK_ROOT))

import scripts.prepare_scalar_session1_data as prepare_scalar_session1_data
import scripts.session1_modeling as session1_modeling

importlib.invalidate_caches()
prepare_scalar_session1_data = importlib.reload(prepare_scalar_session1_data)
session1_modeling = importlib.reload(session1_modeling)

SLURM_TMPDIR = os.environ.get("SLURM_TMPDIR")
SCRATCH = os.environ.get("SCRATCH")
RUNTIME_OUTPUT_ROOT = (
    Path(SLURM_TMPDIR) / "daqathon" / "session1_outputs"
    if SLURM_TMPDIR
    else (Path(SCRATCH) / "daqathon" / "session1_outputs" if SCRATCH else NOTEBOOK_ROOT / "tmp" / "session1_outputs")
)
RUNTIME_RAW_DATA_DIR = RUNTIME_OUTPUT_ROOT / "data" / "raw" / Path(READ_RAW_DATA_DIR).name
RUNTIME_CACHE_DIR = RUNTIME_OUTPUT_ROOT / "cache" / "session1"
ARTIFACT_DIR = RUNTIME_OUTPUT_ROOT / "artifacts"
MODEL_OUTPUT_DIR = RUNTIME_OUTPUT_ROOT / "models"
PLOT_OUTPUT_DIR = RUNTIME_OUTPUT_ROOT / "plots"
REPORT_OUTPUT_DIR = RUNTIME_OUTPUT_ROOT / "reports"

for output_dir in [RUNTIME_OUTPUT_ROOT, ARTIFACT_DIR, MODEL_OUTPUT_DIR, PLOT_OUTPUT_DIR, REPORT_OUTPUT_DIR]:
    output_dir.mkdir(parents=True, exist_ok=True)

build_cache_bundle_paths = session1_modeling.build_cache_bundle_paths
resolve_cache_bundle_paths = session1_modeling.resolve_cache_bundle_paths

def read_parquet_head(parquet_path: str | Path, columns: list[str] | None = None, max_rows: int | None = None) -> pd.DataFrame:
    parquet_path = Path(parquet_path)
    if max_rows is None:
        return pd.read_parquet(parquet_path, columns=columns)

    parquet_file = pq.ParquetFile(parquet_path)
    batch_size = max(1, min(int(max_rows), 65536))
    frames: list[pd.DataFrame] = []
    rows_remaining = int(max_rows)

    for batch in parquet_file.iter_batches(columns=columns, batch_size=batch_size):
        if rows_remaining <= 0:
            break
        frame = batch.to_pandas()
        if len(frame) > rows_remaining:
            frame = frame.iloc[:rows_remaining].copy()
        frames.append(frame)
        rows_remaining -= len(frame)

    if not frames:
        return pd.DataFrame(columns=columns or parquet_file.schema.names)
    return pd.concat(frames, ignore_index=True)

def choose_cache_paths(cache_roots: list[str | Path | None], cache_stem: str):
    fallback_root: Path | None = None
    for cache_root in cache_roots:
        if cache_root is None:
            continue
        root_path = Path(cache_root).expanduser()
        if fallback_root is None:
            fallback_root = root_path
        candidate = resolve_cache_bundle_paths(root_path, cache_stem=cache_stem)
        if candidate.metadata_path.exists():
            return candidate
    if fallback_root is None:
        raise ValueError("At least one cache root is required")
    return build_cache_bundle_paths(fallback_root, cache_stem)

USE_RUNTIME_RAW_DATA_FOR_READS = bool(SLURM_TMPDIR)
USE_RUNTIME_CACHE_FOR_READS = bool(SLURM_TMPDIR)
RAW_DATA_DIR = str(RUNTIME_RAW_DATA_DIR if USE_RUNTIME_RAW_DATA_FOR_READS else Path(READ_RAW_DATA_DIR))
CACHE_DIR = str(RUNTIME_CACHE_DIR if USE_RUNTIME_CACHE_FOR_READS else Path(READ_CACHE_DIR))
cache_paths = build_cache_bundle_paths(CACHE_DIR, CACHE_BUNDLE_NAME)
ROW_CACHE_DIR = str(cache_paths.row_level_dir)
WINDOW_CACHE_PATH = str(cache_paths.window_cache_path)
METADATA_PATH = str(cache_paths.metadata_path)

RF_MODEL_PATH = MODEL_OUTPUT_DIR / "best_random_forest.pkl"
CNN_MODEL_PATH = MODEL_OUTPUT_DIR / "best_cnn_checkpoint.pt"
TRANSFORMER_MODEL_PATH = MODEL_OUTPUT_DIR / "best_transformer_checkpoint.pt"

add_temporal_context_features = session1_modeling.add_temporal_context_features
add_tabular_baseline_features = session1_modeling.add_tabular_baseline_features
apply_target_strategy = session1_modeling.apply_target_strategy
build_labeled_intervals = session1_modeling.build_labeled_intervals
build_model_frame_impl = session1_modeling.build_model_frame
build_reviewed_target_frame_impl = session1_modeling.build_reviewed_target_frame
build_sequence_split_bundle = session1_modeling.build_sequence_split_bundle
build_sequence_split_bundle_from_frames = session1_modeling.build_sequence_split_bundle_from_frames
build_sequence_label_interval_data = session1_modeling.build_sequence_label_interval_data
build_window_classification_interval_data = session1_modeling.build_window_classification_interval_data
compute_contiguous_split_target_distribution = session1_modeling.compute_contiguous_split_target_distribution
compute_interval_classification_metrics = session1_modeling.compute_interval_classification_metrics
compute_split_share_gap = session1_modeling.compute_split_share_gap
DEFAULT_FLAG_PALETTE = session1_modeling.DEFAULT_FLAG_PALETTE
clean_source_file_label = session1_modeling.clean_source_file_label
evaluate_classifier = session1_modeling.evaluate_classifier
fit_extra_trees = session1_modeling.fit_extra_trees
fit_kmeans = session1_modeling.fit_kmeans
fit_random_forest = session1_modeling.fit_random_forest
infer_interval_origin = session1_modeling.infer_interval_origin
load_rows_for_time_range = session1_modeling.load_rows_for_time_range
load_full_row_level_frame = session1_modeling.load_full_row_level_frame
load_selected_row_level_frame = session1_modeling.load_selected_row_level_frame
materialize_reviewed_split_frames = session1_modeling.materialize_reviewed_split_frames
merge_adjacent_intervals = session1_modeling.merge_adjacent_intervals
plot_cluster_window_examples = session1_modeling.plot_cluster_window_examples
plot_flag_examples = session1_modeling.plot_flag_examples
plot_time_series_with_bands = session1_modeling.plot_time_series_with_bands
predict_cnn_window_model = session1_modeling.predict_cnn_window_model
predict_sequence_label_cnn = session1_modeling.predict_sequence_label_cnn
predict_transformer_sequence_model = session1_modeling.predict_transformer_sequence_model
predict_transformer_window_model = session1_modeling.predict_transformer_window_model
resolve_runtime_output_root = session1_modeling.resolve_runtime_output_root
run_cnn_search_from_frames = session1_modeling.run_cnn_search_from_frames
select_time_range = session1_modeling.select_time_range
run_cnn_search = session1_modeling.run_cnn_search
run_rf_search = session1_modeling.run_rf_search
scan_interleaved_block_rows = session1_modeling.scan_interleaved_block_rows
sample_frame_by_strategy = session1_modeling.sample_frame_by_strategy
stage_directory_into_runtime = session1_modeling.stage_directory_into_runtime
stage_cache_into_runtime = session1_modeling.stage_cache_into_runtime
split_frame_by_strategy_impl = session1_modeling.split_frame_by_strategy
summarize_split_distributions = session1_modeling.summarize_split_distributions
summarize_target_by_time_bin = session1_modeling.summarize_target_by_time_bin
SUPPORTED_SPLIT_STRATEGIES = session1_modeling.SUPPORTED_SPLIT_STRATEGIES

DEFAULT_MEASUREMENT_COLUMNS = prepare_scalar_session1_data.DEFAULT_MEASUREMENT_COLUMNS
locate_header = prepare_scalar_session1_data.locate_header
read_scalar_csv = prepare_scalar_session1_data.read_scalar_csv

# Notebook status summaries are often easiest to scan as structured JSON rather than
# as one long wrapped line. The helpers below keep the later cells simple: when a cell
# calls print({...}), it will render as formatted JSON automatically.
_builtin_print = builtins.print

def to_jsonable(value):
    if isinstance(value, Path):
        return str(value)
    if isinstance(value, pd.Timestamp):
        return value.isoformat()
    if isinstance(value, pd.Timedelta):
        return str(value)
    if isinstance(value, np.generic):
        return value.item()
    if isinstance(value, np.ndarray):
        return [to_jsonable(item) for item in value.tolist()]
    if isinstance(value, Mapping):
        return {str(key): to_jsonable(item) for key, item in value.items()}
    if isinstance(value, (list, tuple, set)):
        return [to_jsonable(item) for item in value]
    return value

def pretty_json(value):
    _builtin_print(json.dumps(to_jsonable(value), indent=2, ensure_ascii=False))

def notebook_print(*args, **kwargs):
    if len(args) == 1 and not kwargs and isinstance(args[0], Mapping):
        pretty_json(args[0])
        return
    _builtin_print(*args, **kwargs)

print = notebook_print

plt.style.use("seaborn-v0_8-whitegrid")
pd.set_option("display.max_columns", 100)
np.random.seed(SEED)
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
MODEL_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
REPORT_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
PLOT_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(
    {
        "TASK_MODE": TASK_MODE,
        "TARGET_FLAG": TARGET_FLAG,
        "DATA_FRACTION": DATA_FRACTION,
        "RUN_CNN_APPENDIX": RUN_CNN_APPENDIX,
        "SHARED_DAQATHON_ROOT": str(SHARED_DAQATHON_ROOT) if SHARED_DAQATHON_ROOT else None,
        "READ_RAW_DATA_DIR": READ_RAW_DATA_DIR,
        "READ_CACHE_DIR": READ_CACHE_DIR,
        "USE_RUNTIME_RAW_DATA_FOR_READS": USE_RUNTIME_RAW_DATA_FOR_READS,
        "USE_RUNTIME_CACHE_FOR_READS": USE_RUNTIME_CACHE_FOR_READS,
        "RUNTIME_OUTPUT_ROOT": str(RUNTIME_OUTPUT_ROOT),
        "RUNTIME_RAW_DATA_DIR": str(RUNTIME_RAW_DATA_DIR),
        "CACHE_BUNDLE_NAME": CACHE_BUNDLE_NAME,
        "ROW_CACHE_DIR": ROW_CACHE_DIR,
        "WINDOW_CACHE_PATH": WINDOW_CACHE_PATH,
    }
)


## 🚀 FIR Input Staging

On FIR, the shared project space is the long-lived home for the data. But interactive notebook jobs on Alliance systems usually also get a directory called `$SLURM_TMPDIR`.

`SLURM_TMPDIR` is a **job-specific temporary working directory** created for you when Slurm starts a scheduled job or interactive session on a compute node. In other words:

- it appears when your Jupyter job is actually running on allocated resources,
- it lives on storage attached to that compute node,
- and it disappears when the job ends.

That makes it very different from your shared `HOME`, `SCRATCH`, or project directories:

- shared filesystems are persistent and great for canonical storage,
- `SLURM_TMPDIR` is temporary but usually much faster for repeated reads and writes during one run.

That speed matters here because notebooks repeatedly scan parquet parts, load row samples, and build model inputs. Copying the raw CSV directory and the prepared parquet cache into `SLURM_TMPDIR` once at the start usually makes the rest of the session feel much more responsive and reduces repeated traffic to the shared filesystem.

That means:

- the shared project space stays as the canonical source,
- the notebook reads from a fast local working copy during the run,
- and all generated files stay in your job-local runtime area instead of polluting the shared project tree.

One important caution: `SLURM_TMPDIR` is **not** long-term storage. Anything you want to keep after the job ends should be copied to `SCRATCH`, project storage, or a Git repo. This notebook uses `SLURM_TMPDIR` for speed, not for permanence.

The next code cell shows a progress bar while those copies are happening.


In [ ]:
# On FIR, stage shared inputs into node-local storage before we start reading them.
read_raw_data_dir = Path(READ_RAW_DATA_DIR)
runtime_raw_data_dir = Path(RUNTIME_RAW_DATA_DIR)
read_cache_dir = Path(READ_CACHE_DIR)
runtime_cache_dir = Path(RUNTIME_CACHE_DIR)

raw_stage_result = {"staged": False, "reason": "raw-data staging is disabled because SLURM_TMPDIR is not available"}
cache_stage_result = {"staged": False, "reason": "cache staging is disabled because SLURM_TMPDIR is not available"}
if USE_RUNTIME_RAW_DATA_FOR_READS:
    raw_stage_result = stage_directory_into_runtime(
        source_dir=read_raw_data_dir,
        runtime_dir=runtime_raw_data_dir,
        force=False,
        show_progress=True,
        progress_label=f"Staging raw scalar CSV files for {DATASET_PROFILE['label']} to SLURM_TMPDIR",
    )
if USE_RUNTIME_CACHE_FOR_READS:
    cache_stage_result = stage_cache_into_runtime(
        persistent_cache_dir=read_cache_dir,
        runtime_cache_dir=runtime_cache_dir,
        force=False,
        show_progress=True,
        progress_label="Staging parquet cache to SLURM_TMPDIR",
    )

active_raw_data_dir = runtime_raw_data_dir if runtime_raw_data_dir.exists() and any(runtime_raw_data_dir.glob("*.csv")) else read_raw_data_dir
active_cache_paths = choose_cache_paths([runtime_cache_dir, read_cache_dir], cache_stem=CACHE_BUNDLE_NAME)
RAW_DATA_DIR = str(active_raw_data_dir)
CACHE_DIR = str(active_cache_paths.root)
ROW_CACHE_DIR = str(active_cache_paths.row_level_dir)
WINDOW_CACHE_PATH = str(active_cache_paths.window_cache_path)
METADATA_PATH = str(active_cache_paths.metadata_path)

print(
    {
        "DATASET_PROFILE_ID": DATASET_PROFILE_ID,
        "raw_stage_result": raw_stage_result,
        "cache_stage_result": cache_stage_result,
        "active_raw_data_dir": RAW_DATA_DIR,
        "active_cache_dir": CACHE_DIR,
        "cache_bundle_name": CACHE_BUNDLE_NAME,
        "resolved_cache_stem": active_cache_paths.stem,
    }
)


## Part 1 — Orientation and Dataset

Start by understanding what the selected scalar dataset contains, what the QC flags mean, and what success looks like before we touch any modeling code.


#### About The Dataset

This notebook now supports a small set of **scalar dataset presets** rather than one hard-coded instrument export.

The active preset is chosen in the configuration cell near the top. That preset tells the notebook:

- which raw-data folder to read,
- which QC flag column is the target,
- which measurement columns to model,
- which two variables to emphasize in plots,
- and whether k-means should cluster window summaries or row-level values.

Across all of these presets, the mental model stays the same:

- each row is one timestamp in UTC,
- measurement columns hold scalar sensor values,
- QC flag columns hold ONC quality assessments for those values.

The printed config summary tells you exactly which preset is active for the current run.


#### What The QC Flags Mean

ONC uses a standard QAQC flag system across scalar data products. The exact target column can change across presets, but the flag meanings are shared. The official reference is:

- [ONC Quality Assurance Quality Control](https://wiki.oceannetworks.ca/spaces/DP/pages/42174414/Quality+Assurance+Quality+Control)

A practical reading of the most common flags is:

- `0`: no quality control has been applied to that value yet
- `1`: passed all tests and is treated as good data
- `2`: probably good, but worth a little more caution than `1`
- `3`: probably bad or suspect
- `4`: bad and usually treated as failed data
- `6`: insufficient valid data for reliable down-sampling
- `7`: averaged value
- `8`: interpolated value
- `9`: missing value (`NaN`)

A few practical notes are especially important for this notebook:

- `1` is the clearest "normal / healthy" label.
- `3` and `4` are the main issue labels we usually care about in supervised QC experiments.
- `9` means the value is missing, not merely noisy.
- `0` does **not** mean good; it means no QC decision has been applied.

ONC's reference page also explains that manual review can override automated tests, which is useful context when you see the same instrument behavior receive different final flags over time.

The selected target flag column is shown in the config summary, and later cells will show which QC values actually appear in the active dataset and which ones we treat as reviewed labels for modeling.


### Looking At Real Examples Early

Before moving on to caching or feature engineering, it helps to inspect a lightweight sample of the raw CSV data directly.

In the next few cells we:

- load a small sample from a few raw CSV files spread across the time range,
- look at a few real rows,
- and plot representative QC intervals in context.

This gives you a concrete picture of what one timestamp looks like before the notebook starts building model inputs from it.


### Small Utilities For Exploration Samples

The next helper cell defines a few short utilities that we first need here in Part 1:

- to spread a sample across time instead of taking only the earliest rows,
- to select files across the full time range,
- and to read a small raw-CSV sample without editing the original exports.

They stay available for later sections too, but this is the first place they become useful.


In [ ]:
# Take an evenly spaced subset without scrambling time order.
def evenly_spaced_take(frame: pd.DataFrame, limit: int | None) -> pd.DataFrame:
    if limit is None or len(frame) <= limit:
        return frame.reset_index(drop=True)
    indices = np.linspace(0, len(frame) - 1, num=limit, dtype=int)
    return frame.iloc[indices].reset_index(drop=True)


# Choose files spread across the whole time range instead of just the first few.
def select_part_paths(part_paths: list[Path], limit: int | None, mode: str) -> list[Path]:
    if limit is None or limit >= len(part_paths):
        return part_paths
    if mode == "first":
        return part_paths[:limit]
    if mode == "spread":
        indices = np.linspace(0, len(part_paths) - 1, num=limit, dtype=int)
        selected = []
        seen = set()
        for index in indices:
            candidate = part_paths[int(index)]
            if candidate not in seen:
                selected.append(candidate)
                seen.add(candidate)
        return selected
    raise ValueError(f"Unsupported PART_SELECTION_MODE: {mode}")


# Read a lightweight sample directly from raw CSV files for early exploration.
def load_raw_scalar_sample(
    csv_paths: list[Path],
    sample_rows_per_file: int | None,
    columns: list[str] | None = None,
) -> pd.DataFrame:
    row_frames = []
    for path in csv_paths:
        # For quick notebook previews, sample during the CSV read instead of
        # loading an entire multi-gigabyte file and trimming it afterward.
        frame = read_scalar_csv(
            path,
            sample_rows=sample_rows_per_file,
            required_columns=columns,
            allow_missing_columns=True,
        ).sort_values("Time UTC").reset_index(drop=True)
        row_frames.append(frame)

    return pd.concat(row_frames, ignore_index=True).sort_values("Time UTC").reset_index(drop=True)


# Scan raw CSV files in chunks and pull local context windows around
# requested QC flags. This keeps the Part 1 plots tied to the raw CSVs
# while avoiding the "first N rows only" artifact from the preview table.
def load_raw_flag_context_sample(
    csv_paths: list[Path],
    *,
    target_flag: str,
    classes: list[int] | tuple[int, ...],
    context_rows_per_class: int,
    columns: list[str] | None = None,
    chunk_rows: int | None = None,
) -> pd.DataFrame:
    requested_classes = [int(flag) for flag in classes]
    remaining_classes = list(dict.fromkeys(requested_classes))
    context_rows_per_class = max(int(context_rows_per_class), 1)
    chunk_rows = max(int(chunk_rows or max(context_rows_per_class * 2, 50000)), context_rows_per_class)
    use_columns = list(dict.fromkeys(["Time UTC", target_flag, *(columns or [])]))
    context_frames = []

    for path in csv_paths:
        if not remaining_classes:
            break

        header_line_number, csv_columns = locate_header(path)
        available_columns = [column for column in use_columns if column in csv_columns]
        trailing_context = pd.DataFrame(columns=available_columns)

        for chunk in pd.read_csv(
            path,
            header=None,
            names=csv_columns,
            skiprows=header_line_number,
            usecols=available_columns,
            chunksize=chunk_rows,
            low_memory=False,
        ):
            chunk["Time UTC"] = pd.to_datetime(chunk["Time UTC"], utc=True, errors="coerce", format="ISO8601")
            for column in [column for column in chunk.columns if column != "Time UTC"]:
                chunk[column] = pd.to_numeric(chunk[column], errors="coerce")
            chunk = chunk.dropna(subset=["Time UTC"]).sort_values("Time UTC").reset_index(drop=True)
            chunk["source_file"] = path.name
            if chunk.empty:
                continue

            combined = (
                pd.concat([trailing_context, chunk], ignore_index=True)
                if not trailing_context.empty
                else chunk.copy()
            )
            combined_flags = combined[target_flag].fillna(-1).astype(int)

            for flag in remaining_classes.copy():
                flag_positions = combined.index[combined_flags == flag].tolist()
                if not flag_positions:
                    continue

                span_start_index = flag_positions[0]
                while span_start_index > 0 and int(combined_flags.iloc[span_start_index - 1]) == flag:
                    span_start_index -= 1
                span_end_index = flag_positions[0]
                while span_end_index + 1 < len(combined) and int(combined_flags.iloc[span_end_index + 1]) == flag:
                    span_end_index += 1

                span_midpoint_index = (span_start_index + span_end_index) // 2
                start = max(span_midpoint_index - context_rows_per_class // 2, 0)
                stop = min(start + context_rows_per_class, len(combined))
                if stop - start < context_rows_per_class:
                    start = max(stop - context_rows_per_class, 0)

                context_frames.append(combined.iloc[start:stop].copy())
                remaining_classes.remove(flag)

            trailing_context = combined.iloc[-context_rows_per_class:].copy()

    if not context_frames:
        return pd.DataFrame(columns=use_columns + ["source_file"])
    return pd.concat(context_frames, ignore_index=True).sort_values("Time UTC").reset_index(drop=True)


In [ ]:
# Load a lightweight exploration sample directly from raw CSV files.
raw_csv_paths = sorted(Path(RAW_DATA_DIR).glob("*.csv"))
if not raw_csv_paths:
    raise FileNotFoundError(f"No CSV files found in {RAW_DATA_DIR}")

orientation_file_limit = ROW_FILE_LIMIT if ROW_FILE_LIMIT is not None else min(3, len(raw_csv_paths))
orientation_selected_paths = select_part_paths(
    raw_csv_paths,
    limit=orientation_file_limit,
    mode=PART_SELECTION_MODE,
)
ORIENTATION_ROWS_PER_FILE = 4000
orientation_columns = list(
    dict.fromkeys(
        [
            "Time UTC",
            TARGET_FLAG,
            *OPTIONAL_QC_COLUMNS,
            PLOT_MEASUREMENT_COLUMN,
            PLOT_SECONDARY_COLUMN,
            *MEASUREMENT_COLUMNS,
            "Depth (m)",
        ]
    )
)

exploration_df = load_raw_scalar_sample(
    orientation_selected_paths,
    sample_rows_per_file=ORIENTATION_ROWS_PER_FILE,
    columns=orientation_columns,
)

flag_context_df = load_raw_flag_context_sample(
    orientation_selected_paths,
    target_flag=TARGET_FLAG,
    classes=FLAG_EXAMPLE_CLASSES,
    context_rows_per_class=BASE_FLAG_EXAMPLE_POINTS_PER_PANEL,
    columns=orientation_columns,
)

print(
    {
        "orientation_raw_data_dir": RAW_DATA_DIR,
        "orientation_selected_files": len(orientation_selected_paths),
        "orientation_rows_loaded": len(exploration_df),
        "flag_context_rows_loaded": len(flag_context_df),
        "ORIENTATION_ROWS_PER_FILE": ORIENTATION_ROWS_PER_FILE,
        "orientation_preview_columns": orientation_columns,
    }
)


In [ ]:
# Look at a few rows so the dataset description is tied to the actual table.
preview_candidates = [
    "Time UTC",
    PLOT_MEASUREMENT_COLUMN,
    PLOT_SECONDARY_COLUMN,
    "Depth (m)",
    TARGET_FLAG,
    *OPTIONAL_QC_COLUMNS,
]
preview_columns = [column for column in preview_candidates if column in exploration_df.columns]
display(exploration_df[preview_columns].head(8))
print(
    {
        "time_start": exploration_df["Time UTC"].min().isoformat(),
        "time_end": exploration_df["Time UTC"].max().isoformat(),
        "preview_columns": preview_columns,
    }
)


### Seeing The QC Flags In Context

Before training anything, it helps to look at the time series directly.

The plots below show local windows centred on representative QC flags pulled directly from the raw CSV files. This is useful because:

- it connects the abstract label numbers to real sensor behaviour,
- it shows that some problematic labels appear in coherent stretches rather than isolated single rows,
- it reminds you that the model is trying to learn from patterns in time, not from labels in isolation.

If you want to make these panels wider or more annotated, the next config cell exposes:

- `FLAG_EXAMPLE_POINTS_PER_PANEL`
- `FLAG_EXAMPLE_REGION_ALPHA`
- `FLAG_EXAMPLE_SHOW_POINTS`
- `FLAG_EXAMPLE_CLASSES`


In [ ]:
FLAG_EXAMPLE_POINTS_PER_PANEL = BASE_FLAG_EXAMPLE_POINTS_PER_PANEL
FLAG_EXAMPLE_REGION_ALPHA = 0.18
FLAG_EXAMPLE_SHOW_POINTS = False

print(
    {
        "FLAG_EXAMPLE_POINTS_PER_PANEL": FLAG_EXAMPLE_POINTS_PER_PANEL,
        "FLAG_EXAMPLE_REGION_ALPHA": FLAG_EXAMPLE_REGION_ALPHA,
        "FLAG_EXAMPLE_SHOW_POINTS": FLAG_EXAMPLE_SHOW_POINTS,
        "FLAG_EXAMPLE_CLASSES": FLAG_EXAMPLE_CLASSES,
    }
)


In [ ]:
# Plot representative local windows for the main QC flag values we see in this dataset.
timeseries_figure, timeseries_examples = plot_flag_examples(
    flag_context_df,
    target_flag=TARGET_FLAG,
    measurement_column=PLOT_MEASUREMENT_COLUMN,
    secondary_column=PLOT_SECONDARY_COLUMN,
    points_per_panel=FLAG_EXAMPLE_POINTS_PER_PANEL,
    classes=FLAG_EXAMPLE_CLASSES,
    good_labels=GOOD_LABELS,
    region_alpha=FLAG_EXAMPLE_REGION_ALPHA,
    show_flag_points=FLAG_EXAMPLE_SHOW_POINTS,
)
plt.show()
display(timeseries_examples)


## Part 2 — Data Preparation and Caching

This part explains how to turn raw ONC CSV exports into typed parquet tables that are much easier to reuse in ML experiments.


### 🧹 How We Prepare The Data

This is the part you can reuse most directly on your own data.

No matter which scalar dataset you start from, the same four questions show up:

1. what should count as one training example,
2. which fields are essential,
3. how should the raw values be cleaned and typed,
4. what saved format will be fast to reuse later.

In this notebook the raw preparation is handled by `scripts/prepare_scalar_session1_data.py`. We run that once up front because the original ONC CSV exports are great for provenance, but awkward for repeated ML experiments.


### Step-By-Step: Turning Raw Data Into ML-Ready Data

A good preparation pipeline is usually simpler than it first sounds.

**1. Decide what one example means**

For this notebook, one cleaned row is the basic reusable unit. Later sections can regroup those rows into windows when a model needs more temporal context.

**2. Keep an explicit list of the fields you need**

For scalar time series, that usually means:

- a reliable timestamp,
- one or more measurement columns,
- the target QC flag,
- any extra context columns,
- and an identifier such as `source_file`.

**3. Clean and type the raw values**

In our prep script we:

- locate the real ONC header row and skip the metadata lines above it,
- parse `Time UTC` into typed datetimes,
- convert measurement and QC columns into numeric values,
- drop unusable timestamps,
- sort by time,
- and remember which file each row came from.

**4. Save reusable artifacts**

We keep two views of the same cleaned data:

- **row-level parquet**: one row per timestamp, useful for tabular models and sequence building,
- **window-summary parquet**: one row per fixed window, useful when you want short stretches summarized into one feature vector.

That is the core pattern to reuse on your own data: define the unit, clean it carefully, and save it in a form that later experiments can read quickly.


---


### 📦 Optional Step: Build The Parquet Cache From Raw CSV Files

Raw CSV is still the source of truth, but it is a poor format for repeated ML work. These ONC files include a metadata block before the real table, need timestamp and numeric parsing, and are slower to reread than a typed columnar format.

So we do one preparation step, save parquet, and reuse that cache for the rest of the notebook.

The script that performs this conversion is:

- `scripts/prepare_scalar_session1_data.py`

If you want to understand or adapt the preparation logic for your own data, that is the file to read.

Any cache that this notebook generates is written into the runtime output area, not back into the shared FIR cache.

The next cells let us either:

- rebuild the cache from raw CSV, or
- inspect the cache that already exists.


In [ ]:
RUN_RAW_PREP = False
FORCE_REBUILD_CACHE = False
PREP_MAX_FILES = None
PREP_SAMPLE_ROWS = DEFAULT_PREP_SAMPLE_ROWS
PREP_WINDOW_SIZE = 256
PREP_MERGE_TOLERANCE_SECONDS = 5
PREP_CACHE_ROOT = str(RUNTIME_CACHE_DIR)
PREP_CACHE_BUNDLE_NAME = CACHE_BUNDLE_NAME
PREP_PRIMARY_DEVICE = PRIMARY_DEVICE
PREP_ISSUE_LABELS = ISSUE_LABELS.copy()
PREP_MEASUREMENT_COLUMNS = MEASUREMENT_COLUMNS.copy()

print(
    {
        "RUN_RAW_PREP": RUN_RAW_PREP,
        "FORCE_REBUILD_CACHE": FORCE_REBUILD_CACHE,
        "PREP_MAX_FILES": PREP_MAX_FILES,
        "PREP_SAMPLE_ROWS": PREP_SAMPLE_ROWS,
        "PREP_WINDOW_SIZE": PREP_WINDOW_SIZE,
        "PREP_MERGE_TOLERANCE_SECONDS": PREP_MERGE_TOLERANCE_SECONDS,
        "PREP_CACHE_ROOT": PREP_CACHE_ROOT,
        "PREP_CACHE_BUNDLE_NAME": PREP_CACHE_BUNDLE_NAME,
        "PREP_PRIMARY_DEVICE": PREP_PRIMARY_DEVICE,
        "PREP_ISSUE_LABELS": PREP_ISSUE_LABELS,
        "PREP_MEASUREMENT_COLUMNS": PREP_MEASUREMENT_COLUMNS,
    }
)


In [ ]:
# Build the command that calls scripts/prepare_scalar_session1_data.py with the
# dataset-specific settings chosen earlier in the notebook.
raw_data_path = Path(RAW_DATA_DIR)
cache_root_path = Path(PREP_CACHE_ROOT)
prep_script_path = NOTEBOOK_ROOT / "scripts" / "prepare_scalar_session1_data.py"
existing_cache_paths = choose_cache_paths(
    [cache_root_path, Path(READ_CACHE_DIR)],
    cache_stem=PREP_CACHE_BUNDLE_NAME,
)
cache_metadata_path = existing_cache_paths.metadata_path

prep_command = [
    sys.executable,
    str(prep_script_path),
    "--data-root",
    str(raw_data_path),
    "--cache-root",
    str(cache_root_path),
    "--cache-stem",
    PREP_CACHE_BUNDLE_NAME,
    "--target-flag",
    TARGET_FLAG,
    "--primary-device",
    PREP_PRIMARY_DEVICE,
    "--window-size",
    str(PREP_WINDOW_SIZE),
    "--merge-tolerance-seconds",
    str(PREP_MERGE_TOLERANCE_SECONDS),
]
if PREP_MAX_FILES is not None:
    prep_command.extend(["--max-files", str(PREP_MAX_FILES)])
if PREP_SAMPLE_ROWS is not None:
    prep_command.extend(["--sample-rows", str(PREP_SAMPLE_ROWS)])
for issue_label in PREP_ISSUE_LABELS:
    prep_command.extend(["--issue-label", str(issue_label)])
for measurement_column in PREP_MEASUREMENT_COLUMNS:
    prep_command.extend(["--measurement-column", measurement_column])

missing_cache = not cache_metadata_path.exists()
should_run_prep = (
    FORCE_REBUILD_CACHE
    or RUN_RAW_PREP
    or (AUTO_BUILD_MISSING_CACHE and missing_cache)
)
print(
    {
        "should_run_prep": should_run_prep,
        "cache_exists": cache_metadata_path.exists(),
        "auto_build_missing_cache": AUTO_BUILD_MISSING_CACHE,
    }
)
print("Prep command:")
print(" ".join(prep_command))
print(
    {
        "prep_output_cache_dir": str(cache_root_path),
        "requested_cache_bundle_name": PREP_CACHE_BUNDLE_NAME,
        "resolved_existing_cache_stem": existing_cache_paths.stem,
        "resolved_existing_cache_dir": str(existing_cache_paths.root),
    }
)

if should_run_prep:
    completed = subprocess.run(prep_command, cwd=NOTEBOOK_ROOT, check=True, text=True, capture_output=True)
    if completed.stdout:
        print(completed.stdout)
    if completed.stderr:
        print(completed.stderr)
elif missing_cache:
    print(
        "No prepared cache was found for this dataset profile. "
        "This notebook will not auto-build it in the current environment because that full prep can exceed a typical notebook memory limit. "
        "Point READ_CACHE_DIR at a prebuilt cache or set RUN_RAW_PREP = True with a smaller PREP_SAMPLE_ROWS value if you want a reduced demo cache."
    )
else:
    print("Skipping cache rebuild because a prepared runtime cache already exists. Set RUN_RAW_PREP = True to rebuild it.")


### A Quick Guide To The Prep Script

You do not need to memorize the whole file, but these functions are worth knowing:

- `locate_header(...)`: finds the real table header below the ONC metadata block.
- `read_scalar_csv(...)`: loads one CSV, types the timestamp and numeric columns, and sorts the rows.
- `discover_available_columns(...)`: scans the candidate files first so schema discovery still works even if you later limit the file count.
- `select_time_aligned_paths(...)`: keeps reduced multi-device selections aligned by filename time range.
- `choose_measurement_columns(...)`: decides which numeric signals should be carried into the cache.
- `build_window_features(...)`: creates the one-row-per-window summary table used later for clustering.
- `main()`: orchestrates the full workflow and writes metadata about the generated cache.

A simple reading strategy is:

1. start with `main()` to see the overall flow,
2. jump to `read_scalar_csv(...)` for row cleaning and typing,
3. then look at `build_window_features(...)` for the window-summary cache.


---


### ⚡ Why The Parquet Cache Helps

The point of the cache is not just convenience. It changes the workflow:

- raw CSV is for ingestion and provenance,
- parquet is for repeated analysis and model training.

In the summary below:

- `parsed columns` means every column recovered from a representative raw ONC table,
- `row columns used in notebook` means the smaller row-level subset we load for supervised modeling,
- `window columns used in notebook` means the summary-feature subset we load from the window cache for clustering.

The **row parquet cache** keeps one row per timestamp.

The **window-summary parquet** keeps one row per fixed time window, with summary features such as means, standard deviations, and issue rates. That makes it useful when we want to reason about short stretches of time rather than isolated rows.

Some dataset profiles use that window summary directly for k-means. Others, such as spike-driven turbidity examples, intentionally switch k-means to row-level features so short events are not averaged away.

The next cells make those benefits visible with concrete measurements.


In [ ]:
def directory_size_bytes(path: Path, pattern: str) -> int:
    return sum(file_path.stat().st_size for file_path in path.glob(pattern) if file_path.is_file())

active_cache_paths = choose_cache_paths([Path(RUNTIME_CACHE_DIR), Path(READ_CACHE_DIR)], cache_stem=CACHE_BUNDLE_NAME)
cache_metadata_path = active_cache_paths.metadata_path
row_cache_dir = active_cache_paths.row_level_dir
window_cache_path = active_cache_paths.window_cache_path

raw_files = sorted(Path(RAW_DATA_DIR).glob("*.csv"))
row_cache_parts = sorted(row_cache_dir.glob("*.parquet"))
if not raw_files:
    raise FileNotFoundError(f"No raw CSV files found in {RAW_DATA_DIR}")
if not row_cache_parts:
    raise FileNotFoundError(f"No parquet parts found in {row_cache_dir}")
if not cache_metadata_path.exists():
    raise FileNotFoundError(f"No cache metadata found at {cache_metadata_path}")

cache_metadata = json.loads(cache_metadata_path.read_text())
available_row_columns = cache_metadata.get("row_columns", [])
available_window_columns = cache_metadata.get("window_columns", [])

if TARGET_FLAG not in available_row_columns:
    raise KeyError(
        f"Target flag {TARGET_FLAG!r} is not present in the prepared row-level cache. "
        "Choose a different dataset profile or override TARGET_FLAG."
    )

requested_measurement_columns = MEASUREMENT_COLUMNS.copy()
requested_optional_qc_columns = OPTIONAL_QC_COLUMNS.copy()
requested_plot_measurement_column = PLOT_MEASUREMENT_COLUMN
requested_plot_secondary_column = PLOT_SECONDARY_COLUMN

MEASUREMENT_COLUMNS = [column for column in requested_measurement_columns if column in available_row_columns]
OPTIONAL_QC_COLUMNS = [column for column in requested_optional_qc_columns if column in available_row_columns]

missing_measurement_columns = [
    column for column in requested_measurement_columns if column not in MEASUREMENT_COLUMNS
]
missing_optional_qc_columns = [
    column for column in requested_optional_qc_columns if column not in OPTIONAL_QC_COLUMNS
]

if not MEASUREMENT_COLUMNS:
    raise ValueError(
        "None of the requested measurement columns are present in the prepared cache. "
        "Check the dataset profile or use manual overrides."
    )

if requested_plot_measurement_column in MEASUREMENT_COLUMNS:
    PLOT_MEASUREMENT_COLUMN = requested_plot_measurement_column
else:
    PLOT_MEASUREMENT_COLUMN = MEASUREMENT_COLUMNS[0]

remaining_plot_candidates = [
    column for column in MEASUREMENT_COLUMNS if column != PLOT_MEASUREMENT_COLUMN
]
if requested_plot_secondary_column in remaining_plot_candidates:
    PLOT_SECONDARY_COLUMN = requested_plot_secondary_column
elif remaining_plot_candidates:
    PLOT_SECONDARY_COLUMN = remaining_plot_candidates[0]
else:
    PLOT_SECONDARY_COLUMN = PLOT_MEASUREMENT_COLUMN

WINDOW_FEATURE_COLUMNS = [
    f"{column}_{stat}"
    for column in MEASUREMENT_COLUMNS
    for stat in ("mean", "std")
    if f"{column}_{stat}" in available_window_columns
]
ROW_USE_COLUMNS = list(
    dict.fromkeys(["Time UTC", "source_file", TARGET_FLAG, *OPTIONAL_QC_COLUMNS, *MEASUREMENT_COLUMNS])
)
WINDOW_USE_COLUMNS = [
    "window_start",
    "window_end",
    "source_file",
    "issue_rate",
    *WINDOW_FEATURE_COLUMNS,
]

show_setup_json(
    {
        "requested_measurement_columns": requested_measurement_columns,
        "active_measurement_columns": MEASUREMENT_COLUMNS,
        "missing_measurement_columns": missing_measurement_columns,
        "requested_optional_qc_columns": requested_optional_qc_columns,
        "active_optional_qc_columns": OPTIONAL_QC_COLUMNS,
        "missing_optional_qc_columns": missing_optional_qc_columns,
        "plot_measurement_column": PLOT_MEASUREMENT_COLUMN,
        "plot_secondary_column": PLOT_SECONDARY_COLUMN,
    }
)

representative_info = cache_metadata["processed_files"][0]
representative_raw_path = Path(RAW_DATA_DIR) / representative_info["source_file"]
representative_parquet_path = row_cache_dir / Path(representative_info["row_level_part"]).name
header_line_number, parsed_columns = locate_header(representative_raw_path)

with representative_raw_path.open("r", encoding="utf-8") as handle:
    raw_header_preview = [next(handle).rstrip("\n") for _ in range(8)]

raw_total_bytes = directory_size_bytes(Path(RAW_DATA_DIR), "*.csv")
parquet_total_bytes = directory_size_bytes(row_cache_dir, "*.parquet")
window_cache_bytes = window_cache_path.stat().st_size if window_cache_path.exists() else 0

cache_comparison = pd.DataFrame(
    [
        {"artifact": "Raw CSV files", "bytes": raw_total_bytes},
        {"artifact": "Row parquet cache", "bytes": parquet_total_bytes},
        {"artifact": "Window parquet cache", "bytes": window_cache_bytes},
    ]
)
cache_comparison["megabytes"] = cache_comparison["bytes"] / (1024 ** 2)
cache_comparison["gigabytes"] = cache_comparison["bytes"] / (1024 ** 3)
display(cache_comparison)

column_sets = pd.DataFrame(
    [
        {
            "column_set": "Parsed columns from raw CSV",
            "count": len(parsed_columns),
            "examples": ", ".join(parsed_columns[:6]) + (" ..." if len(parsed_columns) > 6 else ""),
        },
        {
            "column_set": "Row columns loaded in this notebook",
            "count": len(cache_metadata.get("row_columns", ROW_USE_COLUMNS)),
            "examples": ", ".join(cache_metadata.get("row_columns", ROW_USE_COLUMNS)[:6]) + (" ..." if len(cache_metadata.get("row_columns", ROW_USE_COLUMNS)) > 6 else ""),
        },
        {
            "column_set": "Window columns loaded in this notebook",
            "count": len(cache_metadata.get("window_columns", WINDOW_USE_COLUMNS)),
            "examples": ", ".join(cache_metadata.get("window_columns", WINDOW_USE_COLUMNS)[:6]) + (" ..." if len(cache_metadata.get("window_columns", WINDOW_USE_COLUMNS)) > 6 else ""),
        },
    ]
)
display(column_sets)

print(
    {
        "raw_file_count": len(raw_files),
        "parquet_part_count": len(row_cache_parts),
        "metadata_lines_before_table": header_line_number - 1,
        "parsed_column_count": len(parsed_columns),
        "row_columns_used_in_notebook": len(cache_metadata.get("row_columns", ROW_USE_COLUMNS)),
        "window_columns_used_in_notebook": len(cache_metadata.get("window_columns", WINDOW_USE_COLUMNS)),
        "full_row_count": cache_metadata["row_count"],
        "full_window_count": cache_metadata["window_count"],
        "cache_bundle_name": cache_metadata.get("cache_stem", CACHE_BUNDLE_NAME),
    }
)
print("Representative raw CSV preview:")
print("\n".join(raw_header_preview))

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
axes[0].bar(cache_comparison["artifact"], cache_comparison["gigabytes"], color=["#5B7C99", "#2A9D8F", "#E9C46A"])
axes[0].set_ylabel("Size on disk (GB)")
axes[0].set_title("Disk footprint of raw and cached artifacts")
axes[0].tick_params(axis="x", rotation=20)

axes[1].bar(
    ["Parsed columns", "Row cols used here", "Window cols used here"],
        [
            len(parsed_columns),
            len(cache_metadata.get("row_columns", ROW_USE_COLUMNS)),
            len(cache_metadata.get("window_columns", WINDOW_USE_COLUMNS)),
        ],
        color=["#7A6FF0", "#3A86FF", "#00A896"],
)
axes[1].set_ylabel("Column count")
axes[1].set_title("Parquet lets us load only the columns we need")
plt.tight_layout()
plt.show()


In [ ]:
benchmark_scope_label = (
    f"the first {RAW_BENCHMARK_SAMPLE_ROWS:,} rows"
    if RAW_BENCHMARK_SAMPLE_ROWS is not None
    else "one full representative file"
)

csv_start = perf_counter()
csv_frame = read_scalar_csv(representative_raw_path, sample_rows=RAW_BENCHMARK_SAMPLE_ROWS)
csv_elapsed = perf_counter() - csv_start

parquet_full_start = perf_counter()
parquet_full_frame = read_parquet_head(
    representative_parquet_path,
    max_rows=RAW_BENCHMARK_SAMPLE_ROWS,
)
parquet_full_elapsed = perf_counter() - parquet_full_start

parquet_pruned_start = perf_counter()
parquet_pruned_frame = read_parquet_head(
    representative_parquet_path,
    columns=ROW_USE_COLUMNS,
    max_rows=RAW_BENCHMARK_SAMPLE_ROWS,
)
parquet_pruned_elapsed = perf_counter() - parquet_pruned_start

benchmark_summary = pd.DataFrame(
    [
        {
            "read_path": "Raw CSV parse",
            "seconds": csv_elapsed,
            "rows_loaded": len(csv_frame),
            "memory_mb": csv_frame.memory_usage(deep=True).sum() / (1024 ** 2),
        },
        {
            "read_path": "Parquet full load",
            "seconds": parquet_full_elapsed,
            "rows_loaded": len(parquet_full_frame),
            "memory_mb": parquet_full_frame.memory_usage(deep=True).sum() / (1024 ** 2),
        },
        {
            "read_path": "Parquet column-pruned load",
            "seconds": parquet_pruned_elapsed,
            "rows_loaded": len(parquet_pruned_frame),
            "memory_mb": parquet_pruned_frame.memory_usage(deep=True).sum() / (1024 ** 2),
        },
    ]
)
benchmark_summary["rows_per_second"] = benchmark_summary["rows_loaded"] / benchmark_summary["seconds"].clip(lower=1e-9)
display(benchmark_summary)

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
axes[0].bar(benchmark_summary["read_path"], benchmark_summary["seconds"], color=["#E76F51", "#2A9D8F", "#457B9D"])
axes[0].set_ylabel("Elapsed time (s)")
axes[0].set_title(f"Read time for {benchmark_scope_label}")
axes[0].tick_params(axis="x", rotation=15)

axes[1].bar(benchmark_summary["read_path"], benchmark_summary["rows_per_second"], color=["#E76F51", "#2A9D8F", "#457B9D"])
axes[1].set_ylabel("Rows per second")
axes[1].set_title("Typed parquet is faster for repeated analysis reads")
axes[1].tick_params(axis="x", rotation=15)
plt.tight_layout()
plt.show()

print(
    {
        "benchmark_scope": benchmark_scope_label,
        "csv_seconds": round(csv_elapsed, 4),
        "parquet_full_seconds": round(parquet_full_elapsed, 4),
        "parquet_pruned_seconds": round(parquet_pruned_elapsed, 4),
        "full_file_speedup_ratio": round(csv_elapsed / max(parquet_full_elapsed, 1e-9), 2),
        "column_pruned_speedup_ratio": round(csv_elapsed / max(parquet_pruned_elapsed, 1e-9), 2),
    }
)


### Choosing A Useful Storage Format

Parquet is a strong fit for **typed tabular data** and many **1D time-series** workflows, but it is not the only good option.

A simple rule of thumb is: store the data in a format that matches the natural unit of your experiment.

- **Scalar tables / 1D time series**: parquet is often ideal because you can read only the columns you need.
- **Sequences that will be windowed later**: row-level parquet still works well because you can build windows on demand.
- **Images or 2D products** such as spectrograms: a folder of image files can be perfectly reasonable if each file is already one example.
- **Large 2D or 3D numeric arrays**: formats like NumPy `.npy` / `.npz`, HDF5, Zarr, or NetCDF are often a better fit than CSV.
- **Text examples**: JSONL and parquet are common because each row can hold one document plus labels and metadata.

So the real lesson is not "always use parquet." It is "pick a reusable format that preserves the structure of your examples and is efficient for the experiments you plan to run."


## Part 3 — Feature Engineering and Data Loading

This part uses the prepared parquet cache to inspect the full dataset, compare split strategies, and build the benchmark train / validation / test datasets that the models will use.


### What This Part Is For

Up to this point, the data is **clean and stored efficiently**, but we have not yet turned it into an honest ML benchmark.

In this part you will:

- inspect the full selected parquet data,
- verify how reviewed flags are distributed through time,
- compare split strategies on the full reviewed rows,
- keep validation/test fixed,
- and only then choose how much of the training split to keep.

This is an important transition point. A lot of ML performance and clarity comes from how you define the **dataset and evaluation protocol**, not just from which model you choose afterward.


---


In [ ]:
# Load the metadata that was written during cache preparation.
persistent_cache_dir = Path(READ_CACHE_DIR)
runtime_cache_dir = Path(RUNTIME_CACHE_DIR)
active_cache_paths = choose_cache_paths([runtime_cache_dir, persistent_cache_dir], cache_stem=CACHE_BUNDLE_NAME)
CACHE_DIR = str(active_cache_paths.root)
ROW_CACHE_DIR = str(active_cache_paths.row_level_dir)
WINDOW_CACHE_PATH = str(active_cache_paths.window_cache_path)
METADATA_PATH = str(active_cache_paths.metadata_path)

row_cache_dir = active_cache_paths.row_level_dir
window_cache_path = active_cache_paths.window_cache_path
metadata_path = active_cache_paths.metadata_path

if not metadata_path.exists():
    raise FileNotFoundError(
        f"Missing cache metadata at {metadata_path}. Run scripts/prepare_scalar_session1_data.py first."
    )

metadata = json.loads(metadata_path.read_text())
part_paths = sorted(row_cache_dir.glob("*.parquet"))
if not part_paths:
    raise FileNotFoundError(f"No parquet parts found in {row_cache_dir}")

benchmark_part_selection_mode = globals().get("BENCHMARK_PART_SELECTION_MODE", PART_SELECTION_MODE)
benchmark_row_file_limit = globals().get("BENCHMARK_ROW_FILE_LIMIT", ROW_FILE_LIMIT)

# Choose the parquet parts that define the benchmark dataset for this run.
selected_paths = select_part_paths(
    part_paths,
    limit=benchmark_row_file_limit,
    mode=benchmark_part_selection_mode,
)
part_to_source = {
    Path(file_info["row_level_part"]).name: file_info["source_file"]
    for file_info in metadata["processed_files"]
}
part_to_row_count = {
    Path(file_info["row_level_part"]).name: int(file_info["row_count"])
    for file_info in metadata["processed_files"]
}
source_to_row_part = {
    file_info["source_file"]: file_info["row_level_part"]
    for file_info in metadata["processed_files"]
}
selected_source_files = {part_to_source[path.name] for path in selected_paths}

print(
    {
        "active_cache_dir": str(active_cache_paths.root),
        "cache_bundle_name": CACHE_BUNDLE_NAME,
        "persistent_cache_dir": str(persistent_cache_dir),
        "runtime_output_root": str(RUNTIME_OUTPUT_ROOT),
        "model_output_dir": str(MODEL_OUTPUT_DIR),
        "benchmark_part_selection_mode": benchmark_part_selection_mode,
        "benchmark_row_file_limit": benchmark_row_file_limit,
    }
)


In [ ]:
# Summarize the overall cache before loading row samples.
cache_summary = {
    "full_row_count": metadata["row_count"],
    "full_window_count": metadata["window_count"],
    "processed_file_count": metadata["processed_file_count"],
    "issue_fraction": round(float(metadata["issue_fraction"]), 4),
    "task_mode": TASK_MODE,
}
print(cache_summary)

selected_file_summary = pd.DataFrame(metadata["processed_files"])
selected_file_summary = selected_file_summary[selected_file_summary["source_file"].isin(selected_source_files)].reset_index(drop=True)
selected_file_summary = selected_file_summary[["source_file", "row_count", "time_start", "time_end"]].copy()
selected_file_summary["source_file"] = selected_file_summary["source_file"].str.replace(
    r"_\d{8}T\d{6}Z_\d{8}T\d{6}Z-NaN\.csv$",
    "",
    regex=True,
)
display(selected_file_summary.head(5))


## 🚚 Use The Prepared Cache To Understand The Full Dataset

Parquet gives us fast access to the prepared dataset, so we can inspect the full selected data directly with only the columns we need.

The flow in this part is:

- inspect the full selected parquet data with only the columns we need,
- compare split strategies on the full reviewed rows,
- check whether validation contains enough flagged data to be useful,
- then choose a **train-only** subset strategy if you want a smaller run.

`DATA_FRACTION` mainly affects the later training budget and a few heavier visualizations. It does not determine which reviewed rows belong to validation or test.


## 🎚️ Dataset Scope For This Benchmark

These settings control **how much of the prepared parquet dataset we load for the benchmark setup cells**, and how detailed the early summary plots should be.

What each one does:

- `BENCHMARK_PART_SELECTION_MODE`: how to choose parquet parts when we are not using all of them. A spread-style choice gives a broad sweep through time, while a sequential choice keeps the earliest parts.
- `BENCHMARK_ROW_FILE_LIMIT`: how many row-level parquet parts to include in the benchmark setup. Lower values make the next few cells much faster, but show less of the full dataset.
- `SUMMARY_TIME_BIN_COUNT`: how many time bins to use in the label-through-time summaries. More bins show finer detail; fewer bins give a simpler overview.
- `KMEANS_WINDOW_LIMIT`: how many windowed examples to load for the k-means section later. Lower values keep clustering faster and lighter on memory.

These settings live here, right beside the cells they affect, so you can change them and immediately rerun the next few cells to see what changed.


In [ ]:
# Decide how parquet parts are chosen if we are not loading the entire row-level cache.
BENCHMARK_PART_SELECTION_MODE = PART_SELECTION_MODE

# Limit the number of row-level parquet parts used in the benchmark setup cells.
BENCHMARK_ROW_FILE_LIMIT = ROW_FILE_LIMIT

# Control how finely we slice time in the label-through-time summary charts.
SUMMARY_TIME_BIN_COUNT = 24

# Keep the later k-means section responsive by capping how many windows it loads.
KMEANS_WINDOW_LIMIT = None if DATA_FRACTION >= 0.999 else max(2000, int(5000 * DATA_FRACTION))

print(
    {
        "BENCHMARK_PART_SELECTION_MODE": BENCHMARK_PART_SELECTION_MODE,
        "BENCHMARK_ROW_FILE_LIMIT": BENCHMARK_ROW_FILE_LIMIT,
        "SUMMARY_TIME_BIN_COUNT": SUMMARY_TIME_BIN_COUNT,
        "KMEANS_WINDOW_LIMIT": KMEANS_WINDOW_LIMIT,
    }
)


### Small Helper For Validation Adequacy

Before training anything, we want to know whether validation contains enough flagged data to make model selection meaningful.

The helper below summarizes that question for any candidate split.


In [ ]:
def summarize_issue_adequacy(
    split_frames: dict[str, pd.DataFrame],
    *,
    label_column: str,
    issue_labels: list[int] | tuple[int, ...],
    min_issue_rows: int,
    min_issue_share_pct: float,
    min_rows_per_issue_label: int,
) -> pd.DataFrame:
    adequacy_rows = []
    normalized_issue_labels = [int(label) for label in issue_labels]

    for split_name, frame in split_frames.items():
        labels = pd.to_numeric(frame[label_column], errors="coerce").dropna().astype(int)
        total_rows = int(len(labels))
        label_counts = labels.value_counts().sort_index()
        issue_counts = label_counts.reindex(normalized_issue_labels, fill_value=0).astype(int)
        issue_rows = int(issue_counts.sum())
        issue_share_pct = round(100 * issue_rows / total_rows, 2) if total_rows else 0.0
        per_label_ok = all(int(issue_counts.get(label, 0)) >= min_rows_per_issue_label for label in normalized_issue_labels)

        adequacy_rows.append(
            {
                "split": split_name,
                "rows": total_rows,
                "issue_rows": issue_rows,
                "issue_share_pct": issue_share_pct,
                "meets_issue_row_floor": issue_rows >= min_issue_rows,
                "meets_issue_share_floor": issue_share_pct >= min_issue_share_pct,
                "meets_per_issue_label_floor": per_label_ok,
                "adequate_for_model_selection": (
                    issue_rows >= min_issue_rows
                    and issue_share_pct >= min_issue_share_pct
                    and per_label_ok
                ),
                **{f"flag_{label}_count": int(issue_counts.get(label, 0)) for label in normalized_issue_labels},
            }
        )

    return pd.DataFrame(adequacy_rows).set_index("split")


def build_split_strategy_tables(
    strategy_metric_rows: list[dict[str, object]],
    strategy_adequacy_frames: list[pd.DataFrame],
) -> tuple[pd.DataFrame, pd.DataFrame]:
    summary_frame = pd.DataFrame(strategy_metric_rows).set_index("strategy").copy()
    summary_frame = summary_frame.rename(
        columns={
            "rows": "reviewed_rows_total",
            "train_rows": "train_rows",
            "validation_rows": "validation_rows",
            "test_rows": "test_rows",
            "validation_issue_rows": "validation_issue_rows",
            "validation_issue_share_pct": "validation_issue_share_pct",
            "validation_adequate": "validation_ok",
            "max_pairwise_total_variation": "max_split_gap",
            "mean_pairwise_total_variation": "mean_split_gap",
        }
    )
    if "validation_ok" in summary_frame.columns:
        summary_frame["validation_ok"] = summary_frame["validation_ok"].map({True: "yes", False: "no"})

    detail_frame = pd.concat(strategy_adequacy_frames, ignore_index=True).copy()
    if detail_frame.empty:
        return summary_frame, detail_frame

    detail_frame["adequate_for_model_selection"] = detail_frame["adequate_for_model_selection"].map(
        {True: "yes", False: "no"}
    )
    detail_frame = detail_frame.rename(
        columns={
            "adequate_for_model_selection": "adequate",
        }
    )
    pivot_frame = (
        detail_frame.set_index(["strategy", "split"])[["rows", "issue_rows", "issue_share_pct", "adequate"]]
        .unstack("split")
        .swaplevel(0, 1, axis=1)
        .sort_index(axis=1, level=0)
    )
    pivot_frame.columns = [f"{split}_{metric}" for split, metric in pivot_frame.columns]
    pivot_frame = pivot_frame.rename(
        columns={
            "train_rows": "train_rows",
            "train_issue_rows": "train_issue_rows",
            "train_issue_share_pct": "train_issue_share_pct",
            "train_adequate": "train_ok",
            "validation_rows": "validation_rows",
            "validation_issue_rows": "validation_issue_rows",
            "validation_issue_share_pct": "validation_issue_share_pct",
            "validation_adequate": "validation_ok",
            "test_rows": "test_rows",
            "test_issue_rows": "test_issue_rows",
            "test_issue_share_pct": "test_issue_share_pct",
            "test_adequate": "test_ok",
        }
    )
    return summary_frame, pivot_frame


def split_frame_by_strategy(
    frame: pd.DataFrame,
    *,
    train_fraction: float,
    validation_fraction: float,
    strategy: str,
    block_rows: int,
    issue_column: str = "issue",
    target_column: str | None = None,
    issue_labels: list[int] | tuple[int, ...] | None = None,
    episode_context_rows: int = 0,
    episode_merge_gap_rows: int = 0,
    purge_gap_rows: int = 0,
) -> dict[str, pd.DataFrame]:
    import inspect

    split_kwargs = {
        "train_fraction": train_fraction,
        "validation_fraction": validation_fraction,
        "strategy": strategy,
        "block_rows": block_rows,
        "issue_column": issue_column,
        "target_column": target_column,
        "issue_labels": issue_labels,
        "episode_context_rows": episode_context_rows,
        "episode_merge_gap_rows": episode_merge_gap_rows,
        "purge_gap_rows": purge_gap_rows,
    }
    supported_parameters = set(inspect.signature(split_frame_by_strategy_impl).parameters)

    if strategy == "episode_aware" and "episode_context_rows" not in supported_parameters:
        raise RuntimeError(
            "The active kernel is still using an older split helper that does not support "
            "episode-aware splitting. Restart the kernel or rerun the imports/helper cells near the "
            "top of Part 3, then try again."
        )

    filtered_kwargs = {
        key: value
        for key, value in split_kwargs.items()
        if key in supported_parameters
    }
    return split_frame_by_strategy_impl(frame, **filtered_kwargs)


In [ ]:
# Load a lightweight parquet view of the full selected dataset.
# We only need time, source, and target here to understand the dataset
# and compare split strategies honestly.
overview_columns = list(dict.fromkeys(["Time UTC", "source_file", TARGET_FLAG]))
dataset_label_df = load_full_row_level_frame(selected_paths, columns=overview_columns)
dataset_label_df[TARGET_FLAG] = pd.to_numeric(dataset_label_df[TARGET_FLAG], errors="coerce")
if "source_file" in dataset_label_df.columns:
    dataset_label_df["source_file"] = dataset_label_df["source_file"].astype("category")

reviewed_mask = session1_modeling.reviewed_label_mask(
    dataset_label_df[TARGET_FLAG],
    good_labels=GOOD_LABELS,
    issue_labels=ISSUE_LABELS,
)
reviewed_label_df = dataset_label_df.loc[reviewed_mask].copy().reset_index(drop=True)

selected_target_counts = dataset_label_df[TARGET_FLAG].dropna().astype(int).value_counts().sort_index()
selected_reviewed_counts = reviewed_label_df[TARGET_FLAG].dropna().astype(int).value_counts().sort_index()

if KMEANS_FEATURE_MODE == "window_summary":
    # Load the cached window summaries for clustering when averaging and variability are the useful signal.
    window_df = pd.read_parquet(window_cache_path, columns=WINDOW_USE_COLUMNS)
    window_df["window_start"] = pd.to_datetime(window_df["window_start"], utc=True)
    window_df["window_end"] = pd.to_datetime(window_df["window_end"], utc=True)
    window_df = (
        window_df[window_df["source_file"].isin(selected_source_files)]
        .sort_values("window_start")
        .reset_index(drop=True)
    )
    window_limit = KMEANS_WINDOW_LIMIT
    window_df = evenly_spaced_take(window_df, window_limit)
elif KMEANS_FEATURE_MODE == "row_level":
    # For row-level k-means we will build the feature frame later from the
    # benchmark modeling table, after the split and train-budget choices.
    window_df = pd.DataFrame()
    window_limit = None
else:
    raise ValueError(f"Unsupported KMEANS_FEATURE_MODE: {KMEANS_FEATURE_MODE}")

print(
    {
        "selected_parts": len(selected_paths),
        "selected_rows": len(dataset_label_df),
        "reviewed_rows": len(reviewed_label_df),
        "unreviewed_rows": int(len(dataset_label_df) - len(reviewed_label_df)),
        "loaded_windows": len(window_df),
        "data_fraction": DATA_FRACTION,
        "kmeans_feature_mode": KMEANS_FEATURE_MODE,
        "overview_columns_loaded": overview_columns,
        "window_columns_loaded": len(WINDOW_USE_COLUMNS) if KMEANS_FEATURE_MODE == "window_summary" else None,
        "window_limit": window_limit,
    }
)


In [ ]:
full_cache_counts = pd.Series({int(key): int(value) for key, value in metadata["target_distribution"].items()}).sort_index()
selected_dataset_counts = selected_target_counts.reindex(sorted(set(full_cache_counts.index).union(selected_target_counts.index)), fill_value=0)
reviewed_labels = sorted(set(GOOD_LABELS).union(ISSUE_LABELS))

distribution_counts = (
    pd.DataFrame(
        {
            "full_cache": full_cache_counts,
            "selected_dataset": selected_dataset_counts,
        }
    )
    .fillna(0)
    .astype(int)
)
distribution_shares = distribution_counts.div(distribution_counts.sum(axis=0), axis=1).fillna(0.0)

flag_meanings = pd.DataFrame(
    {
        "QC flag": [0, 1, 2, 3, 4, 6, 7, 8, 9],
        "Meaning": [
            "no QC",
            "good",
            "probably good",
            "probably bad",
            "bad",
            "bad down-sampling",
            "averaged",
            "interpolated",
            "missing / NaN",
        ],
    }
)
display(flag_meanings[flag_meanings["QC flag"].isin(distribution_counts.index.tolist())].reset_index(drop=True))

coverage_summary = pd.DataFrame(
    {
        "rows": [len(dataset_label_df), len(reviewed_label_df)],
    },
    index=["selected_dataset", "reviewed_only"],
)
coverage_summary["share_of_selected_pct"] = (
    100 * coverage_summary["rows"] / max(len(dataset_label_df), 1)
).round(2)
display(coverage_summary)

reviewed_summary = pd.DataFrame(index=reviewed_labels)
reviewed_summary["reviewed_count"] = selected_reviewed_counts.reindex(reviewed_labels, fill_value=0)
reviewed_summary["reviewed_share_pct"] = (
    100 * reviewed_summary["reviewed_count"] / max(int(reviewed_summary["reviewed_count"].sum()), 1)
).round(2)
reviewed_summary.index.name = "QC flag"
display(reviewed_summary)

share_plot = distribution_shares.T.copy()
share_plot.index = [
    f"full cache (n={int(distribution_counts['full_cache'].sum()):,})",
    f"selected dataset (n={int(distribution_counts['selected_dataset'].sum()):,})",
]
palette = [DEFAULT_FLAG_PALETTE.get(int(label), "#64748b") for label in share_plot.columns]

fig, ax = plt.subplots(figsize=(11.8, 4.2))
share_plot.plot(kind="barh", stacked=True, ax=ax, color=palette, width=0.62)
ax.set_xlim(0, 1)
ax.xaxis.set_major_formatter(PercentFormatter(1.0))
ax.set_xlabel("Share of rows")
ax.set_ylabel("")
ax.set_title(f"{TARGET_FLAG}: full cache vs the selected benchmark dataset")
ax.grid(axis="x", alpha=0.2)
ax.legend(title="QC flag", bbox_to_anchor=(1.01, 1.0), loc="upper left")
plt.tight_layout()
plt.show()

print(
    {
        "active_target": TARGET_FLAG,
        "full_cache_rows": int(distribution_counts["full_cache"].sum()),
        "selected_dataset_rows": int(len(dataset_label_df)),
        "reviewed_rows": int(len(reviewed_label_df)),
        "unreviewed_rows": int(len(dataset_label_df) - len(reviewed_label_df)),
    }
)


### Verify How The Flags Change Across Time

Before choosing a split strategy, look at **where the labels live on the time axis** in the full selected dataset.

This is one of the most important dataset checks you can do before any machine learning:

- if issue labels only appear in one short period, a strict contiguous split may put almost all of them into only one split,
- if the label mix changes gradually across time, that may reflect a real regime shift rather than a sampling mistake,
- if different flags appear in different time periods, you should expect the split strategy to change the difficulty of the task.

This is why dataset preparation is never just "convert files and start training." You also need to **verify the label distribution, the time coverage, and the assumptions behind your split** before trusting any model result.

The plots below are not model outputs. They are a dataset sanity check on the selected parquet data before we define train / validation / test.


In [ ]:
temporal_bin_count = min(SUMMARY_TIME_BIN_COUNT, max(8, len(selected_paths) * 2))
temporal_labels = [
    label
    for label in sorted(set(GOOD_LABELS).union(ISSUE_LABELS))
    if label in set(dataset_label_df[TARGET_FLAG].dropna().astype(int).unique().tolist())
]
temporal_counts, temporal_shares, temporal_summary = summarize_target_by_time_bin(
    dataset_label_df,
    time_column="Time UTC",
    label_column=TARGET_FLAG,
    bin_count=temporal_bin_count,
    labels=temporal_labels,
    good_labels=GOOD_LABELS,
    issue_labels=ISSUE_LABELS,
)

if not temporal_summary.empty:
    dominant_flags = []
    for time_bin in temporal_counts.index:
        bin_counts = temporal_counts.loc[time_bin]
        dominant_flags.append(int(bin_counts.idxmax()) if int(bin_counts.sum()) > 0 else None)
    temporal_summary["dominant_flag"] = dominant_flags
    temporal_summary["time_window"] = (
        temporal_summary["time_start"].dt.strftime("%Y-%m-%d %H:%M")
        + " -> "
        + temporal_summary["time_end"].dt.strftime("%Y-%m-%d %H:%M")
    )
    display(
        temporal_summary[
            ["time_window", "rows", "reviewed_rows", "unreviewed_rows", "reviewed_share_pct", "issue_rows", "issue_share_pct", "dominant_flag"]
        ].reset_index(drop=True)
    )

    temporal_plot_labels = temporal_summary["time_start"].dt.strftime("%m-%d\n%H:%M").tolist()
    x_positions = np.arange(len(temporal_summary))

    fig, (ax_top, ax_bottom) = plt.subplots(
        2,
        1,
        figsize=(13.5, 8.0),
        sharex=True,
        gridspec_kw={"height_ratios": [1.0, 1.5]},
    )

    ax_top.bar(
        x_positions,
        temporal_summary["reviewed_rows"],
        color="#cbd5e1",
        edgecolor="#94a3b8",
        width=0.82,
        label="reviewed rows in time bin",
    )
    ax_top.set_ylabel("Reviewed rows")
    ax_top.grid(axis="y", alpha=0.2)
    ax_top.set_title(f"{TARGET_FLAG}: where the reviewed flags live through time in the selected dataset")

    ax_top_issue = ax_top.twinx()
    ax_top_issue.plot(
        x_positions,
        temporal_summary["issue_share_pct"],
        color="#dc2626",
        marker="o",
        linewidth=2.0,
        label="issue share among reviewed rows (%)",
    )
    ax_top_issue.set_ylabel("Issue share (%)", color="#dc2626")
    ax_top_issue.tick_params(axis="y", colors="#dc2626")

    top_handles, top_labels = ax_top.get_legend_handles_labels()
    issue_handles, issue_labels = ax_top_issue.get_legend_handles_labels()
    ax_top.legend(top_handles + issue_handles, top_labels + issue_labels, bbox_to_anchor=(1.01, 1.0), loc="upper left")

    temporal_share_plot = temporal_shares.copy()
    temporal_share_plot.index = temporal_plot_labels
    temporal_palette = [DEFAULT_FLAG_PALETTE.get(int(label), "#64748b") for label in temporal_share_plot.columns]
    temporal_share_plot.plot(
        kind="bar",
        stacked=True,
        ax=ax_bottom,
        color=temporal_palette,
        width=0.85,
    )
    ax_bottom.set_ylim(0, 1)
    ax_bottom.yaxis.set_major_formatter(PercentFormatter(1.0))
    ax_bottom.set_ylabel("Share of reviewed rows")
    ax_bottom.set_xlabel("Time bin start (UTC)")
    ax_bottom.set_title("Reviewed flag composition changes across the time axis")
    ax_bottom.grid(axis="y", alpha=0.2)
    ax_bottom.legend(title="QC flag", bbox_to_anchor=(1.01, 1.0), loc="upper left")
    plt.tight_layout()
    plt.show()

print(
    {
        "temporal_bin_count": temporal_bin_count,
        "time_bins_shown": len(temporal_summary),
        "issue_labels": ISSUE_LABELS,
        "max_issue_share_pct": float(temporal_summary["issue_share_pct"].max()) if len(temporal_summary) else 0.0,
        "min_issue_share_pct": float(temporal_summary["issue_share_pct"].min()) if len(temporal_summary) else 0.0,
    }
)


### Split Comparison Settings

These settings control the split-strategy comparison in the next cells.

The train / validation / test split changes the **question** you are asking:

- `global_contiguous`: one early / middle / late cut across the whole time axis. Use this when you want the strictest deployment-style test of temporal generalization.
- `per_source_contiguous`: make an early / middle / late cut inside each source file, then combine the pieces. Use this when files represent distinct deployments, maintenance periods, or acquisition sessions that should each contribute to every split.
- `interleaved_blocks`: keep short local time blocks intact, but distribute those blocks across train / validation / test. Use this when interesting events are sparse and a single long cut would leave one split with almost none of them.

One important rule: **never artificially balance validation or test**. Those splits should reflect the real distribution that falls out of the chosen strategy.

So the goal here is not to inflate issue labels in validation. The goal is to choose a split that is both scientifically meaningful **and** leaves validation with enough naturally occurring issue examples to support model selection.


In [ ]:
SUPPORTED_SPLIT_STRATEGIES = (
    "global_contiguous",
    "per_source_contiguous",
    "interleaved_blocks",
)

AUTO_SELECT_SPLIT_BLOCK_ROWS = True
SPLIT_BLOCK_ROWS = 1024
SPLIT_BLOCK_ROW_CANDIDATES = (512, 1024, 2048, 4096)
VALIDATION_MIN_ISSUE_ROWS = 100
VALIDATION_MIN_ISSUE_SHARE_PCT = 1.0
VALIDATION_MIN_ROWS_PER_ISSUE_LABEL = 20

show_setup_json(
    {
        "AUTO_SELECT_SPLIT_BLOCK_ROWS": AUTO_SELECT_SPLIT_BLOCK_ROWS,
        "SPLIT_BLOCK_ROWS": SPLIT_BLOCK_ROWS,
        "SPLIT_BLOCK_ROW_CANDIDATES": SPLIT_BLOCK_ROW_CANDIDATES,
        "VALIDATION_MIN_ISSUE_ROWS": VALIDATION_MIN_ISSUE_ROWS,
        "VALIDATION_MIN_ISSUE_SHARE_PCT": VALIDATION_MIN_ISSUE_SHARE_PCT,
        "VALIDATION_MIN_ROWS_PER_ISSUE_LABEL": VALIDATION_MIN_ROWS_PER_ISSUE_LABEL,
    }
)


### Compare Split Strategies On The Full Reviewed Dataset

This comparison uses the **full reviewed rows** from the selected parquet parts.

That is deliberate. We want the split decision to be based on the real reviewed dataset, not on a notebook-sized sample.

The tables below show two things:

- a compact one-row summary for each strategy,
- and a side-by-side train / validation / test breakdown so you can compare issue coverage without scrolling through a long stacked table.


In [ ]:
comparison_labels = [
    label
    for label in sorted(set(GOOD_LABELS).union(ISSUE_LABELS))
    if label in set(reviewed_label_df[TARGET_FLAG].dropna().astype(int).unique().tolist())
]
present_issue_labels = [label for label in ISSUE_LABELS if label in comparison_labels]

block_scan_frame = scan_interleaved_block_rows(
    reviewed_label_df,
    label_column=TARGET_FLAG,
    train_fraction=TRAIN_FRACTION,
    validation_fraction=VALIDATION_FRACTION,
    candidate_block_rows=SPLIT_BLOCK_ROW_CANDIDATES,
)
ACTIVE_SPLIT_BLOCK_ROWS = (
    int(block_scan_frame.iloc[0]["block_rows"])
    if AUTO_SELECT_SPLIT_BLOCK_ROWS and len(block_scan_frame)
    else int(SPLIT_BLOCK_ROWS)
)

if len(block_scan_frame):
    display(block_scan_frame)

split_strategy_labels = {
    "global_contiguous": "Global contiguous",
    "per_source_contiguous": "Per-source contiguous",
    "interleaved_blocks": "Interleaved blocks",
    "episode_aware": "Episode-aware blocks",
}

strategy_metric_rows = []
strategy_adequacy_frames = []
strategy_frames_by_name = {}

for strategy_name in ["global_contiguous", "per_source_contiguous", "interleaved_blocks"]:
    strategy_frames = split_frame_by_strategy(
        reviewed_label_df,
        train_fraction=TRAIN_FRACTION,
        validation_fraction=VALIDATION_FRACTION,
        strategy=strategy_name,
        block_rows=ACTIVE_SPLIT_BLOCK_ROWS,
    )
    strategy_frames_by_name[strategy_name] = strategy_frames
    strategy_counts, strategy_shares = summarize_split_distributions(
        strategy_frames,
        label_column=TARGET_FLAG,
    )
    adequacy_frame = summarize_issue_adequacy(
        strategy_frames,
        label_column=TARGET_FLAG,
        issue_labels=present_issue_labels,
        min_issue_rows=VALIDATION_MIN_ISSUE_ROWS,
        min_issue_share_pct=VALIDATION_MIN_ISSUE_SHARE_PCT,
        min_rows_per_issue_label=VALIDATION_MIN_ROWS_PER_ISSUE_LABEL,
    )
    adequacy_frame = adequacy_frame.reset_index()
    adequacy_frame.insert(0, "strategy", split_strategy_labels[strategy_name])
    strategy_adequacy_frames.append(adequacy_frame)

    gap_metrics = compute_split_share_gap(strategy_shares)
    validation_row = adequacy_frame[adequacy_frame["split"] == "validation"].iloc[0]
    strategy_metric_rows.append(
        {
            "strategy": split_strategy_labels[strategy_name],
            "rows": int(strategy_counts.sum().sum()),
            "train_rows": int(strategy_counts["train"].sum()),
            "validation_rows": int(strategy_counts["validation"].sum()),
            "test_rows": int(strategy_counts["test"].sum()),
            "validation_issue_rows": int(validation_row["issue_rows"]),
            "validation_issue_share_pct": float(validation_row["issue_share_pct"]),
            "validation_adequate": bool(validation_row["adequate_for_model_selection"]),
            **gap_metrics,
        }
    )

strategy_metric_frame, strategy_split_detail = build_split_strategy_tables(
    strategy_metric_rows,
    strategy_adequacy_frames,
)
display(
    strategy_metric_frame.style.format(
        {
            "reviewed_rows_total": "{:,.0f}",
            "train_rows": "{:,.0f}",
            "validation_rows": "{:,.0f}",
            "test_rows": "{:,.0f}",
            "validation_issue_rows": "{:,.0f}",
            "validation_issue_share_pct": "{:.2f}",
            "max_split_gap": "{:.4f}",
            "mean_split_gap": "{:.4f}",
        }
    )
)
display(
    strategy_split_detail.style.format(
        {
            "train_rows": "{:,.0f}",
            "train_issue_rows": "{:,.0f}",
            "train_issue_share_pct": "{:.2f}",
            "validation_rows": "{:,.0f}",
            "validation_issue_rows": "{:,.0f}",
            "validation_issue_share_pct": "{:.2f}",
            "test_rows": "{:,.0f}",
            "test_issue_rows": "{:,.0f}",
            "test_issue_share_pct": "{:.2f}",
        }
    )
)


### Compare Split Strategies Visually

Now that you have seen the label drift through time, we can ask a more focused question: **how does the split itself change the balance?**

We compare three strategies on the same full reviewed dataset:

- **Global contiguous**: the earliest rows go to train, the middle rows go to validation, and the latest rows go to test.
- **Per-source contiguous**: each source file is split early / middle / late, then the per-file pieces are combined.
- **Interleaved blocks**: short local time blocks stay intact, but the blocks are distributed across train / validation / test in a repeating schedule.

The plot below compares those strategies directly. The choice cell comes immediately after that comparison.


In [ ]:
strategy_plot_rows = []
for strategy_name, strategy_frames in strategy_frames_by_name.items():
    strategy_counts, strategy_shares = summarize_split_distributions(
        strategy_frames,
        label_column=TARGET_FLAG,
    )
    for split_name in ["train", "validation", "test"]:
        row = strategy_shares[split_name].reindex(comparison_labels, fill_value=0.0).copy()
        row.name = (
            f"{split_strategy_labels[strategy_name]} | "
            f"{split_name} (n={int(strategy_counts[split_name].sum()):,})"
        )
        strategy_plot_rows.append(row)

strategy_share_plot = pd.DataFrame(strategy_plot_rows)
palette = [DEFAULT_FLAG_PALETTE.get(int(label), "#64748b") for label in strategy_share_plot.columns]

fig, ax = plt.subplots(figsize=(13.2, 6.6))
strategy_share_plot.plot(kind="barh", stacked=True, ax=ax, color=palette, width=0.72)
ax.set_xlim(0, 1)
ax.xaxis.set_major_formatter(PercentFormatter(1.0))
ax.set_xlabel("Share of reviewed rows in the split")
ax.set_ylabel("")
ax.set_title("How the split strategy changes label balance on the full reviewed dataset")
ax.grid(axis="x", alpha=0.2)
ax.legend(title="QC flag", bbox_to_anchor=(1.01, 1.0), loc="upper left")
for divider in [2.5, 5.5]:
    ax.axhline(divider, color="#cbd5e1", linewidth=1.2)
plt.tight_layout()
plt.show()

show_setup_json(
    {
        "ACTIVE_SPLIT_BLOCK_ROWS": ACTIVE_SPLIT_BLOCK_ROWS,
        "AUTO_SELECT_SPLIT_BLOCK_ROWS": AUTO_SELECT_SPLIT_BLOCK_ROWS,
        "present_issue_labels": present_issue_labels,
        "validation_is_rebalanced": False,
    }
)


### Questions To Ask Before You Lock The Split

Before choosing a benchmark split, pause and look for structure that a simple label-balance table cannot show on its own:

1. Are the issue flags isolated points, or do they cluster into longer episodes?
2. Could one real bad-data event be getting cut across train, validation, and test?
3. If the model uses windows or lagged context, would nearby timestamps leak information across split boundaries?
4. Does this split answer the question you care about: generalize to nearby conditions, or generalize to a completely new bad episode?

Keep those questions in mind as you pick the split strategy below. The advanced notebook comes back to them with a stronger episode-aware benchmark split.


### Choose The Split Strategy To Use For Modeling

Pick the split strategy that best matches your scientific question **and** leaves validation with enough naturally occurring issue examples to support model selection.

Validation and test remain unbalanced. This choice only decides **how the reviewed rows are divided**, not how their labels are reweighted.


In [ ]:
SPLIT_STRATEGY = "global_contiguous"
if SPLIT_STRATEGY not in SUPPORTED_SPLIT_STRATEGIES:
    raise ValueError(f"Unsupported split strategy: {SPLIT_STRATEGY}. Choose from {SUPPORTED_SPLIT_STRATEGIES}.")

BENCHMARK_SPLIT_STRATEGY = SPLIT_STRATEGY
BENCHMARK_SPLIT_BLOCK_ROWS = ACTIVE_SPLIT_BLOCK_ROWS
EPISODE_CONTEXT_ROWS = 0
EPISODE_MERGE_GAP_ROWS = 0
EPISODE_PURGE_GAP_ROWS = 0

show_setup_json(
    {
        "SPLIT_STRATEGY": SPLIT_STRATEGY,
        "BENCHMARK_SPLIT_STRATEGY": BENCHMARK_SPLIT_STRATEGY,
        "BENCHMARK_SPLIT_BLOCK_ROWS": BENCHMARK_SPLIT_BLOCK_ROWS,
        "EPISODE_CONTEXT_ROWS": EPISODE_CONTEXT_ROWS,
        "EPISODE_MERGE_GAP_ROWS": EPISODE_MERGE_GAP_ROWS,
        "EPISODE_PURGE_GAP_ROWS": EPISODE_PURGE_GAP_ROWS,
        "validation_is_rebalanced": False,
    }
)


### Build The Benchmark Split

The next cells define the benchmark dataset used by the models:

1. load the **full reviewed dataset** for the selected parquet parts,
2. apply the chosen split strategy to that full reviewed dataset,
3. keep validation and test fixed,
4. optionally shrink **train only** to control runtime.

This gives you a cleaner evaluation because validation and test are not artificially rebalanced. They stay as the chosen split naturally produced them.


### 🧪 Turning The Scalar Stream Into Model Features

A machine-learning model does not automatically know what type of scalar instrument stream this is. We have to decide what each training example looks like.

Here we start from the **full reviewed row-level dataset** for the selected parquet parts. After that, we:

- define one benchmark split on the full reviewed rows,
- keep validation/test fixed and unbalanced,
- and only reduce or rebalance the training split if we need a smaller run.

For the **Random Forest baseline**, we convert those reviewed rows into a **tabular** problem where each row is one example. We include:

- the selected scalar measurements,
- simple clock features such as hour and day of year,
- `abs_delta` features that measure how sharply each variable changed from the previous row.

This is a useful first step because it is simple, fast, and easy to inspect. It also gives us a good reference point before we try more sophisticated ideas.


### Small Utilities For Splits And Metrics

The next helper cell defines small wrappers that the feature-building
and modeling sections reuse from this point onward:

- `split_frame_by_strategy(...)` applies the notebook's active split policy,
- `contiguous_split(...)` is kept as a convenience alias for the strict global time split,
- `report_average(...)` chooses the right averaging rule for the metric summaries below.

Keeping them here makes it easier to see why they exist before they start showing up in the modeling cells.


In [ ]:
# Wrap the shared split helper so the later modeling cells can use the same
# strategy without repeating the configuration arguments everywhere.
def split_frame_by_strategy(
    frame: pd.DataFrame,
    *,
    train_fraction: float,
    validation_fraction: float,
    strategy: str,
    block_rows: int,
    issue_column: str = "issue",
    target_column: str | None = None,
    issue_labels: list[int] | tuple[int, ...] | None = None,
    episode_context_rows: int = 0,
    episode_merge_gap_rows: int = 0,
    purge_gap_rows: int = 0,
) -> dict[str, pd.DataFrame]:
    split_kwargs = {
        "train_fraction": train_fraction,
        "validation_fraction": validation_fraction,
        "strategy": strategy,
        "block_rows": block_rows,
        "issue_column": issue_column,
        "target_column": target_column,
        "issue_labels": issue_labels,
        "episode_context_rows": episode_context_rows,
        "episode_merge_gap_rows": episode_merge_gap_rows,
        "purge_gap_rows": purge_gap_rows,
    }
    supported_parameters = set(inspect.signature(split_frame_by_strategy_impl).parameters)

    if strategy == "episode_aware" and "episode_context_rows" not in supported_parameters:
        raise RuntimeError(
            "The active kernel is still using an older split helper that does not support "
            "episode-aware splitting. Restart the kernel or rerun the imports/helper cells near the "
            "top of Part 3, then try again."
        )

    filtered_kwargs = {
        key: value
        for key, value in split_kwargs.items()
        if key in supported_parameters
    }
    return split_frame_by_strategy_impl(frame, **filtered_kwargs)


# Keep a short alias for the strict early / middle / late split because it is still
# a useful reference point when comparing alternative strategies.
def contiguous_split(
    frame: pd.DataFrame,
    train_fraction: float,
    validation_fraction: float,
) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    split_frames = split_frame_by_strategy(
        frame,
        train_fraction=train_fraction,
        validation_fraction=validation_fraction,
        strategy="global_contiguous",
        block_rows=SPLIT_BLOCK_ROWS,
    )
    return split_frames["train"], split_frames["validation"], split_frames["test"]


# Use binary averaging for binary tasks and macro averaging for multi-class tasks.
def report_average(task_mode: str) -> str:
    return "binary" if task_mode == "binary" else "macro"


# Keep the notebook resilient when a live kernel is still holding an older
# version of session1_modeling that expects `columns=` instead of
# `measurement_columns=`.
def build_model_frame(
    frame: pd.DataFrame,
    *,
    target_flag: str,
    measurement_columns: list[str],
    task_mode: str,
    good_labels: list[int] | tuple[int, ...] | None = None,
    issue_labels: list[int] | tuple[int, ...] | None = None,
    model_row_limit: int | None = None,
) -> tuple[pd.DataFrame, list[str], list[str]]:
    if good_labels is None:
        good_labels = GOOD_LABELS
    if issue_labels is None:
        issue_labels = ISSUE_LABELS
    signature = inspect.signature(build_model_frame_impl)
    if "measurement_columns" in signature.parameters:
        model_df, feature_columns, active_labels = build_model_frame_impl(
            frame,
            target_flag=target_flag,
            measurement_columns=measurement_columns,
            task_mode=task_mode,
            good_labels=good_labels,
            issue_labels=issue_labels,
            model_row_limit=model_row_limit,
        )
    elif "columns" in signature.parameters:
        model_df, feature_columns, active_labels = build_model_frame_impl(
            frame,
            target_flag=target_flag,
            columns=measurement_columns,
            task_mode=task_mode,
            issue_labels=issue_labels,
            model_row_limit=model_row_limit,
        )
    else:
        model_df, feature_columns, active_labels = build_model_frame_impl(
            frame,
            target_flag,
            measurement_columns,
            task_mode,
            issue_labels,
            model_row_limit,
        )

    reviewed_labels = set(good_labels).union(issue_labels)
    model_df = model_df[model_df[target_flag].astype(int).isin(reviewed_labels)].reset_index(drop=True)
    active_labels = sorted(model_df["model_target"].dropna().astype(int).unique().tolist())
    return model_df, feature_columns, active_labels


def build_reviewed_target_frame(
    frame: pd.DataFrame,
    *,
    target_flag: str,
    task_mode: str,
    good_labels: list[int] | tuple[int, ...] | None = None,
    issue_labels: list[int] | tuple[int, ...] | None = None,
    model_row_limit: int | None = None,
) -> tuple[pd.DataFrame, list[str]]:
    if good_labels is None:
        good_labels = GOOD_LABELS
    if issue_labels is None:
        issue_labels = ISSUE_LABELS
    return build_reviewed_target_frame_impl(
        frame,
        target_flag=target_flag,
        task_mode=task_mode,
        good_labels=good_labels,
        issue_labels=issue_labels,
        model_row_limit=model_row_limit,
    )


# Show both absolute counts and composition for whichever split strategy is active.
def summarize_split_distributions(
    split_frames: dict[str, pd.DataFrame],
    *,
    label_column: str = "model_target",
) -> tuple[pd.DataFrame, pd.DataFrame]:
    count_frame = pd.DataFrame(
        {
            split_name: frame[label_column].value_counts().sort_index()
            for split_name, frame in split_frames.items()
        }
    ).fillna(0).astype(int)
    share_frame = count_frame.div(count_frame.sum(axis=0), axis=1).fillna(0.0)
    return count_frame, share_frame


In [ ]:
# Reuse the centrally defined measurement list from the configuration cell.
measurement_columns = MEASUREMENT_COLUMNS.copy()
task_mode = TASK_MODE.lower().strip()
if task_mode not in {"multiclass", "binary"}:
    raise ValueError(f"Unsupported TASK_MODE: {TASK_MODE}")
target_name = TARGET_FLAG if task_mode == "multiclass" else "issue"
benchmark_label_columns = ["Time UTC", "source_file", TARGET_FLAG]
benchmark_label_source_df = load_full_row_level_frame(selected_paths, columns=benchmark_label_columns)
source_rows = len(benchmark_label_source_df)

model_df, active_labels = build_reviewed_target_frame(
    benchmark_label_source_df,
    target_flag=TARGET_FLAG,
    task_mode=task_mode,
    good_labels=GOOD_LABELS,
    issue_labels=ISSUE_LABELS,
    model_row_limit=None,
)
benchmark_model_df = model_df.copy()

display(model_df[["Time UTC", "source_file", TARGET_FLAG, "issue", "model_target"]].head(8))
print(
    {
        "benchmark_source_rows": source_rows,
        "model_rows": len(model_df),
        "dropped_unreviewed_rows": int(source_rows - len(model_df)),
        "active_labels": active_labels,
    }
)


In [ ]:
# Apply the active split strategy to the full reviewed benchmark dataset.
benchmark_split_frames = split_frame_by_strategy(
    benchmark_model_df,
    train_fraction=TRAIN_FRACTION,
    validation_fraction=VALIDATION_FRACTION,
    strategy=BENCHMARK_SPLIT_STRATEGY,
    block_rows=BENCHMARK_SPLIT_BLOCK_ROWS,
    issue_column="issue",
    target_column=TARGET_FLAG,
    issue_labels=ISSUE_LABELS,
    episode_context_rows=EPISODE_CONTEXT_ROWS,
    episode_merge_gap_rows=EPISODE_MERGE_GAP_ROWS,
    purge_gap_rows=EPISODE_PURGE_GAP_ROWS,
)
train_full_df = benchmark_split_frames["train"]
valid_df = benchmark_split_frames["validation"]
test_df = benchmark_split_frames["test"]
model_issue_labels = [1] if task_mode == "binary" else [label for label in active_labels if label in ISSUE_LABELS]

print(
    {
        "split_strategy": BENCHMARK_SPLIT_STRATEGY,
        "benchmark_split_block_rows": BENCHMARK_SPLIT_BLOCK_ROWS if BENCHMARK_SPLIT_STRATEGY in {"interleaved_blocks", "episode_aware"} else None,
        "episode_context_rows": EPISODE_CONTEXT_ROWS if BENCHMARK_SPLIT_STRATEGY == "episode_aware" else 0,
        "episode_merge_gap_rows": EPISODE_MERGE_GAP_ROWS if BENCHMARK_SPLIT_STRATEGY == "episode_aware" else 0,
        "episode_purge_gap_rows": EPISODE_PURGE_GAP_ROWS if BENCHMARK_SPLIT_STRATEGY == "episode_aware" else 0,
        "model_issue_labels": model_issue_labels,
        "full_train_rows": len(train_full_df),
        "validation_rows": len(valid_df),
        "test_rows": len(test_df),
        "train_time_span": [train_full_df["Time UTC"].min().isoformat(), train_full_df["Time UTC"].max().isoformat()],
        "validation_time_span": [valid_df["Time UTC"].min().isoformat(), valid_df["Time UTC"].max().isoformat()],
        "test_time_span": [test_df["Time UTC"].min().isoformat(), test_df["Time UTC"].max().isoformat()],
    }
)


### Review The Full Benchmark Split

These next cells stay focused on the **full reviewed benchmark split** only.

First we summarize split size and validation issue coverage. Then we visualize the target mix in train / validation / test.


In [ ]:
split_frames = {"train": train_full_df, "validation": valid_df, "test": test_df}
split_counts, split_shares = summarize_split_distributions(split_frames, label_column="model_target")

split_adequacy = summarize_issue_adequacy(
    split_frames,
    label_column="model_target",
    issue_labels=model_issue_labels,
    min_issue_rows=VALIDATION_MIN_ISSUE_ROWS,
    min_issue_share_pct=VALIDATION_MIN_ISSUE_SHARE_PCT,
    min_rows_per_issue_label=VALIDATION_MIN_ROWS_PER_ISSUE_LABEL,
)

split_overview_rows = []
for split_name, frame in split_frames.items():
    label_counts = frame["model_target"].value_counts().sort_index()
    issue_rows = int(label_counts.reindex(model_issue_labels, fill_value=0).sum())
    split_overview_rows.append(
        {
            "split": split_name,
            "rows": len(frame),
            "issue_rows": issue_rows,
            "issue_share_pct": round(100 * issue_rows / max(len(frame), 1), 2),
            "time_start": frame["Time UTC"].min().isoformat(),
            "time_end": frame["Time UTC"].max().isoformat(),
        }
    )
split_overview = pd.DataFrame(split_overview_rows).set_index("split")

validation_adequacy = (
    split_adequacy.loc[["validation"]]
    .rename(
        columns={
            "meets_issue_row_floor": "enough_issue_rows",
            "meets_issue_share_floor": "enough_issue_share",
            "meets_per_issue_label_floor": "enough_each_issue_label",
        }
    )
)

display(
    split_overview.style.format(
        {
            "rows": "{:,.0f}",
            "issue_rows": "{:,.0f}",
            "issue_share_pct": "{:.2f}",
        }
    )
)
display(validation_adequacy)


In [ ]:
split_share_plot = split_shares.T.copy()
split_share_plot.index = [
    f"{split_name} (n={int(split_counts[split_name].sum()):,})"
    for split_name in split_share_plot.index
]
palette = [DEFAULT_FLAG_PALETTE.get(int(label), "#64748b") for label in split_share_plot.columns]

fig, ax = plt.subplots(figsize=(12, 4.2))
split_share_plot.plot(kind="barh", stacked=True, ax=ax, color=palette, width=0.62)
ax.set_xlim(0, 1)
ax.xaxis.set_major_formatter(PercentFormatter(1.0))
ax.set_xlabel("Share of reviewed rows in the split")
ax.set_ylabel("")
ax.set_title(
    f"Target composition for {split_strategy_labels.get(BENCHMARK_SPLIT_STRATEGY, BENCHMARK_SPLIT_STRATEGY)} "
    "train / validation / test splits"
)
ax.grid(axis="x", alpha=0.2)
ax.legend(title="target_label", bbox_to_anchor=(1.01, 1.0), loc="upper left")
plt.tight_layout()
plt.show()


### Set Up The Train-Only Comparison

The benchmark split is fixed now. These settings only control the **candidate training subsets** compared in the next block.

`BALANCED_REVIEWED_TARGET_ISSUE_SHARE` only matters for `balanced_reviewed`. For example:

- `0.20` means you are asking for about 20% issue rows inside the reviewed training subset,
- `0.50` means a much stronger 50/50 reviewed good-vs-issue mix.

What to be careful about:

- increasing the issue share can help the model see more rare events,
- but pushing it too high can make the training distribution unrealistic,
- and that can hurt calibration or make the model behave as if issues are much more common than they really are.

Validation and test are left untouched so the comparison stays fair.


In [ ]:
SUPPORTED_TRAIN_SUBSET_STRATEGIES = ("full_train", "time_spread", "balanced_reviewed", "issue_focused")
TRAIN_SUBSET_MAX_ROWS = None if DATA_FRACTION >= 0.999 else max(100000, int(BASE_MODEL_ROW_LIMIT * DATA_FRACTION))
ISSUE_ROWS_PER_FILE = max(1000, int(BASE_ISSUE_ROWS_PER_FILE * DATA_FRACTION))
BALANCED_REVIEWED_TARGET_ISSUE_SHARE = 0.25
if not 0 < BALANCED_REVIEWED_TARGET_ISSUE_SHARE < 1:
    raise ValueError("BALANCED_REVIEWED_TARGET_ISSUE_SHARE must be in the interval (0, 1).")

show_setup_json(
    {
        "TRAIN_SUBSET_MAX_ROWS": TRAIN_SUBSET_MAX_ROWS,
        "ISSUE_ROWS_PER_FILE": ISSUE_ROWS_PER_FILE,
        "BALANCED_REVIEWED_TARGET_ISSUE_SHARE": BALANCED_REVIEWED_TARGET_ISSUE_SHARE,
    }
)


### Compare Train-Only Subset Options

These next cells stay focused on the training set only. Validation and test do not change here.

First we summarize the candidate train-only subsets. Then we visualize how the target mix changes inside train.


In [ ]:
train_subset_preview_columns = ["Time UTC", "model_target"]
train_subset_preview_source = train_full_df[train_subset_preview_columns].copy()
train_subset_rows_limit = (
    len(train_full_df)
    if TRAIN_SUBSET_MAX_ROWS is None
    else min(int(TRAIN_SUBSET_MAX_ROWS), len(train_full_df))
)

train_subset_preview_frames = {"full_train": train_subset_preview_source}
for subset_strategy in ("time_spread", "balanced_reviewed", "issue_focused"):
    train_subset_preview_frames[subset_strategy] = sample_frame_by_strategy(
        train_subset_preview_source,
        rows_limit=train_subset_rows_limit,
        sample_strategy=subset_strategy,
        target_flag="model_target",
        good_labels=GOOD_LABELS,
        issue_labels=model_issue_labels,
        issue_rows=ISSUE_ROWS_PER_FILE,
        balanced_issue_share=BALANCED_REVIEWED_TARGET_ISSUE_SHARE,
    )

train_subset_labels = {
    "full_train": "full train",
    "time_spread": "time-spread subset",
    "balanced_reviewed": f"balanced-reviewed subset ({BALANCED_REVIEWED_TARGET_ISSUE_SHARE:.0%} issue target)",
    "issue_focused": "issue-focused subset",
}

train_subset_overview_rows = []
for subset_name, subset_frame in train_subset_preview_frames.items():
    subset_counts = subset_frame["model_target"].value_counts().sort_index()
    issue_rows = int(subset_counts.reindex(model_issue_labels, fill_value=0).sum())
    train_subset_overview_rows.append(
        {
            "subset": train_subset_labels[subset_name],
            "rows": len(subset_frame),
            "issue_rows": issue_rows,
            "issue_share_pct": round(100 * issue_rows / max(len(subset_frame), 1), 2),
            "time_start": subset_frame["Time UTC"].min().isoformat(),
            "time_end": subset_frame["Time UTC"].max().isoformat(),
        }
    )
train_subset_overview = pd.DataFrame(train_subset_overview_rows).set_index("subset")

train_budget_counts = pd.DataFrame(
    {
        subset_name: subset_frame["model_target"].value_counts().sort_index()
        for subset_name, subset_frame in train_subset_preview_frames.items()
    }
).fillna(0).astype(int)
train_budget_shares = train_budget_counts.div(train_budget_counts.sum(axis=0), axis=1).fillna(0.0)
display(
    train_subset_overview.style.format(
        {
            "rows": "{:,.0f}",
            "issue_rows": "{:,.0f}",
            "issue_share_pct": "{:.2f}",
        }
    )
)


In [ ]:
train_budget_plot = train_budget_shares.T.copy()
train_budget_plot.index = [
    f"{train_subset_labels[subset_name]} (n={int(train_budget_counts[subset_name].sum()):,})"
    for subset_name in train_budget_plot.index
]
palette = [DEFAULT_FLAG_PALETTE.get(int(label), "#64748b") for label in train_budget_plot.columns]

fig, ax = plt.subplots(figsize=(11.8, 4.6))
train_budget_plot.plot(kind="barh", stacked=True, ax=ax, color=palette, width=0.6)
ax.set_xlim(0, 1)
ax.xaxis.set_major_formatter(PercentFormatter(1.0))
ax.set_xlabel("Share of reviewed rows in train")
ax.set_ylabel("")
ax.set_title("How the train-only subset strategy changes the training set")
ax.grid(axis="x", alpha=0.2)
ax.legend(title="target_label", bbox_to_anchor=(1.01, 1.0), loc="upper left")
plt.tight_layout()
plt.show()


### Choose The Training Subset To Use For Modeling

Pick the train-only subset strategy you want the models to fit on. This is the main knob to change when you want to see how train-set construction affects the models.

If you choose `balanced_reviewed`, keep an eye on `BALANCED_REVIEWED_TARGET_ISSUE_SHARE`. Raising it gives the model more issue examples, but it also makes the training distribution less like the real deployment distribution.

Validation and test stay fixed. This choice only affects the training rows that are handed to the models.


In [ ]:
# Change this directly to compare how different train-only subset choices affect the models.
# Options: "full_train", "time_spread", "balanced_reviewed", "issue_focused"
TRAIN_SUBSET_STRATEGY = "full_train" if DATA_FRACTION >= 0.999 else "time_spread"
if TRAIN_SUBSET_STRATEGY not in SUPPORTED_TRAIN_SUBSET_STRATEGIES:
    raise ValueError(
        f"Unsupported TRAIN_SUBSET_STRATEGY: {TRAIN_SUBSET_STRATEGY}. Choose from {SUPPORTED_TRAIN_SUBSET_STRATEGIES}."
    )

train_df = (
    train_full_df
    if TRAIN_SUBSET_STRATEGY == "full_train"
    else sample_frame_by_strategy(
        train_full_df,
        rows_limit=TRAIN_SUBSET_MAX_ROWS,
        sample_strategy=TRAIN_SUBSET_STRATEGY,
        target_flag=TARGET_FLAG,
        good_labels=GOOD_LABELS,
        issue_labels=ISSUE_LABELS,
        issue_rows=ISSUE_ROWS_PER_FILE,
        balanced_issue_share=BALANCED_REVIEWED_TARGET_ISSUE_SHARE,
    )
)

show_setup_json(
    {
        "TRAIN_SUBSET_STRATEGY": TRAIN_SUBSET_STRATEGY,
        "TRAIN_SUBSET_MAX_ROWS": TRAIN_SUBSET_MAX_ROWS,
        "BALANCED_REVIEWED_TARGET_ISSUE_SHARE": BALANCED_REVIEWED_TARGET_ISSUE_SHARE,
        "train_rows": len(train_df),
        "validation_rows": len(valid_df),
        "test_rows": len(test_df),
    }
)


### Optional Row Caps For The Tabular Benchmark Dataset

The Random Forest uses tabular rows rather than sequence windows, so this is where we put practical row caps on the **already chosen** benchmark split.

The order matters:

1. choose the split strategy,
2. choose the train-only subset strategy,
3. then, if needed, cap the tabular train / validation / test rows for runtime.

These caps do **not** rebalance validation or test. They only thin each split down to a manageable size while keeping the existing time structure as intact as possible.


In [ ]:
TABULAR_TRAIN_MAX_ROWS = min(len(train_df), 250_000)
TABULAR_VALIDATION_MAX_ROWS = min(len(valid_df), 150_000)
TABULAR_TEST_MAX_ROWS = min(len(test_df), 150_000)

TABULAR_TRAIN_CAP_STRATEGY = (
    TRAIN_SUBSET_STRATEGY
    if TRAIN_SUBSET_STRATEGY != "full_train"
    else "time_spread"
)

rf_train_fit_df = (
    sample_frame_by_strategy(
        train_df,
        rows_limit=TABULAR_TRAIN_MAX_ROWS,
        sample_strategy=TABULAR_TRAIN_CAP_STRATEGY,
        target_flag=TARGET_FLAG,
        good_labels=GOOD_LABELS,
        issue_labels=ISSUE_LABELS,
        issue_rows=ISSUE_ROWS_PER_FILE,
        balanced_issue_share=BALANCED_REVIEWED_TARGET_ISSUE_SHARE,
    )
    if TABULAR_TRAIN_MAX_ROWS is not None and len(train_df) > TABULAR_TRAIN_MAX_ROWS
    else train_df.copy()
)
rf_valid_eval_df = (
    evenly_spaced_take(valid_df.sort_values("Time UTC").reset_index(drop=True), TABULAR_VALIDATION_MAX_ROWS)
    if TABULAR_VALIDATION_MAX_ROWS is not None and len(valid_df) > TABULAR_VALIDATION_MAX_ROWS
    else valid_df.copy()
)
rf_test_eval_df = (
    evenly_spaced_take(test_df.sort_values("Time UTC").reset_index(drop=True), TABULAR_TEST_MAX_ROWS)
    if TABULAR_TEST_MAX_ROWS is not None and len(test_df) > TABULAR_TEST_MAX_ROWS
    else test_df.copy()
)

show_setup_json(
    {
        "TABULAR_TRAIN_MAX_ROWS": TABULAR_TRAIN_MAX_ROWS,
        "TABULAR_VALIDATION_MAX_ROWS": TABULAR_VALIDATION_MAX_ROWS,
        "TABULAR_TEST_MAX_ROWS": TABULAR_TEST_MAX_ROWS,
        "tabular_train_cap_strategy": TABULAR_TRAIN_CAP_STRATEGY,
        "rf_train_rows_used": len(rf_train_fit_df),
        "rf_validation_rows_used": len(rf_valid_eval_df),
        "rf_test_rows_used": len(rf_test_eval_df),
    }
)


## Part 4 — Random Forest

We start with a strong tabular baseline: one row becomes one supervised example, and the forest learns to separate QC classes from engineered numeric features.


### 🌲 Supervised Learning: Random Forest

A **Random Forest** is an ensemble of many decision trees.

Here is the basic idea:

1. make many slightly different training samples from the data,
2. train one decision tree on each sample,
3. let the trees vote on the final class.

Each tree asks simple if/then questions such as:

- is conductivity above some threshold?
- did temperature change sharply?
- is the measurement happening at a particular time of day?

Why this is a good Session 1 model:

- it works well on numeric tabular data,
- it usually needs less feature scaling ceremony than many other models,
- it gives us interpretable outputs such as feature importance.

Limits to keep in mind:

- it does not naturally understand long sequential context,
- it only learns from the features we explicitly give it,
- rare classes can still be difficult.


![Random Forest bagging illustration](../assets/session1/random_forest_bagging_illustration.png)

Diagram idea: bootstrap samples are drawn from the original dataset, separate trees are trained, and their predictions are aggregated into one model decision.

Image credit: Harry585, Wikimedia Commons, CC BY-SA 4.0. Source: [File:Random Forest Bagging Illustration.png](https://commons.wikimedia.org/wiki/File:Random_Forest_Bagging_Illustration.png)


### Random Forest Settings

These settings control the baseline forest we train below.

Main variables:

- `n_estimators`: number of trees in the forest.
- `max_depth`: maximum depth of each tree. `None` means grow until other stopping rules apply.
- `min_samples_leaf`: minimum number of samples allowed in a leaf.
- `min_samples_split`: minimum number of samples needed to split an internal node.
- `max_features`: how many features each split is allowed to consider. Common choices are `"sqrt"`, `"log2"`, or `None`.
- `class_weight`: how strongly to compensate for imbalance. Common choices are `None`, `"balanced"`, and `"balanced_subsample"`.
- `trees_per_step`: how many trees to add each time we print progress.

Forests do **not** train epoch-by-epoch like neural networks. To make progress visible, we grow the forest in chunks using `warm_start=True` and print the validation F1 after each chunk.

One practical note: when we use `warm_start=True`, this notebook converts `"balanced"` or `"balanced_subsample"` into one fixed class-weight dictionary computed from `y_train`. That avoids a scikit-learn warning and keeps the weighting consistent across growth steps.


In [ ]:
RF_CONFIG = {
    "imputer_strategy": "median",
    "n_estimators": 200,
    "trees_per_step": 25,
    "max_depth": None,
    "min_samples_leaf": 2,
    "min_samples_split": 2,
    "max_features": "sqrt",
    "class_weight": "balanced_subsample",
    "verbose": 0,
}

display(pd.Series(RF_CONFIG, name="value").rename_axis("rf_parameter").to_frame())
print(
    {
        "rf_train_rows_used": len(rf_train_fit_df),
        "rf_validation_rows_used": len(rf_valid_eval_df),
        "rf_test_rows_used": len(rf_test_eval_df),
        "tabular_train_cap_strategy": TABULAR_TRAIN_CAP_STRATEGY,
    }
)


In [ ]:
rf_split_rows = materialize_reviewed_split_frames(
    selected_paths,
    {
        "train": rf_train_fit_df,
        "validation": rf_valid_eval_df,
        "test": rf_test_eval_df,
    },
    columns=ROW_USE_COLUMNS,
    target_flag=TARGET_FLAG,
    task_mode=task_mode,
    good_labels=GOOD_LABELS,
    issue_labels=ISSUE_LABELS,
)
rf_train_rows_df = rf_split_rows["train"]
rf_valid_rows_df = rf_split_rows["validation"]
rf_test_rows_df = rf_split_rows["test"]

rf_train_fit_df, feature_columns, _ = build_model_frame(
    rf_train_rows_df,
    target_flag=TARGET_FLAG,
    measurement_columns=measurement_columns,
    task_mode=task_mode,
    good_labels=GOOD_LABELS,
    issue_labels=ISSUE_LABELS,
    model_row_limit=None,
)
rf_valid_eval_df, _, _ = build_model_frame(
    rf_valid_rows_df,
    target_flag=TARGET_FLAG,
    measurement_columns=measurement_columns,
    task_mode=task_mode,
    good_labels=GOOD_LABELS,
    issue_labels=ISSUE_LABELS,
    model_row_limit=None,
)
rf_test_eval_df, _, _ = build_model_frame(
    rf_test_rows_df,
    target_flag=TARGET_FLAG,
    measurement_columns=measurement_columns,
    task_mode=task_mode,
    good_labels=GOOD_LABELS,
    issue_labels=ISSUE_LABELS,
    model_row_limit=None,
)

# Step 1: fit the imputer once on training data and reuse it for validation/test.
rf_imputer = SimpleImputer(strategy=RF_CONFIG.get("imputer_strategy", "median"))
X_train_rf = rf_imputer.fit_transform(rf_train_fit_df[feature_columns]).astype(np.float32, copy=False)
X_valid_rf = rf_imputer.transform(rf_valid_eval_df[feature_columns]).astype(np.float32, copy=False)
y_train_rf = rf_train_fit_df["model_target"].reset_index(drop=True)
y_valid_rf = rf_valid_eval_df["model_target"].reset_index(drop=True)
y_test_rf = rf_test_eval_df["model_target"].reset_index(drop=True)

# Step 2: compute one stable class-weight dictionary for the whole training split.
requested_class_weight = RF_CONFIG.get("class_weight")
if requested_class_weight in {"balanced", "balanced_subsample"}:
    rf_classes = np.array(sorted(pd.Series(y_train_rf).dropna().unique()))
    rf_weight_values = compute_class_weight(
        class_weight="balanced",
        classes=rf_classes,
        y=y_train_rf,
    )
    effective_class_weight = {
        int(label) if isinstance(label, (np.integer, int)) else label: float(weight)
        for label, weight in zip(rf_classes.tolist(), rf_weight_values.tolist())
    }
else:
    effective_class_weight = requested_class_weight

print(
    {
        "requested_class_weight": requested_class_weight,
        "effective_class_weight": effective_class_weight,
    }
)

# Step 3: build the forest itself. We use warm_start so we can grow it in chunks and print progress.
rf_model = RandomForestClassifier(
    n_estimators=RF_CONFIG["trees_per_step"],
    warm_start=True,
    max_depth=RF_CONFIG["max_depth"],
    min_samples_leaf=RF_CONFIG["min_samples_leaf"],
    min_samples_split=RF_CONFIG["min_samples_split"],
    max_features=RF_CONFIG["max_features"],
    class_weight=effective_class_weight,
    n_jobs=-1,
    random_state=SEED,
    verbose=RF_CONFIG["verbose"],
)

rf_progress_rows = []
total_trees = RF_CONFIG["n_estimators"]
trees_per_step = RF_CONFIG["trees_per_step"]
growth_schedule = list(range(trees_per_step, total_trees + 1, trees_per_step))
if not growth_schedule:
    growth_schedule = [total_trees]
elif growth_schedule[-1] != total_trees:
    growth_schedule.append(total_trees)

for tree_count in growth_schedule:
    rf_model.set_params(n_estimators=tree_count)
    rf_model.fit(X_train_rf, y_train_rf)
    current_valid_predictions = rf_model.predict(X_valid_rf)
    current_valid_f1 = f1_score(
        y_valid_rf,
        current_valid_predictions,
        average=report_average(task_mode),
        zero_division=0,
    )
    rf_progress_rows.append({"trees_built": tree_count, "validation_f1": float(current_valid_f1)})
    print(f"Built {tree_count:>3} trees | validation F1 = {current_valid_f1:.4f}")

# Step 4: wrap the fitted pieces into a reusable sklearn pipeline object.
rf_pipeline = Pipeline(
    steps=[
        ("imputer", rf_imputer),
        ("model", rf_model),
    ]
)

X_test_rf = rf_imputer.transform(rf_test_eval_df[feature_columns]).astype(np.float32, copy=False)
valid_predictions = rf_model.predict(X_valid_rf)
test_predictions = rf_model.predict(X_test_rf)
labels = sorted(pd.unique(pd.concat([y_train_rf, y_valid_rf, y_test_rf])))

display(pd.DataFrame(rf_progress_rows))

with RF_MODEL_PATH.open("wb") as handle:
    pickle.dump(
        {
            "pipeline": rf_pipeline,
            "feature_columns": feature_columns,
            "labels": labels,
            "task_mode": task_mode,
        },
        handle,
    )

print(
    {
        "saved_random_forest_model": str(RF_MODEL_PATH),
        "rf_train_rows_used": int(len(rf_train_fit_df)),
        "rf_validation_rows_used": int(len(rf_valid_eval_df)),
        "rf_test_rows_used": int(len(rf_test_eval_df)),
    }
)


In [ ]:
# Report validation and test metrics separately for the RF evaluation slices.
metric_rows = []
for split_name, y_true, y_pred in [
    ("validation", y_valid_rf, valid_predictions),
    ("test", y_test_rf, test_predictions),
]:
    split_f1 = f1_score(y_true, y_pred, average=report_average(task_mode), zero_division=0)
    metric_rows.append({"split": split_name, "f1": round(float(split_f1), 4)})
    print(f"{split_name.title()} macro/binary F1: {split_f1:.4f}")
    print(classification_report(y_true, y_pred, labels=labels, zero_division=0))

display(pd.DataFrame(metric_rows))

# Plot normalized confusion matrices for both validation and test.
fig, axes = plt.subplots(1, 2, figsize=(15, 5))
ConfusionMatrixDisplay.from_predictions(
    y_valid_rf,
    valid_predictions,
    labels=labels,
    display_labels=[str(label) for label in labels],
    normalize="true",
    cmap="Blues",
    values_format=".2f",
    xticks_rotation=45,
    ax=axes[0],
)
axes[0].set_title(f"Validation confusion matrix for {target_name}")

ConfusionMatrixDisplay.from_predictions(
    y_test_rf,
    test_predictions,
    labels=labels,
    display_labels=[str(label) for label in labels],
    normalize="true",
    cmap="Blues",
    values_format=".2f",
    xticks_rotation=45,
    ax=axes[1],
)
axes[1].set_title(f"Test confusion matrix for {target_name}")
fig.suptitle("How to read these plots: each row is a true class, and darker diagonal cells are better.", y=1.03)
plt.tight_layout()
plt.show()


In [ ]:
# Feature importance tells us which columns the forest used most strongly.
feature_importances = pd.Series(
    rf_pipeline.named_steps["model"].feature_importances_,
    index=feature_columns,
).sort_values(ascending=False)

top_importances = feature_importances.head(12).sort_values()
top_importances.plot(kind="barh", figsize=(8, 6), title="Top Random Forest feature importances")
plt.xlabel("Importance")
plt.tight_layout()
plt.show()
display(feature_importances.head(12).rename("importance").to_frame())

# Show a few test-set mistakes to ground the discussion in real timestamps.
error_preview_columns = [
    "Time UTC",
    TARGET_FLAG,
    "issue",
    PLOT_MEASUREMENT_COLUMN,
    PLOT_SECONDARY_COLUMN,
]
error_preview_columns.extend(
    [
        column
        for column in MEASUREMENT_COLUMNS
        if column not in error_preview_columns
    ][:1]
)
error_preview_columns = [column for column in error_preview_columns if column in rf_test_eval_df.columns]
error_frame = rf_test_eval_df[error_preview_columns].copy()
error_frame["prediction"] = test_predictions
error_frame["correct"] = error_frame["prediction"] == y_test_rf.to_numpy()
error_examples = error_frame.loc[~error_frame["correct"]].head(12)
print({"test_errors_shown": len(error_examples), "test_error_rate": round(float((~error_frame["correct"]).mean()), 4)})
display(error_examples)


### Date-Range Demo: Predict QC Flags with the Random Forest

The test-set metrics above summarize performance across the whole held-out split. This mini-workflow asks a more concrete question:

**What does the Random Forest predict on one specific interval of time?**

Use UTC strings such as `"2025-09-10 00:00:00Z"` if you want to override the default range.


In [ ]:
RF_RANGE_START = None
RF_RANGE_END = None
RF_AUTO_SELECT_TEST_RANGE = True
RF_MAX_POINTS_TO_PLOT = 800

print(
    {
        "RF_RANGE_START": RF_RANGE_START,
        "RF_RANGE_END": RF_RANGE_END,
        "RF_AUTO_SELECT_TEST_RANGE": RF_AUTO_SELECT_TEST_RANGE,
        "RF_MAX_POINTS_TO_PLOT": RF_MAX_POINTS_TO_PLOT,
    }
)


In [ ]:
rf_range_selection = select_time_range(
    test_df,
    time_column="Time UTC",
    label_column=TARGET_FLAG,
    start=RF_RANGE_START,
    end=RF_RANGE_END,
    auto_select=RF_AUTO_SELECT_TEST_RANGE,
    max_points=RF_MAX_POINTS_TO_PLOT,
)

rf_interval_rows = load_rows_for_time_range(
    metadata,
    row_cache_dir=Path(ROW_CACHE_DIR),
    start=rf_range_selection["start"],
    end=rf_range_selection["end"],
    columns=ROW_USE_COLUMNS,
)

if rf_interval_rows.empty:
    print("No row-level data was found in the requested Random Forest range.")
else:
    rf_interval_model_df, _, _ = build_model_frame(
        rf_interval_rows,
        target_flag=TARGET_FLAG,
        measurement_columns=measurement_columns,
        task_mode=task_mode,
        model_row_limit=None,
    )
    rf_interval_model_df = rf_interval_model_df[
        (rf_interval_model_df["Time UTC"] >= rf_range_selection["start"])
        & (rf_interval_model_df["Time UTC"] <= rf_range_selection["end"])
    ].reset_index(drop=True)

    if rf_interval_model_df.empty:
        print("The selected Random Forest range did not contain usable labeled rows after preparation.")
    else:
        rf_interval_predictions = rf_pipeline.predict(rf_interval_model_df[feature_columns])
        rf_interval_origin = infer_interval_origin(
            rf_range_selection["start"],
            rf_range_selection["end"],
            {"train": train_df, "validation": valid_df, "test": test_df},
        )
        rf_interval_metrics = compute_interval_classification_metrics(
            rf_interval_model_df["model_target"],
            rf_interval_predictions,
            labels=labels,
            average=report_average(task_mode),
            target_names=[str(label) for label in labels],
        )
        rf_plot_palette = DEFAULT_FLAG_PALETTE if task_mode == "multiclass" else {0: "#1f77b4", 1: "#d62728"}
        rf_true_intervals = build_labeled_intervals(
            rf_interval_model_df,
            time_column="Time UTC",
            label_column="model_target",
        )
        rf_pred_frame = rf_interval_model_df[["Time UTC"]].copy()
        rf_pred_frame["predicted_label"] = rf_interval_predictions
        rf_pred_intervals = build_labeled_intervals(
            rf_pred_frame,
            time_column="Time UTC",
            label_column="predicted_label",
        )

        print(
            {
                "selection_mode": rf_range_selection["selection_mode"],
                "selected_priority_flag": rf_range_selection["selected_label"],
                "interval_origin": rf_interval_origin,
                "range_start": rf_range_selection["start"].isoformat(),
                "range_end": rf_range_selection["end"].isoformat(),
                "rows_in_interval": int(len(rf_interval_model_df)),
                "interval_f1": rf_interval_metrics["f1"],
            }
        )
        print(rf_interval_metrics["report_text"])

        display(
            pd.DataFrame(
                {
                    "true_count": pd.Series(rf_interval_model_df["model_target"]).value_counts().sort_index(),
                    "predicted_count": pd.Series(rf_interval_predictions).value_counts().sort_index(),
                }
            ).fillna(0).astype(int)
        )

        rf_demo_figure = plot_time_series_with_bands(
            rf_interval_model_df,
            band_specs=[
                {"title": "True flags", "intervals": rf_true_intervals, "palette": rf_plot_palette},
                {"title": "RF predictions", "intervals": rf_pred_intervals, "palette": rf_plot_palette},
            ],
            measurement_column=PLOT_MEASUREMENT_COLUMN,
            secondary_column=PLOT_SECONDARY_COLUMN,
            max_points=RF_MAX_POINTS_TO_PLOT,
            title="Random Forest predictions on a selected time range",
        )
        plt.show()

        rf_cm_fig, rf_cm_ax = plt.subplots(figsize=(5, 4))
        ConfusionMatrixDisplay(
            confusion_matrix=rf_interval_metrics["confusion_matrix"],
            display_labels=rf_interval_metrics["display_labels"],
        ).plot(ax=rf_cm_ax, cmap="Blues", colorbar=False)
        rf_cm_ax.set_title("Random Forest confusion matrix on the selected range")
        plt.tight_layout()
        plt.show()


### Reflection Questions: Random Forest

1. Which input features seem most responsible for the mistakes in this interval: raw measurements, change features, or time-of-day features?
2. Do the errors line up with the class imbalance we saw earlier, or do they suggest a different modeling problem?
3. If you wanted to improve this interval specifically, would you change the features, the target, or the forest settings first?

Add your own notes or follow-up questions here:

- 
- 
- 


## Part 5 — k-means

Next we switch to an unsupervised lens: instead of predicting flags directly, we group windows with similar summary behavior and interpret those clusters.


### 🧭 Unsupervised Learning: k-means On Window Features

Supervised learning uses labels. Unsupervised learning does **not**.

Here we use **k-means**, which groups rows or windows into clusters based on similarity in feature space. The cluster IDs themselves do not carry meaning ahead of time. We interpret them *after* fitting by looking at:

- how many examples ended up in each cluster,
- the mean issue rate inside each cluster,
- where the cluster sits in feature space,
- and what the underlying time-series examples look like.

This is useful when you want to surface interesting operating regimes or suspicious periods even before a label model is mature.

One subtle point: the cluster numbers and colours are arbitrary. They only become meaningful after we interpret them using the plots and summary statistics below.


### 🔎 k-means Settings

Main variables:

- `n_clusters`: how many clusters we ask the algorithm to find.
- `n_init`: how many random initializations to try. `"auto"` is a good default in current scikit-learn.
- `KMEANS_FEATURE_MODE`: inherited from the dataset profile. Some datasets cluster best on window summaries, while spike-driven ones may cluster better on row-level features.

If you want to experiment, the most useful first change is usually `n_clusters`.


In [ ]:
KMEANS_CONFIG = {
    "n_clusters": 5,
    "n_init": "auto",
}

KMEANS_EXAMPLES_PER_CLUSTER = 1
KMEANS_EXAMPLE_CONTEXT_POINTS = 7500
KMEANS_EXAMPLE_HIGHLIGHT_ALPHA = 0.22

display(pd.Series(KMEANS_CONFIG, name="value").rename_axis("kmeans_parameter").to_frame())
print(
    {
        "KMEANS_FEATURE_MODE": KMEANS_FEATURE_MODE,
        "KMEANS_EXAMPLES_PER_CLUSTER": KMEANS_EXAMPLES_PER_CLUSTER,
        "KMEANS_EXAMPLE_CONTEXT_POINTS": KMEANS_EXAMPLE_CONTEXT_POINTS,
        "KMEANS_EXAMPLE_HIGHLIGHT_ALPHA": KMEANS_EXAMPLE_HIGHLIGHT_ALPHA,
    }
)


In [ ]:
if KMEANS_FEATURE_MODE == "window_summary":
    cluster_source_df = window_df
    kmeans_feature_columns = [
        column
        for column in cluster_source_df.columns
        if column.endswith("_mean") or column.endswith("_std")
    ]
    cluster_x_column = f"{PLOT_MEASUREMENT_COLUMN}_mean"
    cluster_y_column = f"{PLOT_SECONDARY_COLUMN}_mean"
    cluster_axis_label_suffix = "mean"
else:
    cluster_source_rows = load_selected_row_level_frame(
        selected_paths,
        benchmark_model_df,
        columns=ROW_USE_COLUMNS,
    )
    cluster_source_df, feature_columns, _ = build_model_frame(
        cluster_source_rows,
        target_flag=TARGET_FLAG,
        measurement_columns=measurement_columns,
        task_mode=task_mode,
        good_labels=GOOD_LABELS,
        issue_labels=ISSUE_LABELS,
        model_row_limit=None,
    )
    cluster_source_df["issue"] = cluster_source_df["issue"].astype(int)
    cluster_source_df["window_start"] = pd.to_datetime(cluster_source_df["Time UTC"], utc=True)
    cluster_source_df["window_end"] = pd.to_datetime(cluster_source_df["Time UTC"], utc=True)
    cluster_source_df["issue_rate"] = cluster_source_df["issue"].astype(float)
    kmeans_feature_columns = [column for column in feature_columns if column in cluster_source_df.columns]
    cluster_x_column = PLOT_MEASUREMENT_COLUMN
    cluster_y_column = PLOT_SECONDARY_COLUMN
    cluster_axis_label_suffix = "value"

if cluster_x_column not in cluster_source_df.columns or cluster_y_column not in cluster_source_df.columns:
    raise ValueError(
        f"k-means plotting columns are missing for profile {DATASET_PROFILE_ID}: "
        f"{cluster_x_column!r}, {cluster_y_column!r}"
    )

window_df, cluster_summary = fit_kmeans(
    cluster_source_df,
    n_clusters=KMEANS_CONFIG["n_clusters"],
    seed=SEED,
    n_init=KMEANS_CONFIG["n_init"],
    feature_mode=KMEANS_FEATURE_MODE,
    feature_columns=kmeans_feature_columns,
)
display(cluster_summary.round({"mean_issue_rate": 3, "max_issue_rate": 3, "avg_distance": 3}))


In [ ]:
# Plot a readable sample of clustered points so the scatter does not become an unreadable blob.
plot_window_df = evenly_spaced_take(window_df.sort_values("window_start"), 5000)
cluster_ids = sorted(cluster_summary.index.tolist())
cluster_colors = plt.cm.tab10(np.linspace(0, 1, len(cluster_ids)))
cluster_palette = {cluster_id: cluster_colors[idx] for idx, cluster_id in enumerate(cluster_ids)}

fig, axes = plt.subplots(1, 2, figsize=(17, 6))
cluster_centers = (
    plot_window_df.groupby("cluster")[[cluster_x_column, cluster_y_column]]
    .mean()
    .reindex(cluster_ids)
)

legend_handles = []
for cluster_id in cluster_ids:
    subset = plot_window_df[plot_window_df["cluster"] == cluster_id]
    axes[0].scatter(
        subset[cluster_x_column],
        subset[cluster_y_column],
        s=18,
        alpha=0.55,
        color=cluster_palette[cluster_id],
        edgecolors="none",
    )
    axes[0].scatter(
        cluster_centers.loc[cluster_id, cluster_x_column],
        cluster_centers.loc[cluster_id, cluster_y_column],
        s=220,
        marker="*",
        color=cluster_palette[cluster_id],
        edgecolors="black",
        linewidths=0.8,
    )
    legend_handles.append(
        Line2D(
            [0],
            [0],
            marker="o",
            color="w",
            markerfacecolor=cluster_palette[cluster_id],
            markersize=8,
            label=(
                f"Cluster {cluster_id} | n={int(cluster_summary.loc[cluster_id, 'window_count'])} | "
                f"mean issue={cluster_summary.loc[cluster_id, 'mean_issue_rate']:.2f}"
            ),
        )
    )

axes[0].set_title(f"k-means clusters in {KMEANS_FEATURE_MODE.replace('_', ' ')} feature space")
axes[0].set_xlabel(f"{PLOT_MEASUREMENT_COLUMN} ({cluster_axis_label_suffix})")
axes[0].set_ylabel(f"{PLOT_SECONDARY_COLUMN} ({cluster_axis_label_suffix})")
axes[0].grid(alpha=0.25)
axes[0].legend(handles=legend_handles, title="Cluster legend", loc="upper left", bbox_to_anchor=(1.02, 1.0), frameon=True)

bar_positions = np.arange(len(cluster_ids))
bar_values = [cluster_summary.loc[cluster_id, "mean_issue_rate"] for cluster_id in cluster_ids]
bar_colors = [cluster_palette[cluster_id] for cluster_id in cluster_ids]
axes[1].bar(bar_positions, bar_values, color=bar_colors, edgecolor="black", linewidth=0.6)
axes[1].set_xticks(bar_positions)
axes[1].set_xticklabels([f"Cluster {cluster_id}" for cluster_id in cluster_ids], rotation=30)
axes[1].set_ylim(0, 1.05)
axes[1].set_ylabel("Mean issue rate")
axes[1].set_title("Average issue rate by cluster")
for idx, cluster_id in enumerate(cluster_ids):
    axes[1].text(
        idx,
        bar_values[idx] + 0.02,
        f"n={int(cluster_summary.loc[cluster_id, 'window_count'])}\n{bar_values[idx]:.1%}",
        ha="center",
        va="bottom",
        fontsize=9,
    )

plt.tight_layout()
plt.show()

interesting_windows = window_df.sort_values(
    ["issue_rate", "distance_to_centroid"],
    ascending=[False, False],
).head(10)
interesting_windows = interesting_windows[
    ["window_start", "window_end", "source_file", "cluster", "issue_rate", "distance_to_centroid"]
].copy()
interesting_windows["source_file"] = interesting_windows["source_file"].map(clean_source_file_label)
display(interesting_windows.round({"issue_rate": 3, "distance_to_centroid": 3}))


### Looking At Real Time-Series Windows From Each Cluster

The scatter plot shows where clusters sit in feature space, but it does not show what the underlying sensor behavior actually looked like.

The next plot closes that loop. For each cluster, we pick a representative window that sits close to its centroid, then:

- show a wider time-series context from the original row-level parquet,
- show the true QC-flag regions from that underlying data,
- highlight the exact window used in k-means,
- mark the datapoints inside that highlighted window.

This makes it much easier to interpret clusters as real operating regimes rather than abstract colored dots.


In [ ]:
cluster_example_figure, cluster_example_windows = plot_cluster_window_examples(
    window_df,
    source_to_row_part=source_to_row_part,
    measurement_column=PLOT_MEASUREMENT_COLUMN,
    secondary_column=PLOT_SECONDARY_COLUMN,
    target_flag=TARGET_FLAG,
    good_labels=GOOD_LABELS,
    examples_per_cluster=KMEANS_EXAMPLES_PER_CLUSTER,
    context_points=KMEANS_EXAMPLE_CONTEXT_POINTS,
    highlight_alpha=KMEANS_EXAMPLE_HIGHLIGHT_ALPHA,
)
plt.show()
display(cluster_example_windows)


### Date-Range Demo: See Which Clusters Appear in a Specific Interval

k-means does not predict QC flags directly. Instead, it groups short windows with similar summary behavior.

This demo lets us ask: **what kinds of windows show up inside one selected time range, and how do their issue rates differ?**


In [ ]:
KMEANS_RANGE_START = "2026-02-9T12:56:27.707000+00:00"
KMEANS_RANGE_END = "2026-02-11T12:56:27.707000+00:00"
KMEANS_AUTO_SELECT_TEST_RANGE = True
KMEANS_MAX_POINTS_TO_PLOT = 800

print(
    {
        "KMEANS_RANGE_START": KMEANS_RANGE_START,
        "KMEANS_RANGE_END": KMEANS_RANGE_END,
        "KMEANS_AUTO_SELECT_TEST_RANGE": KMEANS_AUTO_SELECT_TEST_RANGE,
        "KMEANS_MAX_POINTS_TO_PLOT": KMEANS_MAX_POINTS_TO_PLOT,
    }
)


In [ ]:
kmeans_range_selection = select_time_range(
    test_df,
    time_column="Time UTC",
    label_column=TARGET_FLAG,
    start=KMEANS_RANGE_START,
    end=KMEANS_RANGE_END,
    auto_select=KMEANS_AUTO_SELECT_TEST_RANGE,
    max_points=KMEANS_MAX_POINTS_TO_PLOT,
)

kmeans_interval_rows = load_rows_for_time_range(
    metadata,
    row_cache_dir=Path(ROW_CACHE_DIR),
    start=kmeans_range_selection["start"],
    end=kmeans_range_selection["end"],
    columns=ROW_USE_COLUMNS,
)
kmeans_interval_windows = window_df[
    (window_df["window_start"] <= kmeans_range_selection["end"])
    & (window_df["window_end"] >= kmeans_range_selection["start"])
].copy()

if kmeans_interval_rows.empty or kmeans_interval_windows.empty:
    print("No overlapping k-means windows were found in the requested range.")
else:
    kmeans_interval_origin = infer_interval_origin(
        kmeans_range_selection["start"],
        kmeans_range_selection["end"],
        {"train": train_df, "validation": valid_df, "test": test_df},
    )
    kmeans_interval_intervals = merge_adjacent_intervals(
        kmeans_interval_windows.rename(
            columns={"window_start": "start", "window_end": "end", "cluster": "label"}
        )[["start", "end", "label"]]
    )
    kmeans_cluster_counts = (
        kmeans_interval_windows.groupby("cluster")
        .agg(
            window_count=("cluster", "size"),
            mean_issue_rate=("issue_rate", "mean"),
            max_issue_rate=("issue_rate", "max"),
        )
        .sort_index()
    )

    print(
        {
            "selection_mode": kmeans_range_selection["selection_mode"],
            "selected_priority_flag": kmeans_range_selection["selected_label"],
            "interval_origin": kmeans_interval_origin,
            "range_start": kmeans_range_selection["start"].isoformat(),
            "range_end": kmeans_range_selection["end"].isoformat(),
            "row_points_in_interval": int(len(kmeans_interval_rows)),
            "window_count_in_interval": int(len(kmeans_interval_windows)),
        }
    )
    display(kmeans_cluster_counts.round(3))

    kmeans_demo_figure = plot_time_series_with_bands(
        kmeans_interval_rows,
        band_specs=[
            {
                "title": "k-means clusters",
                "intervals": kmeans_interval_intervals,
                "palette": cluster_palette,
            }
        ],
        measurement_column=PLOT_MEASUREMENT_COLUMN,
        secondary_column=PLOT_SECONDARY_COLUMN,
        max_points=KMEANS_MAX_POINTS_TO_PLOT,
        title="k-means cluster assignments on a selected time range",
    )
    plt.show()


### Reflection Questions: k-means

1. Do these cluster spans look like real operating regimes, or do some clusters still seem hard to interpret physically?
2. If you changed `n_clusters`, which regions in this interval do you expect would split apart or merge together?
3. How closely do the cluster spans line up with QC-flag changes, and what does that say about the usefulness of unsupervised learning here?

Add your own notes or follow-up questions here:

- 
- 
- 


## Part 6 — 1D CNN

Now we keep short stretches of signal together as one training example and see how a sequence model behaves when it learns local time patterns directly.


### Sequential Model: 1D CNN

A **convolutional neural network (CNN)** applies small learnable filters across an ordered signal. For images that order is two-dimensional. In this notebook, the order is **time**.

The key idea is:

1. a short filter looks at a local pattern,
2. the same filter slides across the whole sequence,
3. later layers combine those detected patterns into a final prediction.

For this scalar QC task, a 1D CNN can learn patterns such as:

- sudden spikes or drops,
- flat segments,
- repeated local shapes across several sensor channels.

Why this is useful here:

- Random Forest treats each row mostly as a tabular snapshot,
- CNN keeps a short stretch of signal together as one example,
- the training loop also gives us a chance to teach batching, validation checkpoints, and early stopping.

The CNN section runs by default in this notebook and follows standard training practice:

- train / validation / test split,
- class-aware loss weighting,
- best-checkpoint saving based on validation F1,
- early stopping,
- mini-batch training through `DataLoader`,
- pinned-memory and multi-worker loading when the local machine supports it.

Useful references: [PyTorch tutorial on defining neural networks](https://docs.pytorch.org/tutorials/beginner/blitz/neural_networks_tutorial), [PyTorch DataLoader docs](https://pytorch.org/docs/stable/data.html), [PyTorch performance tuning guide](https://pytorch.org/tutorials/recipes/recipes/tuning_guide.html)


![Convolutional network diagram](../assets/session1/convolutional_network.png)

Diagram idea: small filters slide across the input, produce feature maps, and later layers combine those maps into a final prediction.

In this notebook the same logic is applied to **1D time windows** rather than 2D images.

Image credit: Aphex34, Wikimedia Commons, CC BY-SA 4.0. Source: [File:Convolutional Network.png](https://commons.wikimedia.org/wiki/File:Convolutional_Network.png)


### 🎛️ CNN Settings

These settings control the baseline 1D CNN and its training loop.

Model-shape variables:

- `window_size`: number of time steps in each training window.
- `output_mode`: choose `"window"` for one label per window or `"per_timestep"` for one label at every time step.
- `conv_channels`: number of filters in each convolution layer.
- `dropout`: dropout probability used for regularization.
- `label_reduction`: how row-level labels become one window label. This only matters when `output_mode="window"`.

Optimization variables:

- `epochs`: maximum number of passes through the training data.
- `batch_size`: number of windows per optimization step.
- `learning_rate`: optimizer step size.
- `weight_decay`: L2-style regularization for the optimizer.
- `patience` and `min_delta`: early-stopping controls.
- `lr_decay_factor` and `lr_patience`: learning-rate scheduler controls.
- `gradient_clip_norm`: cap on gradient size to stabilize training.

DataLoader variables:

- `num_workers`: worker processes for loading batches.
- `pin_memory`: can speed up host-to-GPU transfer.
- `persistent_workers`: keeps workers alive between epochs.
- `prefetch_factor`: how many batches each worker prepares ahead of time.

Dataset-aware default:

- the turbidity and conductivity-plug profiles start in `"per_timestep"` mode because short local events are easy to lose inside mixed windows,
- the other profiles start in `"window"` mode to keep the first baseline simpler.


In [ ]:
CNN_CONFIG = {
    "window_size": 128,
    "output_mode": DEFAULT_SEQUENCE_OUTPUT_MODE,
    "epochs": 10,
    "batch_size": 128,
    "learning_rate": 1e-3,
    "weight_decay": 1e-4,
    "patience": 3,
    "min_delta": 0.001,
    "dropout": 0.2,
    "lr_decay_factor": 0.5,
    "lr_patience": 1,
    "gradient_clip_norm": 1.0,
    "conv_channels": [32, 64],
    "label_reduction": "worst",
}

CNN_LOADER_CONFIG = {
    "num_workers": min(4, os.cpu_count() or 1),
    "pin_memory": True,
    "persistent_workers": True,
    "prefetch_factor": 2,
}

display(pd.Series(CNN_CONFIG, name="value").rename_axis("cnn_parameter").to_frame())
display(pd.Series(CNN_LOADER_CONFIG, name="value").rename_axis("loader_parameter").to_frame())


### Utility For Window Labels

The CNN and transformer sections now support two output styles:

- `"window"`: one prediction for the whole window,
- `"per_timestep"`: one prediction for every point inside the window.

When you choose `"window"`, we need one short rule for turning several row-level QC flags inside a window into one label. The helper below keeps that reduction in one place so the later cells stay easier to read.


In [ ]:
# Reduce row-level labels to one window-level label for sequence models.
def reduce_window_target(
    values: np.ndarray,
    mode: str,
    severity_order: tuple[int, ...] = (1, 3, 4, 9),
) -> int:
    labels = [int(value) for value in values if pd.notna(value)]
    if not labels:
        return severity_order[0]
    effective_order = tuple(sorted(set(severity_order).union(labels)))
    severity_rank = {label: index for index, label in enumerate(effective_order)}
    if mode == "worst":
        return max(labels, key=lambda label: severity_rank.get(label, -1))
    counts = pd.Series(labels).value_counts()
    tied_labels = counts[counts == counts.max()].index.tolist()
    return max(tied_labels, key=lambda label: severity_rank.get(int(label), -1))


### CNN Step 1: Turn Rows into Windows

A CNN expects a fixed-size tensor, not an arbitrary-length table.

In this step we:

- collect the measurement columns we want to model,
- group consecutive rows into fixed windows,
- either reduce many row-level QC flags into one window label or keep one target per timestamp,
- split the windows with the active train / validation / test strategy,
- normalize each sensor channel using only the training split.

One important detail: the baseline CNN does **not** take the full Random Forest feature table as input. It sees windows of the selected measurement columns only. That means it gets more temporal context than the RF, but less hand-engineered context.

This is a good place to pause and ask: what information do we lose when we compress several row labels into one window label, and when is that simplification actually helpful?


In [ ]:
if not RUN_CNN_APPENDIX:
    print("CNN appendix skipped. Set RUN_CNN_APPENDIX = True in the configuration cell to enable it.")
    CNN_READY = False
else:
    try:
        import torch
        from torch import nn
        from torch.utils.data import DataLoader, TensorDataset
        CNN_READY = True
        print({
            "torch": torch.__version__,
            "cuda_available": torch.cuda.is_available(),
            "device_name": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
        })
    except ImportError as exc:
        CNN_READY = False
        print("PyTorch is not installed in this environment.")
        raise SystemExit(exc)


In [ ]:
if not CNN_READY:
    CNN_RUN = False
    print("CNN training cell skipped.")
else:
    sequence_split_frames_materialized = materialize_reviewed_split_frames(
        selected_paths,
        {"train": train_df, "validation": valid_df, "test": test_df},
        columns=ROW_USE_COLUMNS,
        target_flag=TARGET_FLAG,
        task_mode=task_mode,
        good_labels=GOOD_LABELS,
        issue_labels=ISSUE_LABELS,
    )
    WINDOW_SIZE = CNN_CONFIG["window_size"]
    CNN_OUTPUT_MODE = CNN_CONFIG["output_mode"]
    cnn_bundle = build_sequence_split_bundle_from_frames(
        sequence_split_frames_materialized,
        measurement_columns=measurement_columns,
        target_column="model_target",
        task_mode=task_mode,
        output_mode=CNN_OUTPUT_MODE,
        window_size=WINDOW_SIZE,
        label_reduction=CNN_CONFIG["label_reduction"],
    )
    class_labels = cnn_bundle.class_labels

    if task_mode == "multiclass" and len(class_labels) < 2:
        CNN_RUN = False
        print(
            {
                "output_mode": CNN_OUTPUT_MODE,
                "windows_total": int(len(cnn_bundle.X_train) + len(cnn_bundle.X_valid) + len(cnn_bundle.X_test)),
                "active_labels": class_labels,
                "skip_reason": "Need at least two active classes to train the CNN in multiclass mode.",
            }
        )
    elif len(cnn_bundle.X_train) == 0 or len(cnn_bundle.X_valid) == 0 or len(cnn_bundle.X_test) == 0:
        CNN_RUN = False
        print(
            {
                "output_mode": CNN_OUTPUT_MODE,
                "split_strategy": BENCHMARK_SPLIT_STRATEGY,
                "train_subset_strategy": TRAIN_SUBSET_STRATEGY,
                "skip_reason": "At least one split produced no full windows for this configuration.",
            }
        )
    else:
        # Move to channel-first layout expected by Conv1D.
        X_train = np.transpose(cnn_bundle.X_train, (0, 2, 1))
        X_valid = np.transpose(cnn_bundle.X_valid, (0, 2, 1))
        X_test = np.transpose(cnn_bundle.X_test, (0, 2, 1))
        y_train = cnn_bundle.y_train
        y_valid = cnn_bundle.y_valid
        y_test = cnn_bundle.y_test

        # Normalize each sensor channel using only the training split.
        channel_mean = X_train.mean(axis=(0, 2), keepdims=True)
        channel_std = X_train.std(axis=(0, 2), keepdims=True) + 1e-6
        X_train = (X_train - channel_mean) / channel_std
        X_valid = (X_valid - channel_mean) / channel_std
        X_test = (X_test - channel_mean) / channel_std

        CNN_RUN = True
        print(
            {
                "output_mode": CNN_OUTPUT_MODE,
                "split_strategy": BENCHMARK_SPLIT_STRATEGY,
                "train_subset_strategy": TRAIN_SUBSET_STRATEGY,
                "benchmark_split_block_rows": BENCHMARK_SPLIT_BLOCK_ROWS if BENCHMARK_SPLIT_STRATEGY in {"interleaved_blocks", "episode_aware"} else None,
                "windows_total": int(len(X_train) + len(X_valid) + len(X_test)),
                "train_windows": int(len(X_train)),
                "validation_windows": int(len(X_valid)),
                "test_windows": int(len(X_test)),
                "window_size": int(WINDOW_SIZE),
                "channels": int(len(measurement_columns)),
            }
        )


### CNN Step 2: Build the Model and the DataLoaders

Here we create four pieces:

- the neural network itself,
- the loss function,
- the optimizer and learning-rate scheduler,
- the `DataLoader` objects that stream mini-batches during training.

The output toggle changes the final prediction head:

- in `"window"` mode, the CNN pools across time and emits one label for the whole window,
- in `"per_timestep"` mode, the CNN keeps the time axis and emits one label per point.

Notice that the `DataLoader` keeps the full dataset in CPU memory and feeds the GPU one batch at a time. That is usually much more efficient than trying to move every window onto the GPU all at once.


In [ ]:
if not CNN_RUN:
    print("CNN model setup skipped.")
else:
    torch.manual_seed(SEED)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    if device.type == "cuda":
        torch.backends.cudnn.benchmark = True

    class TinyQCNet(nn.Module):
        def __init__(self, channels: int, output_dim: int) -> None:
            super().__init__()
            self.output_dim = output_dim
            self.features = nn.Sequential(
                nn.Conv1d(channels, CNN_CONFIG["conv_channels"][0], kernel_size=7, padding=3),
                nn.ReLU(),
                nn.Conv1d(CNN_CONFIG["conv_channels"][0], CNN_CONFIG["conv_channels"][1], kernel_size=5, padding=2),
                nn.ReLU(),
                nn.Dropout(CNN_CONFIG["dropout"]),
            )
            if CNN_OUTPUT_MODE == "window":
                self.pool = nn.AdaptiveAvgPool1d(1)
                self.head = nn.Linear(CNN_CONFIG["conv_channels"][1], output_dim)
            else:
                self.head = nn.Conv1d(CNN_CONFIG["conv_channels"][1], output_dim, kernel_size=1)

        def forward(self, x: torch.Tensor) -> torch.Tensor:
            x = self.features(x)
            if CNN_OUTPUT_MODE == "window":
                x = self.pool(x).squeeze(-1)
                return self.head(x)
            logits = self.head(x)
            if self.output_dim == 1:
                return logits.squeeze(1)
            return logits.transpose(1, 2)

    if task_mode == "multiclass":
        train_targets_tensor = torch.from_numpy(y_train).long()
        valid_targets_tensor = torch.from_numpy(y_valid).long()
        test_targets_tensor = torch.from_numpy(y_test).long()
        class_counts = np.bincount(y_train.reshape(-1), minlength=len(class_labels)).clip(min=1)
        class_weights = y_train.size / (len(class_counts) * class_counts)
        loss_fn = nn.CrossEntropyLoss(weight=torch.tensor(class_weights, dtype=torch.float32, device=device))
        output_dim = len(class_labels)
    else:
        train_targets_tensor = torch.from_numpy(y_train.astype(np.float32))
        valid_targets_tensor = torch.from_numpy(y_valid.astype(np.float32))
        test_targets_tensor = torch.from_numpy(y_test.astype(np.float32))
        positive_count = max(float(y_train.sum()), 1.0)
        negative_count = max(float(y_train.size - y_train.sum()), 1.0)
        pos_weight = torch.tensor([negative_count / positive_count], dtype=torch.float32, device=device)
        loss_fn = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
        output_dim = 1

    train_dataset = TensorDataset(torch.from_numpy(X_train).float(), train_targets_tensor)
    valid_dataset = TensorDataset(torch.from_numpy(X_valid).float(), valid_targets_tensor)
    test_dataset = TensorDataset(torch.from_numpy(X_test).float(), test_targets_tensor)

    loader_kwargs = {}
    num_workers = int(CNN_LOADER_CONFIG["num_workers"])
    if num_workers > 0:
        loader_kwargs["num_workers"] = num_workers
        loader_kwargs["persistent_workers"] = bool(CNN_LOADER_CONFIG["persistent_workers"])
        loader_kwargs["prefetch_factor"] = int(CNN_LOADER_CONFIG["prefetch_factor"])
    if CNN_LOADER_CONFIG["pin_memory"]:
        loader_kwargs["pin_memory"] = True
    use_non_blocking = bool(loader_kwargs.get("pin_memory", False) and device.type == "cuda")

    train_loader = DataLoader(
        train_dataset,
        batch_size=CNN_CONFIG["batch_size"],
        shuffle=True,
        **loader_kwargs,
    )
    valid_loader = DataLoader(
        valid_dataset,
        batch_size=CNN_CONFIG["batch_size"],
        shuffle=False,
        **loader_kwargs,
    )
    test_loader = DataLoader(
        test_dataset,
        batch_size=CNN_CONFIG["batch_size"],
        shuffle=False,
        **loader_kwargs,
    )

    model = TinyQCNet(channels=len(measurement_columns), output_dim=output_dim).to(device)
    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=CNN_CONFIG["learning_rate"],
        weight_decay=CNN_CONFIG["weight_decay"],
    )
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer,
        mode="max",
        factor=CNN_CONFIG["lr_decay_factor"],
        patience=CNN_CONFIG["lr_patience"],
    )

    best_metric = -np.inf
    best_epoch = 0
    patience_counter = 0
    history = []
    best_state = None

    def run_epoch(loader, training: bool) -> tuple[float, np.ndarray, np.ndarray]:
        model.train(training)
        total_loss = 0.0
        predictions = []
        targets = []
        for batch_x, batch_y in loader:
            batch_x = batch_x.to(device, non_blocking=use_non_blocking)
            batch_y = batch_y.to(device, non_blocking=use_non_blocking)
            with torch.set_grad_enabled(training):
                logits = model(batch_x)
                if task_mode == "multiclass":
                    if CNN_OUTPUT_MODE == "window":
                        loss = loss_fn(logits, batch_y)
                        batch_predictions = logits.argmax(dim=1)
                    else:
                        loss = loss_fn(logits.permute(0, 2, 1), batch_y)
                        batch_predictions = logits.argmax(dim=-1)
                else:
                    loss = loss_fn(logits, batch_y)
                    batch_predictions = (torch.sigmoid(logits) >= 0.5).long()
                if training:
                    optimizer.zero_grad(set_to_none=True)
                    loss.backward()
                    if CNN_CONFIG["gradient_clip_norm"]:
                        torch.nn.utils.clip_grad_norm_(model.parameters(), CNN_CONFIG["gradient_clip_norm"])
                    optimizer.step()
            total_loss += float(loss.item()) * len(batch_x)
            predictions.append(batch_predictions.detach().cpu().reshape(-1).numpy())
            targets.append(batch_y.detach().cpu().reshape(-1).numpy())
        predictions_array = np.concatenate(predictions) if predictions else np.array([])
        targets_array = np.concatenate(targets) if targets else np.array([])
        average_loss = total_loss / max(len(loader.dataset), 1)
        return average_loss, predictions_array, targets_array

    print(
        {
            "device": str(device),
            "output_mode": CNN_OUTPUT_MODE,
            "train_batches": len(train_loader),
            "validation_batches": len(valid_loader),
            "test_batches": len(test_loader),
            "loader_config": CNN_LOADER_CONFIG,
        }
    )


### CNN Step 3: Train with Validation Monitoring

This cell runs the optimization loop.

We are following standard training practice:

- train on the training split,
- score the model on the validation split after each epoch,
- save the best checkpoint seen so far,
- stop early if the validation metric stops improving.

Watch the printed validation F1 as the run progresses. When `output_mode="window"`, that F1 is computed per window. When `output_mode="per_timestep"`, it is computed across all timestamps in the validation windows.


In [ ]:
if not CNN_RUN:
    print("CNN fit skipped.")
else:
    for epoch in range(1, CNN_CONFIG["epochs"] + 1):
        train_loss, _, _ = run_epoch(train_loader, training=True)
        valid_loss, valid_preds, valid_targets = run_epoch(valid_loader, training=False)
        valid_f1 = f1_score(valid_targets, valid_preds, average=report_average(task_mode), zero_division=0)
        scheduler.step(valid_f1)
        history.append(
            {
                "epoch": epoch,
                "train_loss": train_loss,
                "valid_loss": valid_loss,
                "valid_f1": valid_f1,
                "learning_rate": optimizer.param_groups[0]["lr"],
            }
        )
        print(
            f"Epoch {epoch:>2} | train_loss={train_loss:.4f} | valid_loss={valid_loss:.4f} | valid_f1={valid_f1:.4f}"
        )

        if valid_f1 > best_metric + CNN_CONFIG["min_delta"]:
            best_metric = valid_f1
            best_epoch = epoch
            patience_counter = 0
            best_state = copy.deepcopy(model.state_dict())
            torch.save(
                {
                    "model_state_dict": best_state,
                    "task_mode": task_mode,
                    "class_labels": class_labels,
                    "feature_columns": measurement_columns,
                    "window_size": WINDOW_SIZE,
                    "output_mode": CNN_OUTPUT_MODE,
                    "config": {**CNN_CONFIG, **CNN_LOADER_CONFIG},
                },
                CNN_MODEL_PATH,
            )
        else:
            patience_counter += 1
            if patience_counter >= CNN_CONFIG["patience"]:
                print(f"Early stopping triggered at epoch {epoch}")
                break


### CNN Step 4: Reload the Best Checkpoint and Evaluate

Training loss alone is not enough. In this final step we:

- reload the weights that gave the best validation F1,
- run the held-out test split once,
- inspect the classification report and confusion matrix.

This keeps the evaluation honest and makes it clear that the validation set guided model selection, while the test set is reserved for final reporting.


In [ ]:
if not CNN_RUN:
    print("CNN evaluation skipped.")
else:
    if best_state is None:
        raise RuntimeError("CNN training did not produce a valid checkpoint")

    model.load_state_dict(best_state)
    test_loss, test_preds, test_targets = run_epoch(test_loader, training=False)

    print(pd.DataFrame(history).tail())
    print(
        {
            "output_mode": CNN_CONFIG["output_mode"],
            "windows": len(train_dataset) + len(valid_dataset) + len(test_dataset),
            "device": str(device),
            "best_validation_f1": float(best_metric),
            "best_epoch": best_epoch,
            "test_loss": float(test_loss),
            "saved_model": str(CNN_MODEL_PATH),
            "loader_config": CNN_LOADER_CONFIG,
        }
    )

    if task_mode == "multiclass":
        report_labels = list(range(len(class_labels)))
        report_names = [str(label) for label in class_labels]
    else:
        report_labels = [0, 1]
        report_names = ["0", "1"]

    print(
        classification_report(
            test_targets,
            test_preds,
            labels=report_labels,
            target_names=report_names,
            zero_division=0,
        )
    )

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    history_frame = pd.DataFrame(history)
    axes[0].plot(history_frame["epoch"], history_frame["train_loss"], label="train")
    axes[0].plot(history_frame["epoch"], history_frame["valid_loss"], label="validation")
    axes[0].set_title("CNN loss history")
    axes[0].legend()
    ConfusionMatrixDisplay.from_predictions(
        test_targets,
        test_preds,
        labels=report_labels,
        display_labels=report_names,
        normalize="true",
        xticks_rotation=45,
        ax=axes[1],
    )
    axes[1].set_title("CNN test confusion matrix")
    plt.tight_layout()
    plt.show()


### Date-Range Demo: See CNN Predictions on a Specific Interval

This demo follows the same `output_mode` toggle as the training section:

- `"window"` shows one true/predicted label span per window,
- `"per_timestep"` shows one true/predicted label span per timestamp.


In [ ]:
CNN_RANGE_START = None
CNN_RANGE_END = None
CNN_AUTO_SELECT_TEST_RANGE = True
CNN_MAX_POINTS_TO_PLOT = 1024

print(
    {
        "CNN_RANGE_START": CNN_RANGE_START,
        "CNN_RANGE_END": CNN_RANGE_END,
        "CNN_AUTO_SELECT_TEST_RANGE": CNN_AUTO_SELECT_TEST_RANGE,
        "CNN_MAX_POINTS_TO_PLOT": CNN_MAX_POINTS_TO_PLOT,
    }
)


In [ ]:
if not CNN_RUN:
    print("CNN date-range demo skipped because the CNN was not trained in this run.")
else:
    cnn_range_selection = select_time_range(
        test_df,
        time_column="Time UTC",
        label_column=TARGET_FLAG,
        start=CNN_RANGE_START,
        end=CNN_RANGE_END,
        auto_select=CNN_AUTO_SELECT_TEST_RANGE,
        max_points=CNN_MAX_POINTS_TO_PLOT,
    )
    cnn_interval_metrics = None

    cnn_interval_rows = load_rows_for_time_range(
        metadata,
        row_cache_dir=Path(ROW_CACHE_DIR),
        start=cnn_range_selection["start"],
        end=cnn_range_selection["end"],
        columns=ROW_USE_COLUMNS,
    )

    if cnn_interval_rows.empty:
        print("No row-level data was found in the requested CNN range.")
    else:
        cnn_interval_model_df, _, _ = build_model_frame(
            cnn_interval_rows,
            target_flag=TARGET_FLAG,
            measurement_columns=measurement_columns,
            task_mode=task_mode,
            model_row_limit=None,
        )
        cnn_interval_model_df = cnn_interval_model_df[
            (cnn_interval_model_df["Time UTC"] >= cnn_range_selection["start"])
            & (cnn_interval_model_df["Time UTC"] <= cnn_range_selection["end"])
        ].reset_index(drop=True)

        cnn_interval_origin = infer_interval_origin(
            cnn_range_selection["start"],
            cnn_range_selection["end"],
            {"train": train_df, "validation": valid_df, "test": test_df},
        )
        cnn_plot_palette = DEFAULT_FLAG_PALETTE if task_mode == "multiclass" else {0: "#1f77b4", 1: "#d62728"}

        if CNN_CONFIG["output_mode"] == "window":
            cnn_interval_bundle = build_window_classification_interval_data(
                cnn_interval_model_df,
                feature_columns=measurement_columns,
                target_column="model_target",
                task_mode=task_mode,
                window_size=CNN_CONFIG["window_size"],
                label_reduction=CNN_CONFIG["label_reduction"],
            )

            if cnn_interval_bundle["window_frame"].empty:
                print("The selected CNN range is shorter than one full window after preprocessing, so the window-level demo is skipped.")
            else:
                cnn_predicted_labels = predict_cnn_window_model(
                    model,
                    cnn_interval_bundle["raw_sequences"],
                    task_mode=task_mode,
                    class_labels=class_labels,
                    device=str(device),
                    channel_mean=channel_mean,
                    channel_std=channel_std,
                    batch_size=CNN_CONFIG["batch_size"],
                )
                cnn_window_frame = cnn_interval_bundle["window_frame"].copy()
                cnn_window_frame["predicted_label"] = cnn_predicted_labels
                cnn_interval_metrics = compute_interval_classification_metrics(
                    cnn_window_frame["true_label"],
                    cnn_window_frame["predicted_label"],
                    labels=class_labels,
                    average=report_average(task_mode),
                    target_names=[str(label) for label in class_labels],
                )
                cnn_true_intervals = merge_adjacent_intervals(
                    cnn_window_frame.rename(columns={"window_start": "start", "window_end": "end", "true_label": "label"})[
                        ["start", "end", "label"]
                    ]
                )
                cnn_pred_intervals = merge_adjacent_intervals(
                    cnn_window_frame.rename(columns={"window_start": "start", "window_end": "end", "predicted_label": "label"})[
                        ["start", "end", "label"]
                    ]
                )

                print(
                    {
                        "output_mode": CNN_CONFIG["output_mode"],
                        "selection_mode": cnn_range_selection["selection_mode"],
                        "selected_priority_flag": cnn_range_selection["selected_label"],
                        "interval_origin": cnn_interval_origin,
                        "range_start": cnn_range_selection["start"].isoformat(),
                        "range_end": cnn_range_selection["end"].isoformat(),
                        "window_count_in_interval": int(len(cnn_window_frame)),
                        "interval_f1": cnn_interval_metrics["f1"],
                    }
                )
                print(cnn_interval_metrics["report_text"])
                display(
                    pd.DataFrame(
                        {
                            "true_count": cnn_window_frame["true_label"].value_counts().sort_index(),
                            "predicted_count": cnn_window_frame["predicted_label"].value_counts().sort_index(),
                        }
                    ).fillna(0).astype(int)
                )

                cnn_demo_figure = plot_time_series_with_bands(
                    cnn_interval_model_df,
                    band_specs=[
                        {"title": "True window label", "intervals": cnn_true_intervals, "palette": cnn_plot_palette},
                        {"title": "CNN window prediction", "intervals": cnn_pred_intervals, "palette": cnn_plot_palette},
                    ],
                    measurement_column=PLOT_MEASUREMENT_COLUMN,
                    secondary_column=PLOT_SECONDARY_COLUMN,
                    max_points=CNN_MAX_POINTS_TO_PLOT,
                    title="Baseline CNN window predictions on a selected time range",
                )
                plt.show()
        else:
            cnn_interval_bundle = build_sequence_label_interval_data(
                cnn_interval_model_df,
                feature_columns=measurement_columns,
                target_column="model_target",
                window_size=CNN_CONFIG["window_size"],
            )

            if len(cnn_interval_bundle["raw_sequences"]) == 0:
                print("The selected CNN range is shorter than one full window after preprocessing, so the per-timestep demo is skipped.")
            else:
                cnn_predicted_labels = predict_sequence_label_cnn(
                    model,
                    cnn_interval_bundle["raw_sequences"],
                    task_mode=task_mode,
                    class_labels=class_labels,
                    device=str(device),
                    channel_mean=channel_mean,
                    channel_std=channel_std,
                    batch_size=CNN_CONFIG["batch_size"],
                )
                cnn_point_frame = pd.DataFrame(
                    {
                        "Time UTC": pd.to_datetime(cnn_interval_bundle["raw_times"].reshape(-1), utc=True),
                        "true_label": cnn_interval_bundle["raw_targets"].reshape(-1).astype(int),
                        "predicted_label": cnn_predicted_labels.reshape(-1).astype(int),
                    }
                )
                cnn_interval_metrics = compute_interval_classification_metrics(
                    cnn_point_frame["true_label"],
                    cnn_point_frame["predicted_label"],
                    labels=class_labels,
                    average=report_average(task_mode),
                    target_names=[str(label) for label in class_labels],
                )
                cnn_true_intervals = merge_adjacent_intervals(
                    build_labeled_intervals(cnn_point_frame, time_column="Time UTC", label_column="true_label")
                )
                cnn_pred_intervals = merge_adjacent_intervals(
                    build_labeled_intervals(cnn_point_frame, time_column="Time UTC", label_column="predicted_label")
                )

                print(
                    {
                        "output_mode": CNN_CONFIG["output_mode"],
                        "selection_mode": cnn_range_selection["selection_mode"],
                        "selected_priority_flag": cnn_range_selection["selected_label"],
                        "interval_origin": cnn_interval_origin,
                        "range_start": cnn_range_selection["start"].isoformat(),
                        "range_end": cnn_range_selection["end"].isoformat(),
                        "point_count_in_interval": int(len(cnn_point_frame)),
                        "interval_f1": cnn_interval_metrics["f1"],
                    }
                )
                print(cnn_interval_metrics["report_text"])
                display(
                    pd.DataFrame(
                        {
                            "true_count": cnn_point_frame["true_label"].value_counts().sort_index(),
                            "predicted_count": cnn_point_frame["predicted_label"].value_counts().sort_index(),
                        }
                    ).fillna(0).astype(int)
                )

                cnn_demo_figure = plot_time_series_with_bands(
                    cnn_interval_model_df,
                    band_specs=[
                        {"title": "True per-timestep label", "intervals": cnn_true_intervals, "palette": cnn_plot_palette},
                        {"title": "CNN per-timestep prediction", "intervals": cnn_pred_intervals, "palette": cnn_plot_palette},
                    ],
                    measurement_column=PLOT_MEASUREMENT_COLUMN,
                    secondary_column=PLOT_SECONDARY_COLUMN,
                    max_points=CNN_MAX_POINTS_TO_PLOT,
                    title="Baseline CNN per-timestep predictions on a selected time range",
                )
                plt.show()

        if cnn_interval_metrics is not None:
            cnn_cm_fig, cnn_cm_ax = plt.subplots(figsize=(5, 4))
            ConfusionMatrixDisplay(
                confusion_matrix=cnn_interval_metrics["confusion_matrix"],
                display_labels=cnn_interval_metrics["display_labels"],
            ).plot(ax=cnn_cm_ax, cmap="Blues", colorbar=False)
            cnn_cm_ax.set_title("CNN confusion matrix on the selected range")
            plt.tight_layout()
            plt.show()


### Reflection Questions: Baseline CNN

1. How much information do we lose when we compress a whole window into one label, and can you see that loss in this interval?
2. If this window-level prediction is too coarse, would you change the window size, the label reduction rule, or the model architecture first?
3. Do the mistakes here look more like a data-representation problem or a training-hyperparameter problem?

Add your own notes or follow-up questions here:

- 
- 
- 


## Part 7 — Transformer

The transformer keeps the same supervised task but changes how the model uses context inside each sequence window.


### Sequential Model: Small Transformer Encoder

If you have used ChatGPT or seen a text-to-image model, you have already interacted with a **transformer**.

Transformers were introduced in 2017 in *Attention Is All You Need*, and they quickly became the dominant architecture for sequence modeling. They are most famous in language, but the underlying math is not tied to text. The same ideas are useful for scalar sensor streams, audio, and video.

A simple reason they matter is that they replaced the older "read one step, then the next step" style used by RNNs and LSTMs.

- **RNN/LSTM:** process sequences step by step, which makes training slower and can make long-range context fade.
- **Transformer:** processes the full window together and learns which positions should pay attention to which other positions.

That core operation is called **self-attention**. In plain language, each timestep asks: *which other timesteps in this window are most relevant to understanding me right now?*

For ONC-style data, that is appealing whenever a local measurement only makes sense in a wider context, such as linking a short anomaly to something that happened much earlier or later in the same window.

In this notebook we use a small **encoder-only transformer** for classification. We are not generating text, so we only keep the encoder side and attach a classifier on top.

If you want a fuller explanation after this section, here are the most useful references:

- [ONC Confluence: Transformers overview](https://internal.oceannetworks.ca/x/mwDcDg)
- [3blue1brown: Attention in transformers, step-by-step](https://www.youtube.com/watch?v=eMlx5fFNoYc)
- [The Illustrated Transformer](https://jalammar.github.io/illustrated-transformer/)
- [Attention Is All You Need](https://arxiv.org/abs/1706.03762)


![Transformer model architecture](../assets/session1/transformer_model_architecture.png)

This is the big-picture map. The original transformer has:

- an **encoder stack** on the left,
- a **decoder stack** on the right.

In this notebook we only use the **encoder stack** and attach a classifier on top. So when you look at the rest of the section, focus mostly on the left half of this picture rather than trying to memorize the whole diagram.

Image credit: Arz, Wikimedia Commons, CC BY-SA 4.0. Source: [File:The-Transformer-model-architecture.png](https://commons.wikimedia.org/wiki/File:The-Transformer-model-architecture.png)


![Detailed encoder self-attention diagram](../assets/session1/transformer_encoder_self_attention_detailed.png)

This picture zooms in on the core idea of **self-attention**.

You do not need to memorize every symbol here. The important story is:

- each position creates a few learned views of itself,
- those views are compared against other positions,
- the model turns those comparisons into attention weights,
- then it blends information from the whole window into a new representation.

In plain language: a timestep can decide which other timesteps are worth listening to.

If you want the clearest visual explanation of this idea, the 3blue1brown video linked above is the best companion resource for this section.

Image credit: dvgodoy, Wikimedia Commons, CC BY-SA 4.0. Source: [File:Encoder self-attention, detailed diagram.png](https://commons.wikimedia.org/wiki/File:Encoder_self-attention,_detailed_diagram.png)


![Positional encoding heatmap](../assets/session1/transformer_positional_encoding.png)

One natural question is: if attention can compare all positions to all other positions, how does the model know **where** each timestep sits in the sequence?

That is the job of **positional encoding**.

This heatmap shows that each position gets its own pattern added to the input representation. That gives the transformer access to order information, so it can tell the difference between "this happened early in the window" and "this happened later".

Image credit: Mdbartosz, Wikimedia Commons, CC BY-SA 4.0. Source: [File:Positional encoding.png](https://commons.wikimedia.org/wiki/File:Positional_encoding.png)


### Transformer Settings

Main model variables:

- `window_size`: number of time steps in each window.
- `output_mode`: choose `"window"` for one label per window or `"per_timestep"` for one label at every time step.
- `d_model`: internal embedding size for each position.
- `nhead`: number of attention heads. This must divide `d_model`.
- `num_layers`: number of stacked encoder layers.
- `dim_feedforward`: hidden size of the feed-forward sublayer inside each encoder block.
- `dropout`: dropout used inside the model.
- `pooling`: how we reduce the sequence to one vector for classification. This only matters when `output_mode="window"`.
- `label_reduction`: how row-level labels become one window label. This only matters when `output_mode="window"`.

Training variables:

- `epochs`, `batch_size`, `learning_rate`, `weight_decay`
- `patience`, `min_delta`
- `lr_decay_factor`, `lr_patience`
- `gradient_clip_norm`

Practical note:

- larger `window_size`, `d_model`, or `num_layers` can improve context capacity,
- but they also increase memory use and training time.

Dataset-aware default:

- the turbidity profile starts in `"per_timestep"` mode because the most interesting events are short and windows are often mixed,
- the other profiles start in `"window"` mode so you can begin with one simpler sequence-classification baseline.

A simple mental model for the main knobs:

- `window_size` controls how much time context the model can see,
- `d_model` controls how much representational space each timestep gets,
- `nhead` controls how many different attention patterns the model can learn in parallel,
- `num_layers` controls how many times the sequence is reprocessed with attention.


In [ ]:
TRANSFORMER_CONFIG = {
    "window_size": 128,
    "output_mode": DEFAULT_SEQUENCE_OUTPUT_MODE,
    "d_model": 64,
    "nhead": 4,
    "num_layers": 2,
    "dim_feedforward": 128,
    "dropout": 0.2,
    "pooling": "mean",
    "label_reduction": "worst",
    "epochs": 8,
    "batch_size": 128,
    "learning_rate": 1e-3,
    "weight_decay": 1e-4,
    "patience": 3,
    "min_delta": 0.001,
    "lr_decay_factor": 0.5,
    "lr_patience": 1,
    "gradient_clip_norm": 1.0,
}

TRANSFORMER_LOADER_CONFIG = {
    "num_workers": min(4, os.cpu_count() or 1),
    "pin_memory": True,
    "persistent_workers": True,
    "prefetch_factor": 2,
}

display(pd.Series(TRANSFORMER_CONFIG, name="value").rename_axis("transformer_parameter").to_frame())
display(pd.Series(TRANSFORMER_LOADER_CONFIG, name="value").rename_axis("loader_parameter").to_frame())


### Transformer Step 1: Prepare Windowed Sequence Data

The transformer uses the same benchmark split as the CNN, but it keeps the sequence in its original time-major layout.

In other words:

- CNN sees `[batch, channels, time]`,
- transformer sees `[batch, time, features]`.

Just like the CNN, the transformer here uses windows of the selected measurement columns rather than the full Random Forest feature table. So it gets temporal context directly from the sequence, not from handcrafted calendar or delta features.

The `output_mode` toggle also matters here:

- `"window"` trains one prediction for the whole window,
- `"per_timestep"` trains one prediction at every position in the window.

That makes this step a nice comparison point between two sequence models trained on the same scalar QC problem.


In [ ]:
if not CNN_READY:
    TRANSFORMER_RUN = False
    print("Transformer training cell skipped.")
else:
    sequence_split_frames_materialized = materialize_reviewed_split_frames(
        selected_paths,
        {"train": train_df, "validation": valid_df, "test": test_df},
        columns=ROW_USE_COLUMNS,
        target_flag=TARGET_FLAG,
        task_mode=task_mode,
        good_labels=GOOD_LABELS,
        issue_labels=ISSUE_LABELS,
    )
    transformer_window_size = TRANSFORMER_CONFIG["window_size"]
    TRANSFORMER_OUTPUT_MODE = TRANSFORMER_CONFIG["output_mode"]
    transformer_bundle = build_sequence_split_bundle_from_frames(
        sequence_split_frames_materialized,
        measurement_columns=measurement_columns,
        target_column="model_target",
        task_mode=task_mode,
        output_mode=TRANSFORMER_OUTPUT_MODE,
        window_size=transformer_window_size,
        label_reduction=TRANSFORMER_CONFIG["label_reduction"],
    )
    transformer_class_labels = transformer_bundle.class_labels

    if task_mode == "multiclass" and len(transformer_class_labels) < 2:
        TRANSFORMER_RUN = False
        print(
            {
                "output_mode": TRANSFORMER_OUTPUT_MODE,
                "windows_total": int(len(transformer_bundle.X_train) + len(transformer_bundle.X_valid) + len(transformer_bundle.X_test)),
                "active_labels": transformer_class_labels,
                "skip_reason": "Need at least two active classes to train the transformer in multiclass mode.",
            }
        )
    elif len(transformer_bundle.X_train) == 0 or len(transformer_bundle.X_valid) == 0 or len(transformer_bundle.X_test) == 0:
        TRANSFORMER_RUN = False
        print(
            {
                "output_mode": TRANSFORMER_OUTPUT_MODE,
                "split_strategy": BENCHMARK_SPLIT_STRATEGY,
                "train_subset_strategy": TRAIN_SUBSET_STRATEGY,
                "skip_reason": "At least one split produced no full windows for this configuration.",
            }
        )
    else:
        X_train = transformer_bundle.X_train
        X_valid = transformer_bundle.X_valid
        X_test = transformer_bundle.X_test
        y_train = transformer_bundle.y_train
        y_valid = transformer_bundle.y_valid
        y_test = transformer_bundle.y_test

        feature_mean = X_train.mean(axis=(0, 1), keepdims=True)
        feature_std = X_train.std(axis=(0, 1), keepdims=True) + 1e-6
        X_train = (X_train - feature_mean) / feature_std
        X_valid = (X_valid - feature_mean) / feature_std
        X_test = (X_test - feature_mean) / feature_std

        TRANSFORMER_RUN = True
        print(
            {
                "output_mode": TRANSFORMER_OUTPUT_MODE,
                "split_strategy": BENCHMARK_SPLIT_STRATEGY,
                "train_subset_strategy": TRAIN_SUBSET_STRATEGY,
                "benchmark_split_block_rows": BENCHMARK_SPLIT_BLOCK_ROWS if BENCHMARK_SPLIT_STRATEGY in {"interleaved_blocks", "episode_aware"} else None,
                "windows_total": int(len(X_train) + len(X_valid) + len(X_test)),
                "train_windows": int(len(X_train)),
                "validation_windows": int(len(X_valid)),
                "test_windows": int(len(X_test)),
                "window_size": int(transformer_window_size),
                "features_per_step": int(len(measurement_columns)),
            }
        )


### Transformer Step 2: Build Attention Blocks, Loss, and DataLoaders

The transformer needs a few ingredients that are specific to attention models:

- an input projection that maps raw sensor features into `d_model`,
- a positional embedding so the model knows where each time step sits in the window,
- stacked encoder blocks with multi-head attention,
- and, in window mode, a pooling rule that turns the whole window into one classification vector.

The output toggle changes the last step:

- in `"window"` mode, we pool the encoded sequence and classify once,
- in `"per_timestep"` mode, we classify every encoded position directly.

Everything else should feel familiar from the CNN section: loss, optimizer, scheduler, and `DataLoader` setup.


In [ ]:
if not TRANSFORMER_RUN:
    print("Transformer model setup skipped.")
else:
    torch.manual_seed(SEED)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    if device.type == "cuda":
        torch.backends.cudnn.benchmark = True

    class TinyQCTransformer(nn.Module):
        def __init__(self, input_dim: int, output_dim: int) -> None:
            super().__init__()
            self.output_dim = output_dim
            self.input_projection = nn.Linear(input_dim, TRANSFORMER_CONFIG["d_model"])
            self.position_embedding = nn.Parameter(
                torch.zeros(1, TRANSFORMER_CONFIG["window_size"], TRANSFORMER_CONFIG["d_model"])
            )
            encoder_layer = nn.TransformerEncoderLayer(
                d_model=TRANSFORMER_CONFIG["d_model"],
                nhead=TRANSFORMER_CONFIG["nhead"],
                dim_feedforward=TRANSFORMER_CONFIG["dim_feedforward"],
                dropout=TRANSFORMER_CONFIG["dropout"],
                activation="gelu",
                batch_first=True,
            )
            self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=TRANSFORMER_CONFIG["num_layers"])
            self.norm = nn.LayerNorm(TRANSFORMER_CONFIG["d_model"])
            self.head = nn.Linear(TRANSFORMER_CONFIG["d_model"], output_dim)

        def forward(self, x: torch.Tensor) -> torch.Tensor:
            x = self.input_projection(x)
            x = x + self.position_embedding[:, : x.size(1)]
            x = self.encoder(x)
            if TRANSFORMER_OUTPUT_MODE == "window":
                if TRANSFORMER_CONFIG["pooling"] == "last":
                    pooled = x[:, -1, :]
                else:
                    pooled = x.mean(dim=1)
                pooled = self.norm(pooled)
                return self.head(pooled)
            x = self.norm(x)
            logits = self.head(x)
            if self.output_dim == 1:
                return logits.squeeze(-1)
            return logits

    if task_mode == "multiclass":
        train_targets_tensor = torch.from_numpy(y_train).long()
        valid_targets_tensor = torch.from_numpy(y_valid).long()
        test_targets_tensor = torch.from_numpy(y_test).long()
        class_counts = np.bincount(y_train.reshape(-1), minlength=len(transformer_class_labels)).clip(min=1)
        class_weights = y_train.size / (len(class_counts) * class_counts)
        loss_fn = nn.CrossEntropyLoss(weight=torch.tensor(class_weights, dtype=torch.float32, device=device))
        output_dim = len(transformer_class_labels)
    else:
        train_targets_tensor = torch.from_numpy(y_train.astype(np.float32))
        valid_targets_tensor = torch.from_numpy(y_valid.astype(np.float32))
        test_targets_tensor = torch.from_numpy(y_test.astype(np.float32))
        positive_count = max(float(y_train.sum()), 1.0)
        negative_count = max(float(y_train.size - y_train.sum()), 1.0)
        pos_weight = torch.tensor([negative_count / positive_count], dtype=torch.float32, device=device)
        loss_fn = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
        output_dim = 1

    train_dataset = TensorDataset(torch.from_numpy(X_train).float(), train_targets_tensor)
    valid_dataset = TensorDataset(torch.from_numpy(X_valid).float(), valid_targets_tensor)
    test_dataset = TensorDataset(torch.from_numpy(X_test).float(), test_targets_tensor)

    loader_kwargs = {}
    num_workers = int(TRANSFORMER_LOADER_CONFIG["num_workers"])
    if num_workers > 0:
        loader_kwargs["num_workers"] = num_workers
        loader_kwargs["persistent_workers"] = bool(TRANSFORMER_LOADER_CONFIG["persistent_workers"])
        loader_kwargs["prefetch_factor"] = int(TRANSFORMER_LOADER_CONFIG["prefetch_factor"])
    if TRANSFORMER_LOADER_CONFIG["pin_memory"]:
        loader_kwargs["pin_memory"] = True
    use_non_blocking = bool(loader_kwargs.get("pin_memory", False) and device.type == "cuda")

    train_loader = DataLoader(train_dataset, batch_size=TRANSFORMER_CONFIG["batch_size"], shuffle=True, **loader_kwargs)
    valid_loader = DataLoader(valid_dataset, batch_size=TRANSFORMER_CONFIG["batch_size"], shuffle=False, **loader_kwargs)
    test_loader = DataLoader(test_dataset, batch_size=TRANSFORMER_CONFIG["batch_size"], shuffle=False, **loader_kwargs)

    model = TinyQCTransformer(input_dim=len(measurement_columns), output_dim=output_dim).to(device)
    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=TRANSFORMER_CONFIG["learning_rate"],
        weight_decay=TRANSFORMER_CONFIG["weight_decay"],
    )
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer,
        mode="max",
        factor=TRANSFORMER_CONFIG["lr_decay_factor"],
        patience=TRANSFORMER_CONFIG["lr_patience"],
    )

    best_metric = -np.inf
    best_epoch = 0
    patience_counter = 0
    history = []
    best_state = None

    def run_epoch(loader, training: bool) -> tuple[float, np.ndarray, np.ndarray]:
        model.train(training)
        total_loss = 0.0
        predictions = []
        targets = []
        for batch_x, batch_y in loader:
            batch_x = batch_x.to(device, non_blocking=use_non_blocking)
            batch_y = batch_y.to(device, non_blocking=use_non_blocking)
            with torch.set_grad_enabled(training):
                logits = model(batch_x)
                if task_mode == "multiclass":
                    if TRANSFORMER_OUTPUT_MODE == "window":
                        loss = loss_fn(logits, batch_y)
                        batch_predictions = logits.argmax(dim=1)
                    else:
                        loss = loss_fn(logits.permute(0, 2, 1), batch_y)
                        batch_predictions = logits.argmax(dim=-1)
                else:
                    loss = loss_fn(logits, batch_y)
                    batch_predictions = (torch.sigmoid(logits) >= 0.5).long()
                if training:
                    optimizer.zero_grad(set_to_none=True)
                    loss.backward()
                    if TRANSFORMER_CONFIG["gradient_clip_norm"]:
                        torch.nn.utils.clip_grad_norm_(model.parameters(), TRANSFORMER_CONFIG["gradient_clip_norm"])
                    optimizer.step()
            total_loss += float(loss.item()) * len(batch_x)
            predictions.append(batch_predictions.detach().cpu().reshape(-1).numpy())
            targets.append(batch_y.detach().cpu().reshape(-1).numpy())
        predictions_array = np.concatenate(predictions) if predictions else np.array([])
        targets_array = np.concatenate(targets) if targets else np.array([])
        average_loss = total_loss / max(len(loader.dataset), 1)
        return average_loss, predictions_array, targets_array

    print(
        {
            "device": str(device),
            "output_mode": TRANSFORMER_OUTPUT_MODE,
            "train_batches": len(train_loader),
            "validation_batches": len(valid_loader),
            "test_batches": len(test_loader),
            "loader_config": TRANSFORMER_LOADER_CONFIG,
        }
    )


### Transformer Step 3: Train with Attention-Based Sequence Modeling

The training loop is almost the same as the CNN loop, which is useful pedagogically: once the data pipeline and validation workflow are clear, we can swap in a different model family without changing the whole experimental process.

Just like the CNN section, the reported validation F1 is per window in `"window"` mode and per timestamp in `"per_timestep"` mode.


In [ ]:
if not TRANSFORMER_RUN:
    print("Transformer fit skipped.")
else:
    for epoch in range(1, TRANSFORMER_CONFIG["epochs"] + 1):
        train_loss, _, _ = run_epoch(train_loader, training=True)
        valid_loss, valid_preds, valid_targets = run_epoch(valid_loader, training=False)
        valid_f1 = f1_score(valid_targets, valid_preds, average=report_average(task_mode), zero_division=0)
        scheduler.step(valid_f1)
        history.append(
            {
                "epoch": epoch,
                "train_loss": train_loss,
                "valid_loss": valid_loss,
                "valid_f1": valid_f1,
                "learning_rate": optimizer.param_groups[0]["lr"],
            }
        )
        print(
            f"Epoch {epoch:>2} | train_loss={train_loss:.4f} | valid_loss={valid_loss:.4f} | valid_f1={valid_f1:.4f}"
        )

        if valid_f1 > best_metric + TRANSFORMER_CONFIG["min_delta"]:
            best_metric = valid_f1
            best_epoch = epoch
            patience_counter = 0
            best_state = copy.deepcopy(model.state_dict())
            torch.save(
                {
                    "model_state_dict": best_state,
                    "task_mode": task_mode,
                    "class_labels": transformer_class_labels,
                    "feature_columns": measurement_columns,
                    "window_size": transformer_window_size,
                    "output_mode": TRANSFORMER_OUTPUT_MODE,
                    "config": {**TRANSFORMER_CONFIG, **TRANSFORMER_LOADER_CONFIG},
                },
                TRANSFORMER_MODEL_PATH,
            )
        else:
            patience_counter += 1
            if patience_counter >= TRANSFORMER_CONFIG["patience"]:
                print(f"Early stopping triggered at epoch {epoch}")
                break


### Transformer Step 4: Evaluate the Best Attention Model

Just like the CNN section, we finish by restoring the best validation checkpoint and scoring the held-out test split exactly once.


In [ ]:
if not TRANSFORMER_RUN:
    print("Transformer evaluation skipped.")
else:
    if best_state is None:
        raise RuntimeError("Transformer training did not produce a valid checkpoint")

    model.load_state_dict(best_state)
    test_loss, test_preds, test_targets = run_epoch(test_loader, training=False)

    print(pd.DataFrame(history).tail())
    print(
        {
            "output_mode": TRANSFORMER_CONFIG["output_mode"],
            "windows": len(train_dataset) + len(valid_dataset) + len(test_dataset),
            "device": str(device),
            "best_validation_f1": float(best_metric),
            "best_epoch": best_epoch,
            "test_loss": float(test_loss),
            "saved_model": str(TRANSFORMER_MODEL_PATH),
        }
    )

    if task_mode == "multiclass":
        report_labels = list(range(len(transformer_class_labels)))
        report_names = [str(label) for label in transformer_class_labels]
    else:
        report_labels = [0, 1]
        report_names = ["0", "1"]

    print(
        classification_report(
            test_targets,
            test_preds,
            labels=report_labels,
            target_names=report_names,
            zero_division=0,
        )
    )

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    history_frame = pd.DataFrame(history)
    axes[0].plot(history_frame["epoch"], history_frame["train_loss"], label="train")
    axes[0].plot(history_frame["epoch"], history_frame["valid_loss"], label="validation")
    axes[0].set_title("Transformer loss history")
    axes[0].legend()
    ConfusionMatrixDisplay.from_predictions(
        test_targets,
        test_preds,
        labels=report_labels,
        display_labels=report_names,
        normalize="true",
        xticks_rotation=45,
        ax=axes[1],
    )
    axes[1].set_title("Transformer test confusion matrix")
    plt.tight_layout()
    plt.show()


### Date-Range Demo: See Transformer Predictions on a Specific Interval

This demo follows the same `output_mode` toggle as the training section:

- `"window"` shows one true/predicted label span per window,
- `"per_timestep"` shows one true/predicted label span per timestamp.


In [ ]:
TRANSFORMER_RANGE_START = None
TRANSFORMER_RANGE_END = None
TRANSFORMER_AUTO_SELECT_TEST_RANGE = True
TRANSFORMER_MAX_POINTS_TO_PLOT = 1024

print(
    {
        "TRANSFORMER_RANGE_START": TRANSFORMER_RANGE_START,
        "TRANSFORMER_RANGE_END": TRANSFORMER_RANGE_END,
        "TRANSFORMER_AUTO_SELECT_TEST_RANGE": TRANSFORMER_AUTO_SELECT_TEST_RANGE,
        "TRANSFORMER_MAX_POINTS_TO_PLOT": TRANSFORMER_MAX_POINTS_TO_PLOT,
    }
)


In [ ]:
if not TRANSFORMER_RUN:
    print("Transformer date-range demo skipped because the transformer was not trained in this run.")
else:
    transformer_range_selection = select_time_range(
        test_df,
        time_column="Time UTC",
        label_column=TARGET_FLAG,
        start=TRANSFORMER_RANGE_START,
        end=TRANSFORMER_RANGE_END,
        auto_select=TRANSFORMER_AUTO_SELECT_TEST_RANGE,
        max_points=TRANSFORMER_MAX_POINTS_TO_PLOT,
    )
    transformer_interval_metrics = None

    transformer_interval_rows = load_rows_for_time_range(
        metadata,
        row_cache_dir=Path(ROW_CACHE_DIR),
        start=transformer_range_selection["start"],
        end=transformer_range_selection["end"],
        columns=ROW_USE_COLUMNS,
    )

    if transformer_interval_rows.empty:
        print("No row-level data was found in the requested transformer range.")
    else:
        transformer_interval_model_df, _, _ = build_model_frame(
            transformer_interval_rows,
            target_flag=TARGET_FLAG,
            measurement_columns=measurement_columns,
            task_mode=task_mode,
            model_row_limit=None,
        )
        transformer_interval_model_df = transformer_interval_model_df[
            (transformer_interval_model_df["Time UTC"] >= transformer_range_selection["start"])
            & (transformer_interval_model_df["Time UTC"] <= transformer_range_selection["end"])
        ].reset_index(drop=True)
        transformer_interval_origin = infer_interval_origin(
            transformer_range_selection["start"],
            transformer_range_selection["end"],
            {"train": train_df, "validation": valid_df, "test": test_df},
        )
        transformer_plot_palette = DEFAULT_FLAG_PALETTE if task_mode == "multiclass" else {0: "#1f77b4", 1: "#d62728"}

        if TRANSFORMER_CONFIG["output_mode"] == "window":
            transformer_interval_bundle = build_window_classification_interval_data(
                transformer_interval_model_df,
                feature_columns=measurement_columns,
                target_column="model_target",
                task_mode=task_mode,
                window_size=TRANSFORMER_CONFIG["window_size"],
                label_reduction=TRANSFORMER_CONFIG["label_reduction"],
            )

            if transformer_interval_bundle["window_frame"].empty:
                print("The selected transformer range is shorter than one full window after preprocessing, so the window-level demo is skipped.")
            else:
                transformer_predicted_labels = predict_transformer_window_model(
                    model,
                    transformer_interval_bundle["raw_sequences"],
                    task_mode=task_mode,
                    class_labels=transformer_class_labels,
                    device=str(device),
                    feature_mean=feature_mean,
                    feature_std=feature_std,
                    batch_size=TRANSFORMER_CONFIG["batch_size"],
                )
                transformer_window_frame = transformer_interval_bundle["window_frame"].copy()
                transformer_window_frame["predicted_label"] = transformer_predicted_labels
                transformer_interval_metrics = compute_interval_classification_metrics(
                    transformer_window_frame["true_label"],
                    transformer_window_frame["predicted_label"],
                    labels=transformer_class_labels,
                    average=report_average(task_mode),
                    target_names=[str(label) for label in transformer_class_labels],
                )
                transformer_true_intervals = merge_adjacent_intervals(
                    transformer_window_frame.rename(columns={"window_start": "start", "window_end": "end", "true_label": "label"})[
                        ["start", "end", "label"]
                    ]
                )
                transformer_pred_intervals = merge_adjacent_intervals(
                    transformer_window_frame.rename(columns={"window_start": "start", "window_end": "end", "predicted_label": "label"})[
                        ["start", "end", "label"]
                    ]
                )

                print(
                    {
                        "output_mode": TRANSFORMER_CONFIG["output_mode"],
                        "selection_mode": transformer_range_selection["selection_mode"],
                        "selected_priority_flag": transformer_range_selection["selected_label"],
                        "interval_origin": transformer_interval_origin,
                        "range_start": transformer_range_selection["start"].isoformat(),
                        "range_end": transformer_range_selection["end"].isoformat(),
                        "window_count_in_interval": int(len(transformer_window_frame)),
                        "interval_f1": transformer_interval_metrics["f1"],
                    }
                )
                print(transformer_interval_metrics["report_text"])
                display(
                    pd.DataFrame(
                        {
                            "true_count": transformer_window_frame["true_label"].value_counts().sort_index(),
                            "predicted_count": transformer_window_frame["predicted_label"].value_counts().sort_index(),
                        }
                    ).fillna(0).astype(int)
                )

                transformer_demo_figure = plot_time_series_with_bands(
                    transformer_interval_model_df,
                    band_specs=[
                        {"title": "True window label", "intervals": transformer_true_intervals, "palette": transformer_plot_palette},
                        {"title": "Transformer prediction", "intervals": transformer_pred_intervals, "palette": transformer_plot_palette},
                    ],
                    measurement_column=PLOT_MEASUREMENT_COLUMN,
                    secondary_column=PLOT_SECONDARY_COLUMN,
                    max_points=TRANSFORMER_MAX_POINTS_TO_PLOT,
                    title="Transformer window predictions on a selected time range",
                )
                plt.show()
        else:
            transformer_interval_bundle = build_sequence_label_interval_data(
                transformer_interval_model_df,
                feature_columns=measurement_columns,
                target_column="model_target",
                window_size=TRANSFORMER_CONFIG["window_size"],
            )

            if len(transformer_interval_bundle["raw_sequences"]) == 0:
                print("The selected transformer range is shorter than one full window after preprocessing, so the per-timestep demo is skipped.")
            else:
                transformer_predicted_labels = predict_transformer_sequence_model(
                    model,
                    transformer_interval_bundle["raw_sequences"],
                    task_mode=task_mode,
                    class_labels=transformer_class_labels,
                    device=str(device),
                    feature_mean=feature_mean,
                    feature_std=feature_std,
                    batch_size=TRANSFORMER_CONFIG["batch_size"],
                )
                transformer_point_frame = pd.DataFrame(
                    {
                        "Time UTC": pd.to_datetime(transformer_interval_bundle["raw_times"].reshape(-1), utc=True),
                        "true_label": transformer_interval_bundle["raw_targets"].reshape(-1).astype(int),
                        "predicted_label": transformer_predicted_labels.reshape(-1).astype(int),
                    }
                )
                transformer_interval_metrics = compute_interval_classification_metrics(
                    transformer_point_frame["true_label"],
                    transformer_point_frame["predicted_label"],
                    labels=transformer_class_labels,
                    average=report_average(task_mode),
                    target_names=[str(label) for label in transformer_class_labels],
                )
                transformer_true_intervals = merge_adjacent_intervals(
                    build_labeled_intervals(transformer_point_frame, time_column="Time UTC", label_column="true_label")
                )
                transformer_pred_intervals = merge_adjacent_intervals(
                    build_labeled_intervals(transformer_point_frame, time_column="Time UTC", label_column="predicted_label")
                )

                print(
                    {
                        "output_mode": TRANSFORMER_CONFIG["output_mode"],
                        "selection_mode": transformer_range_selection["selection_mode"],
                        "selected_priority_flag": transformer_range_selection["selected_label"],
                        "interval_origin": transformer_interval_origin,
                        "range_start": transformer_range_selection["start"].isoformat(),
                        "range_end": transformer_range_selection["end"].isoformat(),
                        "point_count_in_interval": int(len(transformer_point_frame)),
                        "interval_f1": transformer_interval_metrics["f1"],
                    }
                )
                print(transformer_interval_metrics["report_text"])
                display(
                    pd.DataFrame(
                        {
                            "true_count": transformer_point_frame["true_label"].value_counts().sort_index(),
                            "predicted_count": transformer_point_frame["predicted_label"].value_counts().sort_index(),
                        }
                    ).fillna(0).astype(int)
                )

                transformer_demo_figure = plot_time_series_with_bands(
                    transformer_interval_model_df,
                    band_specs=[
                        {"title": "True per-timestep label", "intervals": transformer_true_intervals, "palette": transformer_plot_palette},
                        {"title": "Transformer per-timestep prediction", "intervals": transformer_pred_intervals, "palette": transformer_plot_palette},
                    ],
                    measurement_column=PLOT_MEASUREMENT_COLUMN,
                    secondary_column=PLOT_SECONDARY_COLUMN,
                    max_points=TRANSFORMER_MAX_POINTS_TO_PLOT,
                    title="Transformer per-timestep predictions on a selected time range",
                )
                plt.show()

        if transformer_interval_metrics is not None:
            transformer_cm_fig, transformer_cm_ax = plt.subplots(figsize=(5, 4))
            ConfusionMatrixDisplay(
                confusion_matrix=transformer_interval_metrics["confusion_matrix"],
                display_labels=transformer_interval_metrics["display_labels"],
            ).plot(ax=transformer_cm_ax, cmap="Blues", colorbar=False)
            transformer_cm_ax.set_title("Transformer confusion matrix on the selected range")
            plt.tight_layout()
            plt.show()


### Reflection Questions: Transformer

1. Does the selected interval contain longer-context behavior that you would expect a transformer to use better than the CNN?
2. If the transformer is not clearly helping here, do you think the issue is window length, data volume, or model size?
3. Would you change pooling, positional information, or the target definition first if you wanted a better interval-level result?

Add your own notes or follow-up questions here:

- 
- 
- 


## Part 8 — Reflection and Next Steps

End by comparing what each model family sees, what each one misses, and which changes are most promising for the next round of experiments.


### Wrap-Up, Transformer Note, And Questions To Explore

We now have three very different model families in one notebook:

- a tabular tree ensemble,
- a local-pattern sequence model,
- and a small attention-based sequence model.

That is a useful comparison because each one sees the data a little differently.

Good next questions to explore:

1. Are there other features that might help the Random Forest?
Hint: think about lag features, rolling means, rolling standard deviations, slopes, or relationships between sensors.

2. Are there ways to improve class `3`, `4`, or `9` performance specifically?
Hint: compare the class distribution to the confusion matrix and ask which classes are hardest to distinguish.

3. Does the target itself need to change?
Hint: sometimes a binary `good` vs `issue` target, or a collapsed set of flags, is closer to the operational decision. For conductivity or turbidity, that might mean merging `3` and `4`. For oxygen, it can make more sense to ask whether `1/2` belong together and whether `3/4` belong together, rather than reusing the CTD collapse blindly.

4. What parts of the CNN are worth experimenting with?
Hint: try changing the window length, number of channels, dropout, or the way we reduce row labels into one window label.

5. Could we use even more context from time?
Hint: Random Forest only sees engineered features. CNN sees short windows. Transformer sees relationships across an entire window. There are still other options too, such as TCNs, GRUs/LSTMs, or hierarchical windows.
